# **Loading And Cleaning The Data**


<!-- CELL NOTE -->
> **Cell Note:** This section documents **Loading And Cleaning The Data** and provides context for the following code/results.
> Review it before executing downstream cells so assumptions and outputs remain aligned.


In [1]:
# CELL GUIDE:
# Purpose: Load ECG records and build a padded signal tensor + metadata table.
# Workflow:
# 1. Load ECG records and build a padded signal tensor + metadata table.
# 2. from pathlib import Path
# 3. import re
# 4. import numpy as np
# 5. import pandas as pd
# 6. import plotly.graph_objects as go

# Load ECG records and build a padded signal tensor + metadata table.
from pathlib import Path
import re

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.io import loadmat


def extract_signal_matrix(mat_dict):
    # Prefer the standard WFDB matrix key, otherwise use the first numeric matrix found.
    if "val" in mat_dict and isinstance(mat_dict["val"], np.ndarray):
        return np.asarray(mat_dict["val"], dtype=np.float32)
    for key, value in mat_dict.items():
        if not key.startswith("__") and isinstance(value, np.ndarray) and np.issubdtype(value.dtype, np.number):
            return np.asarray(value, dtype=np.float32)
    raise ValueError("No numeric signal matrix found in .mat file.")


def read_header_fields(hea_path: Path, wanted_fields):
    # Parse selected '#Field: value' entries from a WFDB header.
    fields = {field: "" for field in wanted_fields}
    if not hea_path.exists():
        return fields

    with hea_path.open("r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            for field in wanted_fields:
                prefix = f"#{field}:"
                if line.startswith(prefix):
                    fields[field] = line.split(":", 1)[1].strip()
    return fields


def extract_dx_codes(hea_path: Path):
    # Diagnosis codes are stored in the '#Dx' field and can contain comma-separated values.
    raw_dx = read_header_fields(hea_path, ["Dx"]).get("Dx", "")
    return [code.strip() for code in raw_dx.split(",") if code.strip()]


root_dir = Path("ecg_data/WFDBRecords")
mat_files = sorted(root_dir.rglob("*.mat"))
if not mat_files:
    raise FileNotFoundError(f"No .mat files found under {root_dir}")

records, signals_list = [], []
for mat_path in mat_files:
    signal = extract_signal_matrix(loadmat(mat_path))
    if signal.ndim != 2:
        raise ValueError(f"Expected a 2D signal matrix, got shape {signal.shape} for {mat_path}")

    dx_codes = extract_dx_codes(mat_path.with_suffix(".hea"))
    records.append(
        {
            "record_id": mat_path.stem,
            "relative_path": str(mat_path.relative_to(root_dir)),
            "n_leads": int(signal.shape[0]),
            "n_samples": int(signal.shape[1]),
            "dx_codes": ",".join(dx_codes),
            "n_labels": len(dx_codes),
        }
    )
    signals_list.append(signal)

# Pad variable-length records to one shared tensor shape: [record, lead, sample].
max_leads = max(arr.shape[0] for arr in signals_list)
max_samples = max(arr.shape[1] for arr in signals_list)
signals_tensor = np.full((len(signals_list), max_leads, max_samples), np.nan, dtype=np.float32)
for i, arr in enumerate(signals_list):
    leads, samples = arr.shape
    signals_tensor[i, :leads, :samples] = arr

ecg_data = {
    "signals": signals_tensor,
    "metadata": pd.DataFrame(records),
}

print(f"Loaded {len(mat_files)} .mat files")
print(f"ecg_data['signals'].shape = {ecg_data['signals'].shape}")
print(f"ecg_data['metadata'].shape = {ecg_data['metadata'].shape}")
print(f"metadata columns: {list(ecg_data['metadata'].columns)}")



Loaded 45152 .mat files
ecg_data['signals'].shape = (45152, 12, 5000)
ecg_data['metadata'].shape = (45152, 6)
metadata columns: ['record_id', 'relative_path', 'n_leads', 'n_samples', 'dx_codes', 'n_labels']


In [23]:
# CELL GUIDE:
# Purpose: Run `print(f"Missing Values: {np.isnan(ecg_data['signals']).sum()}") # Check for NaN values` and continue downstream processing.
# Workflow:
# 1. print(f"Missing Values: {np.isnan(ecg_data['signals']).sum()}") # Check for NaN values
# 2. print(f"Infinite Values: {np.isinf(ecg_data['signals']).sum()}") # Check for infinite values

print(f"Missing Values: {np.isnan(ecg_data['signals']).sum()}")  # Check for NaN values
print(f"Infinite Values: {np.isinf(ecg_data['signals']).sum()}") # Check for infinite values

Missing Values: 0
Infinite Values: 0


In [2]:
# CELL GUIDE:
# Purpose: Show header content for a few sample records (sampling rate, units, demographics, etc.).
# Workflow:
# 1. Show header content for a few sample records (sampling rate, units, demographics, etc.).
# 2. sample_indices = [0, 1, 2]
# 3. for sample_idx in sample_indices

# Show header content for a few sample records (sampling rate, units, demographics, etc.).
sample_indices = [0, 1, 2]
for sample_idx in sample_indices:
    relative_mat = Path(ecg_data["metadata"].iloc[sample_idx]["relative_path"])
    hea_path = (root_dir / relative_mat).with_suffix(".hea")

    print(f"\n{'=' * 80}")
    print(f"Sample index: {sample_idx}")
    print(f"Header file: {hea_path}")
    print(f"{'=' * 80}")
    with hea_path.open("r", encoding="utf-8", errors="ignore") as f:
        print(f.read())




Sample index: 0
Header file: ecg_data/WFDBRecords/01/010/JS00001.hea
JS00001 12 500 5000
JS00001.mat 16+24 1000/mV 16 0 -254 21756 0 I
JS00001.mat 16+24 1000/mV 16 0 264 -599 0 II
JS00001.mat 16+24 1000/mV 16 0 517 -22376 0 III
JS00001.mat 16+24 1000/mV 16 0 -5 28232 0 aVR
JS00001.mat 16+24 1000/mV 16 0 -386 16619 0 aVL
JS00001.mat 16+24 1000/mV 16 0 390 15121 0 aVF
JS00001.mat 16+24 1000/mV 16 0 -98 1568 0 V1
JS00001.mat 16+24 1000/mV 16 0 -312 -32761 0 V2
JS00001.mat 16+24 1000/mV 16 0 -98 32715 0 V3
JS00001.mat 16+24 1000/mV 16 0 810 15193 0 V4
JS00001.mat 16+24 1000/mV 16 0 810 14081 0 V5
JS00001.mat 16+24 1000/mV 16 0 527 32579 0 V6
#Age: 85
#Sex: Male
#Dx: 164889003,59118001,164934002
#Rx: Unknown
#Hx: Unknown
#Sx: Unknown


Sample index: 1
Header file: ecg_data/WFDBRecords/01/010/JS00002.hea
JS00002 12 500 5000
JS00002.mat 16+24 1000/mV 16 0 -10 12346 0 I
JS00002.mat 16+24 1000/mV 16 0 10 26962 0 II
JS00002.mat 16+24 1000/mV 16 0 20 14528 0 III
JS00002.mat 16+24 1000/mV 16 0 0 

In [2]:
# CELL GUIDE:
# Purpose: Map diagnosis codes to human-readable SNOMED condition names.
# Workflow:
# 1. Map diagnosis codes to human-readable SNOMED condition names.
# 2. def normalize_snomed_code(code)
# 3. def to_title_case_condition(label)
# 4. Enforce title case and strip presentation/category wrappers from condition names.
# 5. def parse_dx_code_list(raw_codes)
# 6. Split comma-separated codes and discard empty values.

# Map diagnosis codes to human-readable SNOMED condition names.
def normalize_snomed_code(code):
    text = str(code).strip()
    if not text or text.lower() == "nan":
        return ""
    return text[:-2] if re.fullmatch(r"\d+\.0", text) else text


def to_title_case_condition(label):
    # Enforce title case and strip presentation/category wrappers from condition names.
    cleaned = " ".join(str(label).split()).strip()
    if not cleaned:
        return ""
    titled = cleaned.title()
    titled = re.sub(r"^Electrocardiogram:\s*", "", titled, flags=re.IGNORECASE).strip()
    titled = re.sub(r"\s*\((Disorder|Finding)\)\s*$", "", titled, flags=re.IGNORECASE).strip()
    return titled


def parse_dx_code_list(raw_codes):
    # Split comma-separated codes and discard empty values.
    return [norm for norm in (normalize_snomed_code(code) for code in str(raw_codes).split(",")) if norm]


def prepare_lookup(df, code_col, label_col):
    # Standardize lookup schemas so multiple files can be merged consistently.
    out = df.rename(columns={code_col: "dx_code", label_col: "condition_label"}).copy()
    out["dx_code"] = out["dx_code"].apply(normalize_snomed_code)
    out["condition_label"] = out["condition_label"].astype(str).str.strip().apply(to_title_case_condition)
    return out[["dx_code", "condition_label"]]


primary_lookup = prepare_lookup(
# NOTE: Load tabular metadata/lookup input required for downstream joins.
    pd.read_csv("ecg_data/ConditionNames_SNOMED-CT.csv"),
    code_col="Snomed_CT",
    label_col="Full Name",
)
remaining_lookup = prepare_lookup(
# NOTE: Load tabular metadata/lookup input required for downstream joins.
    pd.read_csv("ecg_data/Remaining_DX_Codes_SNOMED_Labels.csv"),
    code_col="dx_code",
    label_col="preferred_label",
)

condition_lookup = pd.concat([primary_lookup, remaining_lookup], ignore_index=True)
condition_lookup = condition_lookup[
    condition_lookup["dx_code"].ne("")
    & condition_lookup["condition_label"].ne("")
    & condition_lookup["condition_label"].str.lower().ne("nan")
].drop_duplicates(subset=["dx_code"], keep="first")

snomed_to_condition = dict(zip(condition_lookup["dx_code"], condition_lookup["condition_label"]))

# Manual overrides for codes requiring corrected labels.
manual_condition_overrides = {
    "67741000119109": "Left Atrial Enlargement (Disorder)",
    "67751000119106": "Right Atrial High Voltage",
}
snomed_to_condition.update({code: to_title_case_condition(label) for code, label in manual_condition_overrides.items()})

metadata = ecg_data["metadata"].copy()
metadata["dx_code_list"] = metadata["dx_codes"].fillna("").astype(str).apply(parse_dx_code_list)
metadata["dx_condition_list"] = metadata["dx_code_list"].apply(
    lambda codes: [to_title_case_condition(snomed_to_condition.get(code, f"Unknown Condition ({code})")) for code in codes]
)
metadata["dx_conditions"] = metadata["dx_condition_list"].apply(lambda labels: ", ".join(labels))
ecg_data["metadata"] = metadata

display(ecg_data["metadata"][["record_id", "dx_codes", "dx_conditions"]].head())


,record_id,dx_codes,dx_conditions
0,JS00001,"164889003,59118001,164934002","Atrial Fibrillation, Right Bundle Branch Block..."
1,JS00002,"426177001,164934002","Sinus Bradycardia, T Wave Change"
2,JS00004,426177001,Sinus Bradycardia
3,JS00005,"164890007,429622005,428750005","Atrial Flutter, St Drop Down, St-T Change"
4,JS00006,426177001,Sinus Bradycardia


In [25]:
# CELL GUIDE:
# Purpose: Extract ECG lead categories from .hea headers and store them in metadata.
# Workflow:
# 1. Extract ECG lead categories from .hea headers and store them in metadata.
# 2. def extract_lead_categories_from_header(hea_path: Path)
# 3. lead_category_lists = ecg_data["metadata"].apply(
# 4. )
# 5. ecg_data["metadata"]["lead_categories"] = lead_category_lists
# 6. lead_tuple_counts = (

# Extract ECG lead categories from .hea headers and store them in metadata.
def extract_lead_categories_from_header(hea_path: Path):
    try:
        with hea_path.open("r", encoding="utf-8", errors="ignore") as f:
            lines = [ln.strip() for ln in f if ln.strip()]
    except FileNotFoundError:
        return []

    if not lines:
        return []

    header_tokens = lines[0].split()
    if len(header_tokens) < 2:
        return []

    try:
        n_leads = int(header_tokens[1])
    except ValueError:
        return []

    lead_names = []
    for lead_line in lines[1:1 + n_leads]:
        tokens = lead_line.split()
        if not tokens:
            lead_names.append("Unknown")
            continue
        lead_names.append(tokens[-1])

    return lead_names


lead_category_lists = ecg_data["metadata"].apply(
    lambda row: extract_lead_categories_from_header(
        root_dir / Path(row["relative_path"]).with_suffix(".hea")
    ),
    axis=1,
)

ecg_data["metadata"]["lead_categories"] = lead_category_lists

lead_tuple_counts = (
    lead_category_lists.apply(tuple)
    .value_counts()
)

if lead_tuple_counts.empty:
    canonical_lead_categories = []
else:
    canonical_lead_categories = list(lead_tuple_counts.index[0])

lead_name_map_from_headers = {
    idx: name for idx, name in enumerate(canonical_lead_categories)
}

ecg_data["lead_name_map_from_headers"] = lead_name_map_from_headers

print(f"Records parsed for lead categories: {len(lead_category_lists)}")
print(f"Unique lead-order patterns found: {int(lead_tuple_counts.shape[0])}")
print(f"Most common lead categories ({len(canonical_lead_categories)} leads): {canonical_lead_categories}")
print(f"Stored as ecg_data['lead_name_map_from_headers'] with {len(lead_name_map_from_headers)} entries.")

Records parsed for lead categories: 45152
Unique lead-order patterns found: 3
Most common lead categories (12 leads): ['I', 'II', 'III', 'aVR', 'aVL', 'aVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']
Stored as ecg_data['lead_name_map_from_headers'] with 12 entries.


# **Exploratory Data Analysis**


<!-- CELL NOTE -->
> **Cell Note:** This section documents **Exploratory Data Analysis** and provides context for the following code/results.
> Review it before executing downstream cells so assumptions and outputs remain aligned.


In [11]:
# CELL GUIDE:
# Purpose: Run `SERIES_COLORS = {` and continue downstream processing.
# Workflow:
# 1. SERIES_COLORS = {
# 2. }

SERIES_COLORS = {
    "default": "#636EFA",  # Plotly default blue
    "f1_micro": "#636EFA",
    "f1_macro": "rgb(20, 20, 204)",
    "precision_micro": "rgb(46, 74, 47)",
    "precision_macro": "rgb(129, 130, 130)",
    "recall_micro": "rgb(156, 126, 107)",
    "recall_macro": "rgb(230, 217, 195)", 
    "val_logloss": "rgb(87, 94, 78)",
    "best_so_far_logloss": "rgb(20, 20, 204)",
}

In [12]:
# CELL GUIDE:
# Purpose: Run `STANDARD_LEGEND = dict(` and continue downstream processing.
# Workflow:
# 1. STANDARD_LEGEND = dict(
# 2. )

STANDARD_LEGEND = dict(
    orientation="h",
    x=0,
    y=1.13,
    xanchor="left",
    yanchor="top",
    traceorder="normal",
)

In [40]:
# CELL GUIDE:
# Purpose: Run `graph_formatting = {` and continue downstream processing.
# Workflow:
# 1. graph_formatting = {
# 2. }

graph_formatting = {
    "font_family": "Helvetica Neue",
    "font_size_base": 18,
    "font_size_axis": 18,
    "font_size_subtitle": 18,
    "color_text": "#b8b8b8",
    "color_title": "white",
    "color_grid": "#4a0000",
    "color_background": "#000000",
    "color_line": "#636EFA",
    "margin_top": 170,
    "margin_right": 80,
    "margin_bottom": 80,
    "margin_left": 80,
}

In [ ]:
# CELL GUIDE:
# Purpose: Plot lead-I ECG traces for three records with a shared visual style.
# Workflow:
# 1. Plot lead-I ECG traces for three records with a shared visual style.
# 2. sample_indices = [0, 1, 2]
# 3. lead_idx = 0
# 4. sampling_rate_hz = 500
# 5. signal_unit = "mV"
# 6. text_gray = graph_formatting["color_text"]

# Plot lead-I ECG traces for three records with a shared visual style.
sample_indices = [0, 1, 2]
lead_idx = 0
sampling_rate_hz = 500
signal_unit = "mV"

text_gray = graph_formatting["color_text"]
grid_red = graph_formatting["color_grid"]
minor_red = "#2a0000"
bg_black = graph_formatting["color_background"]

def get_age_sex(hea_path: Path):
    # Pull optional demographic fields from header; default to 'Unknown' when absent.
    fields = read_header_fields(hea_path, ["Age", "Sex"])
    return fields.get("Age") or "Unknown", fields.get("Sex") or "Unknown"

n_samples = ecg_data["signals"].shape[2]
time_seconds = np.arange(n_samples) / sampling_rate_hz
fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.04)

for row_idx, sample_idx in enumerate(sample_indices, start=1):
    meta_row = ecg_data["metadata"].iloc[sample_idx]
    signal = ecg_data["signals"][sample_idx, lead_idx, :]
    duration_seconds = meta_row["n_samples"] / sampling_rate_hz
    hea_path = (root_dir / Path(meta_row["relative_path"]).with_suffix(".hea"))
    age, sex = get_age_sex(hea_path)

    subtitle = (
        f"Record: {meta_row['record_id']} | Lead: {lead_idx + 1} | Duration: {duration_seconds:.2f} s | "
        f"Unit: {signal_unit} (millivolts) | Age: {age} | Gender: {sex}"
    )

    fig.add_trace(
        go.Scatter(
            x=time_seconds,
            y=signal,
            mode="lines",
            line=dict(width=1.0, color=graph_formatting["color_line"]),
            hovertemplate="Time: %{x:.3f} s<br>Signal Strength: %{y:.2f} mV<extra></extra>",
            showlegend=False,
        ),
        row=row_idx,
        col=1,
    )

# NOTE: Configure y-axis semantics and formatting for accurate interpretation.
    fig.update_yaxes(
        title_font=dict(family=graph_formatting["font_family"], size=graph_formatting["font_size_axis"], color=text_gray),
        tickfont=dict(family=graph_formatting["font_family"], size=graph_formatting["font_size_axis"], color=text_gray),
        range=[-405, 1050],
        showgrid=True,
        gridcolor=grid_red,
        gridwidth=0.2,
        dtick=200,
        minor=dict(showgrid=True, gridcolor=minor_red, gridwidth=0.1, dtick=25),
        zeroline=False,
        row=row_idx,
        col=1,
    )

    fig.add_annotation(
        x=0,
        y=1.09,
        xref="x domain",
        yref="y domain",
        text=subtitle,
        showarrow=False,
        xanchor="left",
        align="left",
        font=dict(family=graph_formatting["font_family"], size=graph_formatting["font_size_subtitle"], color=text_gray),
        row=row_idx,
        col=1,
    )

# NOTE: Configure x-axis semantics and formatting for accurate interpretation.
fig.update_xaxes(
    tickfont=dict(family=graph_formatting["font_family"], size=graph_formatting["font_size_axis"], color=text_gray),
    showgrid=True,
    gridcolor=grid_red,
    gridwidth=0.2,
    dtick=0.5,
    minor=dict(showgrid=True, gridcolor=minor_red, gridwidth=0.1, dtick=0.05),
    zeroline=False,
)
# NOTE: Configure x-axis semantics and formatting for accurate interpretation.
fig.update_xaxes(
    title_text="Time (seconds)",
    title_font=dict(family=graph_formatting["font_family"], size=graph_formatting["font_size_axis"], color=text_gray),
    row=3,
    col=1,
)
# NOTE: Apply global styling (title, typography, spacing, colors) for readability consistency.
fig.update_layout(
    title=dict(
        text=(
            "<span style='font-size:30px;font-weight:bold;'>    Comparing Demographics Reveals Distinct Waveform Patterns Across Patients</span>"
            "<br><span style='font-size:20px;font-weight: normal;'>      Lead 1 ECG Curves in mV (millivolts) of Three Sample Patients Over 10s at 500Hz</span>"
        ),
        x=0,
        xanchor="left",
        pad=dict(t=50),
    ),
    hovermode="x unified",
    font=dict(family=graph_formatting["font_family"], size=graph_formatting["font_size_base"], color=text_gray),
    hoverlabel=dict(font=dict(family=graph_formatting["font_family"], size=graph_formatting["font_size_axis"], color=text_gray)),
    paper_bgcolor=bg_black,
    plot_bgcolor=bg_black,
    margin=dict(
        t=graph_formatting["margin_top"],
        r=graph_formatting["margin_right"],
        b=graph_formatting["margin_bottom"],
        l=graph_formatting["margin_left"],
    ),
    height=980,
)

fig.show()


In [108]:
# CELL GUIDE:
# Purpose: Plot all 12 ECG leads for the same patient using the established visual style.
# Workflow:
# 1. Plot all 12 ECG leads for the same patient using the established visual style.
# 2. sample_idx = 0
# 3. lead_indices = list(range(12))
# 4. lead_name_map = {
# 5. }
# 6. sampling_rate_hz = 500

# Plot all 12 ECG leads for the same patient using the established visual style.
sample_idx = 0
lead_indices = list(range(12))
lead_name_map = {
    0: "I", 1: "II", 2: "III", 3: "aVR", 4: "aVL", 5: "aVF",
    6: "V1", 7: "V2", 8: "V3", 9: "V4", 10: "V5", 11: "V6",
}
sampling_rate_hz = 500
signal_unit = "mV"

if "graph_formatting" not in globals():
    graph_formatting = {
        "font_family": "Helvetica Neue",
        "font_size_base": 16,
        "font_size_axis": 12,
        "font_size_subtitle": 16,
        "color_text": "#b8b8b8",
        "color_title": "white",
        "color_grid": "#4a0000",
        "color_background": "#000000",
        "color_line": "#636EFA",
        "margin_top": 170,
        "margin_right": 80,
        "margin_bottom": 80,
        "margin_left": 80,
    }

text_gray = graph_formatting["color_text"]
grid_red = graph_formatting["color_grid"]
minor_red = "#2a0000"
bg_black = graph_formatting["color_background"]

meta_row = ecg_data["metadata"].iloc[sample_idx]
hea_path = (root_dir / Path(meta_row["relative_path"]).with_suffix(".hea"))
age, sex = get_age_sex(hea_path)

n_samples = ecg_data["signals"].shape[2]
time_seconds = np.arange(n_samples) / sampling_rate_hz
fig_leads = make_subplots(rows=len(lead_indices), cols=1, shared_xaxes=True, vertical_spacing=0.02)

for row_idx, lead_idx in enumerate(lead_indices, start=1):
    signal = ecg_data["signals"][sample_idx, lead_idx, :]
    duration_seconds = meta_row["n_samples"] / sampling_rate_hz
    lead_label = lead_name_map.get(lead_idx, f"Lead {lead_idx + 1}")

    subtitle = (
        f"Record: {meta_row['record_id']} | Lead Category: {lead_label} | Duration: {duration_seconds:.2f} s | "
        f"Unit: {signal_unit} (millivolts) | Age: {age} | Gender: {sex}"
    )

    fig_leads.add_trace(
        go.Scatter(
            x=time_seconds,
            y=signal,
            mode="lines",
            line=dict(width=1.0, color=SERIES_COLORS["default"]),
            hovertemplate="Time: %{x:.3f} s<br>Signal Strength: %{y:.2f} mV<extra></extra>",
            showlegend=False,
        ),
        row=row_idx,
        col=1,
    )

# NOTE: Configure y-axis semantics and formatting for accurate interpretation.
    fig_leads.update_yaxes(
        title_font=dict(family=graph_formatting["font_family"], size=graph_formatting["font_size_axis"], color=text_gray),
        tickfont=dict(family=graph_formatting["font_family"], size=graph_formatting["font_size_axis"], color=text_gray),
        range=[-1505, 3050],
        showgrid=True,
        gridcolor=grid_red,
        gridwidth=0.2,
        dtick=600,
        minor=dict(showgrid=True, gridcolor=minor_red, gridwidth=0.1, dtick=100),
        zeroline=False,
        row=row_idx,
        col=1,
    )

    fig_leads.add_annotation(
        x=0,
        y=1.12,
        yshift=8,
        xref="x domain",
        yref="y domain",
        text=subtitle,
        showarrow=False,
        xanchor="left",
        align="left",
        font=dict(family=graph_formatting["font_family"], size=graph_formatting["font_size_subtitle"], color=text_gray),
        row=row_idx,
        col=1,
    )

# NOTE: Configure x-axis semantics and formatting for accurate interpretation.
fig_leads.update_xaxes(
    tickfont=dict(family=graph_formatting["font_family"], size=graph_formatting["font_size_axis"], color=text_gray),
    showgrid=True,
    gridcolor=grid_red,
    gridwidth=0.2,
    dtick=0.5,
    minor=dict(showgrid=True, gridcolor=minor_red, gridwidth=0.1, dtick=0.05),
    zeroline=False,
)
# NOTE: Configure x-axis semantics and formatting for accurate interpretation.
fig_leads.update_xaxes(
    title_text="Time (seconds)",
    title_font=dict(family=graph_formatting["font_family"], size=graph_formatting["font_size_axis"], color=text_gray),
    row=len(lead_indices),
    col=1,
)

# NOTE: Apply global styling (title, typography, spacing, colors) for readability consistency.
fig_leads.update_layout(
    title=dict(
        text=(
            "<span style='font-size:30px;font-weight:bold;'>    Twelve Leads Reveal Complementary Perspectives on Cardiac Activity</span>"
            "<br><span style='font-size:20px;font-weight: normal;'>      ECG Curves in mV (millivolts) of 12 Leads for One Patient Over 10s at 500Hz<br>          </span>"
        ),
        x=0,
        xanchor="left",
        pad=dict(t=200),
    ),
    paper_bgcolor=bg_black,
    plot_bgcolor=bg_black,
    font=dict(family=graph_formatting["font_family"], size=graph_formatting["font_size_base"], color=text_gray),
    margin=dict(
        t=graph_formatting["margin_top"],
        r=graph_formatting["margin_right"],
        b=graph_formatting["margin_bottom"],
        l=graph_formatting["margin_left"],
    ),
    hovermode="x unified",
    height=max(1600, 250 * len(lead_indices)),
)

fig_leads.show()

In [94]:
# CELL GUIDE:
# Purpose: Count condition frequency across all records and visualize the distribution.
# Workflow:
# 1. Count condition frequency across all records and visualize the distribution.
# 2. condition_pairs = ecg_data["metadata"][["dx_code_list", "dx_condition_list"]].copy()
# 3. condition_pairs["dx_pairs"] = condition_pairs.apply(
# 4. )
# 5. condition_pairs = condition_pairs[["dx_pairs"]].explode("dx_pairs").dropna(subset=["dx_pairs"])
# 6. condition_pairs[["dx_code", "condition_label"]] = pd.DataFrame(

# Count condition frequency across all records and visualize the distribution.
condition_pairs = ecg_data["metadata"][["dx_code_list", "dx_condition_list"]].copy()
condition_pairs["dx_pairs"] = condition_pairs.apply(
    lambda row: list(zip(row["dx_code_list"], row["dx_condition_list"])), axis=1
)

condition_pairs = condition_pairs[["dx_pairs"]].explode("dx_pairs").dropna(subset=["dx_pairs"])
condition_pairs[["dx_code", "condition_label"]] = pd.DataFrame(
    condition_pairs["dx_pairs"].tolist(),
    index=condition_pairs.index,
)
condition_pairs["condition_label"] = condition_pairs["condition_label"].apply(to_title_case_condition)

condition_frequency = (
    condition_pairs.value_counts(["condition_label", "dx_code"])
    .reset_index(name="frequency")
    .sort_values("frequency", ascending=False)
)


condition_fig_linear = go.Figure(
    data=[
        go.Bar(
            x=condition_frequency["frequency"],
            y=condition_frequency["condition_label"],
            customdata=condition_frequency[["dx_code"]],
            text=condition_frequency["frequency"],
            textposition="outside",
            textfont=dict(
                family=graph_formatting["font_family"],
                size=graph_formatting["font_size_axis"]+4,
                color=text_gray,
            ),
            cliponaxis=False,
            orientation="h",
            marker=dict(color=graph_formatting["color_line"]),
            hovertemplate="Condition: %{y}<br>DX Code: %{customdata[0]}<br>Frequency: %{x}<extra></extra>",
        )
    ]
)

# NOTE: Configure x-axis semantics and formatting for accurate interpretation.
condition_fig_linear.update_xaxes(
    title_text="Frequency",
    title_font=dict(family=graph_formatting["font_family"], size=graph_formatting["font_size_axis"], color=text_gray),
    tickfont=dict(family=graph_formatting["font_family"], size=graph_formatting["font_size_axis"], color=text_gray),
    showgrid=False,
)
# NOTE: Configure y-axis semantics and formatting for accurate interpretation.
condition_fig_linear.update_yaxes(
    title_font=dict(family=graph_formatting["font_family"], size=graph_formatting["font_size_axis"], color=text_gray),
    tickfont=dict(family=graph_formatting["font_family"], size=graph_formatting["font_size_axis"], color=text_gray),
    automargin=True,
    autorange="reversed",
    showgrid=False,
)
# NOTE: Apply global styling (title, typography, spacing, colors) for readability consistency.
condition_fig_linear.update_layout(
    title=dict(
        text=(
            "<span style='font-size:30px;font-weight:bold;'>    Heavily Skewed Distribution With <br>    Sinus Bradycardia Occuring Most Often</span>"
            "<br><span style='font-size:20px;font-weight: normal;'>      Frequency of Cardiovascular Conditions in Sample</span>"
        ),
        x=0,
        xanchor="left",
        pad=dict(t=50),
    ),
    font=dict(family=graph_formatting["font_family"], size=graph_formatting["font_size_base"], color=text_gray),
    hoverlabel=dict(font=dict(family=graph_formatting["font_family"], size=graph_formatting["font_size_axis"], color=text_gray)),
    uniformtext_minsize=10,
    uniformtext_mode="show",
    paper_bgcolor=bg_black,
    plot_bgcolor=bg_black,
    margin=dict(
        t=graph_formatting["margin_top"],
        r=max(graph_formatting["margin_right"], 120),
        b=graph_formatting["margin_bottom"],
        l=550,
    ),
    height=3500
)

condition_fig_linear.show()



In [95]:
# CELL GUIDE:
# Purpose: Count total condition annotations by age group and visualize the distribution.
# Workflow:
# 1. Count total condition annotations by age group and visualize the distribution.
# 2. age_fields = ecg_data["metadata"].apply(
# 3. )
# 4. age_values = pd.to_numeric(age_fields.apply(lambda fields: fields.get("Age", "")).replace("", np.nan), errors="coerce")
# 5. age_condition_counts = ecg_data["metadata"][["n_labels"]].copy()
# 6. age_condition_counts["age"] = age_values

# Count total condition annotations by age group and visualize the distribution.
age_fields = ecg_data["metadata"].apply(
    lambda row: read_header_fields(root_dir / Path(row["relative_path"]).with_suffix(".hea"), ["Age"]),
    axis=1,
)
age_values = pd.to_numeric(age_fields.apply(lambda fields: fields.get("Age", "")).replace("", np.nan), errors="coerce")

age_condition_counts = ecg_data["metadata"][["n_labels"]].copy()
age_condition_counts["age"] = age_values
age_condition_counts = age_condition_counts.dropna(subset=["age"])

age_bins = [0, 18, 30, 40, 50, 60, 70, 80, 120]
age_group_labels = ["0-17", "18-29", "30-39", "40-49", "50-59", "60-69", "70-79", "80+"]
age_condition_counts["age_group"] = pd.cut(
    age_condition_counts["age"],
    bins=age_bins,
    labels=age_group_labels,
    right=False,
    include_lowest=True,
)

condition_count_by_age_group = (
    age_condition_counts.dropna(subset=["age_group"])
    .groupby("age_group", observed=False)["n_labels"]
    .sum()
    .reset_index(name="condition_count")
)

age_condition_histogram = go.Figure(
    data=[
        go.Bar(
            x=condition_count_by_age_group["age_group"],
            y=condition_count_by_age_group["condition_count"],
            text=condition_count_by_age_group["condition_count"],
            textposition="outside",
            textfont=dict(
                family=graph_formatting["font_family"],
                size=graph_formatting["font_size_axis"],
                color=text_gray,
            ),
            cliponaxis=False,
            marker=dict(color=graph_formatting["color_line"]),
            hovertemplate="Age Group: %{x}<br>Total Conditions: %{y}<extra></extra>",
        )
    ]
)

# NOTE: Configure x-axis semantics and formatting for accurate interpretation.
age_condition_histogram.update_xaxes(
    title_text="Age Group",
    title_font=dict(family=graph_formatting["font_family"], size=graph_formatting["font_size_axis"], color=text_gray),
    tickfont=dict(family=graph_formatting["font_family"], size=graph_formatting["font_size_axis"], color=text_gray),
    showgrid=False,
)
# NOTE: Configure y-axis semantics and formatting for accurate interpretation.
age_condition_histogram.update_yaxes(
    title_font=dict(family=graph_formatting["font_family"], size=graph_formatting["font_size_axis"], color=text_gray),
    tickfont=dict(family=graph_formatting["font_family"], size=graph_formatting["font_size_axis"], color=text_gray),
    showgrid=False,
)
# NOTE: Apply global styling (title, typography, spacing, colors) for readability consistency.
age_condition_histogram.update_layout(
    title=dict(
        text=(
            "<span style='font-size:30px;font-weight:bold;'>    Patients Over 60 Are Most Often Diagnosed With Heart Conditions </span>"
            "<br><span style='font-size:20px;font-weight: normal;'>      Number of Cardiovascualr Conditions by Age Group</span>"
        ),
        x=0,
        xanchor="left",
        pad=dict(t=50),
    ),
    font=dict(family=graph_formatting["font_family"], size=graph_formatting["font_size_base"], color=text_gray),
    hoverlabel=dict(font=dict(family=graph_formatting["font_family"], size=graph_formatting["font_size_axis"], color=text_gray)),
    uniformtext_minsize=10,
    uniformtext_mode="show",
    paper_bgcolor=bg_black,
    plot_bgcolor=bg_black,
    margin=dict(
        t=graph_formatting["margin_top"],
        r=max(graph_formatting["margin_right"], 120),
        b=graph_formatting["margin_bottom"],
        l=graph_formatting["margin_left"],
    ),
    height=700,
)

age_condition_histogram.show()


In [90]:
# CELL GUIDE:
# Purpose: Plot distribution of condition counts per patient (label count histogram), including zero-condition patients.
# Workflow:
# 1. Plot distribution of condition counts per patient (label count histogram), including zero-condition patients.
# 2. label_counts = ecg_data["metadata"]["n_labels"].fillna(0).astype(int)
# 3. no_condition_patients = int((label_counts == 0).sum())
# 4. max_label_count = int(label_counts.max()) if len(label_counts) else 0
# 5. label_count_distribution = (
# 6. )

# Plot distribution of condition counts per patient (label count histogram), including zero-condition patients.
label_counts = ecg_data["metadata"]["n_labels"].fillna(0).astype(int)
no_condition_patients = int((label_counts == 0).sum())

max_label_count = int(label_counts.max()) if len(label_counts) else 0
label_count_distribution = (
    label_counts.value_counts()
    .reindex(range(0, max_label_count + 1), fill_value=0)
    .sort_index()
    .rename_axis("label_count")
    .reset_index(name="patient_frequency")
)

label_count_histogram = go.Figure(
    data=[
        go.Bar(
            x=label_count_distribution["label_count"],
            y=label_count_distribution["patient_frequency"],
            text=label_count_distribution["patient_frequency"],
            textposition="outside",
            textfont=dict(
                family=graph_formatting["font_family"],
                size=graph_formatting["font_size_axis"],
                color=text_gray,
            ),
            cliponaxis=False,
            marker=dict(color=graph_formatting["color_line"]),
            hovertemplate="Conditions per Patient: %{x}<br>Patients: %{y}<extra></extra>",
        )
    ]
)

# NOTE: Configure x-axis semantics and formatting for accurate interpretation.
label_count_histogram.update_xaxes(
    title_text="Number of Diagnosed Conditions per Patient",
    title_font=dict(family=graph_formatting["font_family"], size=graph_formatting["font_size_axis"], color=text_gray),
    tickfont=dict(family=graph_formatting["font_family"], size=graph_formatting["font_size_axis"], color=text_gray),
    dtick=1,
    showgrid=False,
)
# NOTE: Configure y-axis semantics and formatting for accurate interpretation.
label_count_histogram.update_yaxes(
    title_font=dict(family=graph_formatting["font_family"], size=graph_formatting["font_size_axis"], color=text_gray),
    tickfont=dict(family=graph_formatting["font_family"], size=graph_formatting["font_size_axis"], color=text_gray),
    showgrid=False,
)
# NOTE: Apply global styling (title, typography, spacing, colors) for readability consistency.
label_count_histogram.update_layout(
    title=dict(
        text=(
            "<span style='font-size:30px;font-weight:bold;'>    Almost Half Of Patients Are Diagnosed With Only One Condition</span>"
            f"<br><span style='font-size:20px;font-weight: normal;'>      Frequency of Patients With No, One or More Heart Conditions</span>"
        ),
        x=0,
        xanchor="left",
        pad=dict(t=50),
    ),
    font=dict(family=graph_formatting["font_family"], size=graph_formatting["font_size_base"], color=text_gray),
    hoverlabel=dict(font=dict(family=graph_formatting["font_family"], size=graph_formatting["font_size_axis"], color=text_gray)),
    uniformtext_minsize=10,
    uniformtext_mode="show",
    paper_bgcolor=bg_black,
    plot_bgcolor=bg_black,
    margin=dict(
        t=graph_formatting["margin_top"],
        r=max(graph_formatting["margin_right"], 120),
        b=graph_formatting["margin_bottom"],
        l=graph_formatting["margin_left"],
    ),
    height=700,
)

label_count_histogram.show()



In [97]:
# CELL GUIDE:
# Purpose: Compute diagnosed-condition co-occurrence and conditional probability matrices.
# Workflow:
# 1. Compute diagnosed-condition co-occurrence and conditional probability matrices.
# 2. condition_lists = ecg_data["metadata"]["dx_condition_list"].apply(
# 3. )
# 4. condition_records = [
# 5. ]
# 6. condition_records_df = pd.DataFrame(condition_records)

# Compute diagnosed-condition co-occurrence and conditional probability matrices.
condition_lists = ecg_data["metadata"]["dx_condition_list"].apply(
    lambda labels: sorted({to_title_case_condition(label) for label in labels if str(label).strip()})
)

condition_records = [
    {"patient_index": patient_idx, "condition": condition}
    for patient_idx, conditions in condition_lists.items()
    for condition in conditions
]
condition_records_df = pd.DataFrame(condition_records)

if condition_records_df.empty:
    print("No diagnosed conditions available to compute a co-occurrence matrix.")
else:
    condition_one_hot = pd.crosstab(
        condition_records_df["patient_index"],
        condition_records_df["condition"],
    ).astype(int)

    # Shared patient counts for each condition pair.
    condition_cooccurrence = condition_one_hot.T.dot(condition_one_hot)

    # Conditional probability matrix: P(condition_j | condition_i).
    condition_occurrence_counts = condition_cooccurrence.values.diagonal()
    condition_conditional_probability = condition_cooccurrence.div(
        pd.Series(condition_occurrence_counts, index=condition_cooccurrence.index),
        axis=0,
    )

    # Plot top conditions for readability while preserving full matrices above.
    top_n_conditions = 25
    top_conditions = (
        condition_one_hot.sum(axis=0)
        .sort_values(ascending=False)
        .head(top_n_conditions)
        .index.tolist()
    )

    condition_cooccurrence_plot = condition_cooccurrence.loc[top_conditions, top_conditions]
    condition_conditional_probability_plot = condition_conditional_probability.loc[top_conditions, top_conditions]

    black_white_scale = [[0.0, "#000000"], [1.0, "#ffffff"]]
    condition_ids = list(range(1, len(top_conditions) + 1))
    condition_index_table = pd.DataFrame(
        {
            "Index": condition_ids,
            "Condition": top_conditions,
        }
    )
    condition_labels_array = np.array(top_conditions)
    y_condition_grid = np.repeat(condition_labels_array[:, None], len(top_conditions), axis=1)
    x_condition_grid = np.repeat(condition_labels_array[None, :], len(top_conditions), axis=0)
    condition_pair_customdata = np.dstack((y_condition_grid, x_condition_grid))
    table_border_color = "rgba(200, 200, 200, 0.45)"
    table_font_size = max(10, graph_formatting["font_size_axis"] - 1)
    table_row_height = 23.9
    table_domain_end = 0.42
    total_table_rows = len(condition_ids) + 1  # +1 for header row
    table_top = 0.995     # adjust to match your table top
    table_bottom = 0.009  # adjust to match your table bottom

    row_h = (table_top - table_bottom) / total_table_rows

    condition_cooccurrence_heatmap = make_subplots(
        rows=1,
        cols=2,
        specs=[[{"type": "table"}, {"type": "heatmap"}]],
        column_widths=[0.45, 0.55],
        horizontal_spacing=0.06,
    )
    condition_cooccurrence_heatmap.add_trace(
        go.Table(
            columnwidth=[0.16, 0.84],
            header=dict(
                values=["Index", "Condition"],
                fill_color=bg_black,
                line_color="rgba(0,0,0,0)",
                line_width=0,
                font=dict(family=graph_formatting["font_family"], size=table_font_size, color=text_gray),
                align=["center", "left"],
                height=26,
            ),
            cells=dict(
                values=[condition_index_table["Index"], condition_index_table["Condition"]],
                fill_color=bg_black,
                line_color="rgba(0,0,0,0)",
                line_width=0,
                font=dict(family=graph_formatting["font_family"], size=table_font_size, color=text_gray),
                align=["center", "left"],
                height=table_row_height,
            ),
        ),
        row=1,
        col=1,
    )
    for row_idx in range(total_table_rows + 1):  # +1 if you want the bottom border too
        y_pos = table_top - row_idx * row_h
        condition_cooccurrence_heatmap.add_shape(
            type="line",
            x0=0,
            x1=table_domain_end,
            y0=y_pos,
            y1=y_pos,
            xref="paper",
            yref="paper",
            line=dict(color=table_border_color, width=0.6),
            layer="above",
        )
    condition_cooccurrence_heatmap.add_trace(
        go.Heatmap(
            z=condition_cooccurrence_plot.values,
            x=condition_ids,
            y=condition_ids,
            customdata=condition_pair_customdata,
            colorscale=black_white_scale,
            colorbar=dict(
                title="Co-Occurrence",
                titlefont=dict(
                    family=graph_formatting["font_family"],
                    size=graph_formatting["font_size_axis"],
                    color=text_gray,
                ),
                tickfont=dict(
                    family=graph_formatting["font_family"],
                    size=graph_formatting["font_size_axis"],
                    color=text_gray,
                ),
            ),
            hovertemplate="Condition A [%{y}] %{customdata[0]}<br>Condition B [%{x}] %{customdata[1]}<br>Shared Patients: %{z}<extra></extra>",
        ),
        row=1,
        col=2,
    )

# NOTE: Configure x-axis semantics and formatting for accurate interpretation.
    condition_cooccurrence_heatmap.update_xaxes(
        title_text="Condition Index",
        title_font=dict(family=graph_formatting["font_family"], size=graph_formatting["font_size_axis"], color=text_gray),
        tickfont=dict(family=graph_formatting["font_family"], size=graph_formatting["font_size_axis"], color=text_gray),
        tickmode="array",
        tickvals=condition_ids,
        ticktext=[str(idx) for idx in condition_ids],
        tickangle=0,
        showgrid=False,
        row=1,
        col=2,
    )
# NOTE: Configure y-axis semantics and formatting for accurate interpretation.
    condition_cooccurrence_heatmap.update_yaxes(
        title_text="Condition Index",
        title_font=dict(family=graph_formatting["font_family"], size=graph_formatting["font_size_axis"], color=text_gray),
        tickfont=dict(family=graph_formatting["font_family"], size=graph_formatting["font_size_axis"], color=text_gray),
        tickmode="array",
        tickvals=condition_ids,
        ticktext=[str(idx) for idx in condition_ids],
        showgrid=False,
        automargin=True,
        row=1,
        col=2,
    )
# NOTE: Apply global styling (title, typography, spacing, colors) for readability consistency.
    condition_cooccurrence_heatmap.update_layout(
        title=dict(
            text=(
                "<span style='font-size:30px;font-weight:bold;'>    ECG Diagnoses Cluster Around A Core of Rhythm and ST/T Abnormalities</span>"
                f"<br><span style='font-size:20px;font-weight: normal;'>      Co-Occurence Matrix, Top {len(top_conditions)} Most Frequent Conditions</span>"
            ),
            x=0,
            xanchor="left",
            pad=dict(t=50),
        ),
        font=dict(family=graph_formatting["font_family"], size=graph_formatting["font_size_base"], color=text_gray),
        hoverlabel=dict(font=dict(family=graph_formatting["font_family"], size=graph_formatting["font_size_axis"], color=text_gray)),
        paper_bgcolor=bg_black,
        plot_bgcolor=bg_black,
        margin=dict(
            t=graph_formatting["margin_top"],
            r=max(graph_formatting["margin_right"], 120),
            b=160,
            l=40,
        ),
        height=960,
    )

    condition_conditional_probability_heatmap = make_subplots(
        rows=1,
        cols=2,
        specs=[[{"type": "table"}, {"type": "heatmap"}]],
        column_widths=[0.45, 0.55],
        horizontal_spacing=0.06,
    )
    condition_conditional_probability_heatmap.add_trace(
        go.Table(
            columnwidth=[0.16, 0.84],
            header=dict(
                values=["Index", "Condition"],
                fill_color=bg_black,
                line_color="rgba(0,0,0,0)",
                line_width=0,
                font=dict(family=graph_formatting["font_family"], size=table_font_size, color=text_gray),
                align=["center", "left"],
                height=26,
            ),
            cells=dict(
                values=[condition_index_table["Index"], condition_index_table["Condition"]],
                fill_color=bg_black,
                line_color="rgba(0,0,0,0)",
                line_width=0,
                font=dict(family=graph_formatting["font_family"], size=table_font_size, color=text_gray),
                align=["center", "left"],
                height=table_row_height,
            ),
        ),
        row=1,
        col=1,
    )
    for row_idx in range(total_table_rows + 1):  
        y_pos = table_top - row_idx * row_h
        condition_conditional_probability_heatmap.add_shape(
            type="line",
            x0=0,
            x1=table_domain_end,
            y0=y_pos,
            y1=y_pos,
            xref="paper",
            yref="paper",
            line=dict(color=table_border_color, width=0.6),
            layer="above",
        )
    condition_conditional_probability_heatmap.add_trace(
        go.Heatmap(
            z=condition_conditional_probability_plot.values,
            x=condition_ids,
            y=condition_ids,
            customdata=condition_pair_customdata,
            zmin=0,
            zmax=1,
            colorscale=black_white_scale,
            colorbar=dict(
                title="P(B|A)",
                titlefont=dict(
                    family=graph_formatting["font_family"],
                    size=graph_formatting["font_size_axis"],
                    color=text_gray,
                ),
                tickfont=dict(
                    family=graph_formatting["font_family"],
                    size=graph_formatting["font_size_axis"],
                    color=text_gray,
                ),
            ),
            hovertemplate="Condition A [%{y}] %{customdata[0]}<br>Condition B [%{x}] %{customdata[1]}<br>Conditional Probability: %{z:.3f}<extra></extra>",
        ),
        row=1,
        col=2,
    )

# NOTE: Configure x-axis semantics and formatting for accurate interpretation.
    condition_conditional_probability_heatmap.update_xaxes(
        title_text="Condition B Index",
        title_font=dict(family=graph_formatting["font_family"], size=graph_formatting["font_size_axis"], color=text_gray),
        tickfont=dict(family=graph_formatting["font_family"], size=graph_formatting["font_size_axis"], color=text_gray),
        tickmode="array",
        tickvals=condition_ids,
        ticktext=[str(idx) for idx in condition_ids],
        tickangle=0,
        showgrid=False,
        row=1,
        col=2,
    )
# NOTE: Configure y-axis semantics and formatting for accurate interpretation.
    condition_conditional_probability_heatmap.update_yaxes(
        title_text="Condition A Index",
        title_font=dict(family=graph_formatting["font_family"], size=graph_formatting["font_size_axis"], color=text_gray),
        tickfont=dict(family=graph_formatting["font_family"], size=graph_formatting["font_size_axis"], color=text_gray),
        tickmode="array",
        tickvals=condition_ids,
        ticktext=[str(idx) for idx in condition_ids],
        showgrid=False,
        automargin=True,
        row=1,
        col=2,
    )
# NOTE: Apply global styling (title, typography, spacing, colors) for readability consistency.
    condition_conditional_probability_heatmap.update_layout(
        title=dict(
            text=(
                "<span style='font-size:30px;font-weight:bold;'>    Conditional Dependence is Directional and Concentrated Around Sinus Related Conditions</span>"
                f"<br><span style='font-size:20px;font-weight: normal;'>      Conditional Probability Matrix P(Condition B | Condition A), Top {len(top_conditions)} Most Frequent Conditions</span>"
            ),
            x=0,
            xanchor="left",
            pad=dict(t=50),
        ),
        font=dict(family=graph_formatting["font_family"], size=graph_formatting["font_size_base"], color=text_gray),
        hoverlabel=dict(font=dict(family=graph_formatting["font_family"], size=graph_formatting["font_size_axis"], color=text_gray)),
        paper_bgcolor=bg_black,
        plot_bgcolor=bg_black,
        margin=dict(
            t=graph_formatting["margin_top"],
            r=max(graph_formatting["margin_right"], 120),
            b=160,
            l=40,
        ),
        height=960,
    )

    condition_cooccurrence_heatmap.show()
    condition_conditional_probability_heatmap.show()



In [98]:
# CELL GUIDE:
# Purpose: Identify and visualize the top 5 most common regex-defined condition types in the sample.
# Workflow:
# 1. Identify and visualize the top 5 most common regex-defined condition types in the sample.
# 2. import re
# 3. import plotly.graph_objects as go
# 4. condition_type_patterns = {
# 5. }
# 6. condition_text = ecg_data["metadata"]["dx_condition_list"].apply(

# Identify and visualize the top 5 most common regex-defined condition types in the sample.
import re
import plotly.graph_objects as go

condition_type_patterns = {
    "Arrhythmia": r"arrhythm|atrial fibrillation|flutter|tachycard|bradycard|extrasystol|premature|bigeminy|trigeminy",
    "Conduction Disorder": r"bundle branch|av block|conduction|qrs widening|hemiblock|fascicular",
    "Ischemia": r"ischemi|infarct|stemi|nstemi|coronary|myocardial",
    "Hypertrophy": r"hypertroph|enlargement|dilatation|cardiomegaly",
    "ST-T Abnormality": r"st\s*elev|st\s*depress|t-wave|repolarization|qt prolong",
    "Axis Deviation": r"axis deviation|left axis|right axis",
    "Pacemaker": r"pacemaker|paced rhythm|device",
}

condition_text = ecg_data["metadata"]["dx_condition_list"].apply(
    lambda labels: " | ".join([str(lbl).lower() for lbl in labels if str(lbl).strip()])
)

total_patients = len(condition_text)
condition_type_rows = []

for cond_type, pattern in condition_type_patterns.items():
    regex = re.compile(pattern, flags=re.IGNORECASE)
    hits = condition_text.apply(lambda txt: bool(regex.search(txt)))
    count = int(hits.sum())
    pct = (100.0 * count / total_patients) if total_patients else 0.0
    condition_type_rows.append({
        "condition_type": cond_type,
        "count": count,
        "percentage": pct,
    })

condition_type_df = (
    pd.DataFrame(condition_type_rows)
    .sort_values(["percentage", "count"], ascending=False)
    .head(5)
    .reset_index(drop=True)
)

if condition_type_df.empty:
    print("No condition types were identified from the regex patterns.")
else:
    bar_color = SERIES_COLORS.get("default", "#636EFA") if "SERIES_COLORS" in globals() else "#636EFA"

    condition_type_fig = go.Figure(
        data=[
            go.Bar(
                x=condition_type_df["percentage"],
                y=condition_type_df["condition_type"],
                orientation="h",
                marker=dict(color=bar_color),
                text=condition_type_df.apply(
                    lambda r: f"{r['percentage']:.1f}% ({int(r['count'])})", axis=1
                ),
                textposition="outside",
                hovertemplate=(
                    "Condition Type: %{y}<br>"
                    "Occurrence: %{x:.2f}%<br>"
                    "Patient Count: %{customdata}<extra></extra>"
                ),
                customdata=condition_type_df["count"],
                showlegend=False,
            )
        ]
    )

# NOTE: Configure x-axis semantics and formatting for accurate interpretation.
    condition_type_fig.update_xaxes(
        range=[0, max(5, float(condition_type_df["percentage"].max()) * 1.15)],
        showgrid=True,
        gridcolor="#1f1f1f",
        gridwidth=0.2,
        zeroline=False,
    )
# NOTE: Configure y-axis semantics and formatting for accurate interpretation.
    condition_type_fig.update_yaxes(
        categoryorder="total ascending",
        showgrid=False,
        zeroline=False,
    )

# NOTE: Apply global styling (title, typography, spacing, colors) for readability consistency.
    condition_type_fig.update_layout(
        title=dict(
            text=(
                "<span style='font-size:30px;font-weight:bold;'>    Most Conditions In Sample Are Arrhythmias</span>"
                "<br><span style='font-size:20px;font-weight: normal;'>      Top 5 Condition Categories by Patient-Level Occurrence in %</span>"
            ),
            x=0,
            xanchor="left",
            pad=dict(t=50),
        ),
        paper_bgcolor=graph_formatting.get("color_background", "#000000") if "graph_formatting" in globals() else "#000000",
        plot_bgcolor=graph_formatting.get("color_background", "#000000") if "graph_formatting" in globals() else "#000000",
        font=dict(
            family=graph_formatting.get("font_family", "Helvetica Neue") if "graph_formatting" in globals() else "Helvetica Neue",
            size=graph_formatting.get("font_size_base", 16) if "graph_formatting" in globals() else 16,
            color=graph_formatting.get("color_text", "#b8b8b8") if "graph_formatting" in globals() else "#b8b8b8",
        ),
        margin=dict(
            t=graph_formatting.get("margin_top", 170) if "graph_formatting" in globals() else 170,
            r=graph_formatting.get("margin_right", 80) if "graph_formatting" in globals() else 80,
            b=graph_formatting.get("margin_bottom", 80) if "graph_formatting" in globals() else 80,
            l=200
        ),
        height=760,
    )

    condition_type_fig.show()

ecg_data["condition_type_regex_top5"] = condition_type_df.copy()

# **Feature Engineering**


<!-- CELL NOTE -->
> **Cell Note:** This section documents **Feature Engineering** and provides context for the following code/results.
> Review it before executing downstream cells so assumptions and outputs remain aligned.


In [3]:
# CELL GUIDE:
# Purpose: Compute formula-aligned ECG features for each patient and store them in ecg_data["metrics"].
# Workflow:
# 1. Compute formula-aligned ECG features for each patient and store them in ecg_data["metrics"].
# 2. from scipy.signal import find_peaks, peak_widths, welch
# 3. def _nanmean_or_nan(values): # Compute mean of values, returning NaN if all values are NaN or if the array is empty.
# 4. def _spectral_entropy_from_psd(psd): # Compute spectral entropy from a power spectral density (PSD) array, handling e...
# 5. def _approximate_entropy(u, m=2, r=None): # Compute approximate entropy of a time series u with embedding dimension m...
# 6. def _sample_entropy(u, m=2, r=None): # Compute sample entropy of a time series u with embedding dimension m and toler...

# Compute formula-aligned ECG features for each patient and store them in ecg_data["metrics"].
from scipy.signal import find_peaks, peak_widths, welch


def _nanmean_or_nan(values):                                        # Compute mean of values, returning NaN if all values are NaN or if the array is empty.
    arr = np.asarray(values, dtype=float)
    if arr.size == 0 or np.all(np.isnan(arr)):
        return np.nan
    return float(np.nanmean(arr))


def _spectral_entropy_from_psd(psd):                                # Compute spectral entropy from a power spectral density (PSD) array, handling edge cases to avoid NaN or infinite values.
    psd = np.asarray(psd, dtype=float)
    psd = np.where(np.isfinite(psd) & (psd > 0), psd, 0.0)          # Ensure PSD values are finite and non-negative, replacing invalid values with zero to prevent issues in subsequent calculations.
    total = psd.sum()
    if total <= 0:
        return np.nan
    p = psd / total
    p = p[p > 0]
    if p.size == 0:
        return np.nan
    return float(-(p * np.log(p)).sum())                            # Calculate spectral entropy using the formula -sum(p * log(p)), where p is the normalized PSD. We return the result as a float for consistency.


def _approximate_entropy(u, m=2, r=None):                          # Compute approximate entropy of a time series u with embedding dimension m and tolerance r, handling edge cases for small sample sizes and invalid parameters.
    u = np.asarray(u, dtype=float)                              
    u = u[np.isfinite(u)]
    n = u.size
    if n < m + 2:
        return np.nan
    if r is None:
        r = 0.2 * np.std(u)                                         # Set default tolerance r to 20% of the standard deviation of the time series, which is a common choice for approximate entropy calculations.
    if not np.isfinite(r) or r <= 0:
        return np.nan

    def _phi(mm):                                                   # Helper function to compute the phi value for embedding dimension mm, used in the approximate entropy calculation.
        x = np.array([u[i:i + mm] for i in range(n - mm + 1)])
        c = np.empty(x.shape[0], dtype=float)
        for i in range(x.shape[0]):
            dist = np.max(np.abs(x - x[i]), axis=1)                 # Compute the maximum distance between the i-th embedded vector and all embedded vectors, used to count how many are within the tolerance r.
            c[i] = np.mean(dist <= r)                               # Calculate the proportion of embedded vectors that are within distance r of the i-th vector and store it in c[i].
        c = np.where(c > 0, c, 1e-12)
        return np.mean(np.log(c))

    return float(_phi(m) - _phi(m + 1))


def _sample_entropy(u, m=2, r=None):                                # Compute sample entropy of a time series u with embedding dimension m and tolerance r, handling edge cases for small sample sizes and invalid parameters.
    u = np.asarray(u, dtype=float)
    u = u[np.isfinite(u)]
    n = u.size
    if n < m + 2:
        return np.nan
    if r is None:
        r = 0.2 * np.std(u)                                         # Set default tolerance r to 20% of the standard deviation of the time series, which is a common choice for sample entropy calculations.
    if not np.isfinite(r) or r <= 0:
        return np.nan

    def _count(mm):                                                 # Helper function to count the number of pairs of embedded vectors of dimension mm that are within distance r, used in the sample entropy calculation.     
        x = np.array([u[i:i + mm] for i in range(n - mm + 1)])
        c = 0
        for i in range(x.shape[0] - 1):
            dist = np.max(np.abs(x[i + 1:] - x[i]), axis=1)         # Compute the maximum distance between the i-th embedded vector and all subsequent embedded vectors.
            c += np.sum(dist <= r)                                  # Count how many of those distances are within the tolerance r and accumulate the count.    
        return c

    b = _count(m)
    a = _count(m + 1)
    if b == 0:
        return np.nan
    if a == 0:
        return np.inf
    return float(-np.log(a / b))                                    # Compute the sample entropy as the negative logarithm of the ratio of counts for embedding dimensions m+1 and m.


def _dfa_alpha(series):                                             # Compute the detrended fluctuation analysis (DFA) alpha exponent for a time series, handling edge cases for small sample sizes and invalid values.
    x = np.asarray(series, dtype=float)
    x = x[np.isfinite(x)]                                           
    n = x.size
    if n < 12:                                                      # DFA requires a minimum number of data points to produce a reliable estimate, and 12 is a common threshold for short time series. If the series is too short, we return NaN to indicate that the DFA alpha cannot be computed reliably.
        return np.nan

    y = np.cumsum(x - np.mean(x))
    scales = np.unique(np.floor(np.logspace(np.log10(4), np.log10(max(5, n // 2)), num=6)).astype(int)) # Define a range of window sizes (scales) for the DFA analysis, logarithmically spaced between 4 and n//2, with a total of 6 scales. We ensure that the scales are valid for the given time series length.
    fluct = []
    valid_scales = []
    for s in scales:
        if s < 4 or n // s < 2:
            continue
        rms_vals = []
        for start in range(0, n - s + 1, s):                                        # Divide the integrated time series into non-overlapping segments of length s and compute the root mean square (RMS) of the detrended fluctuations for each segment. We fit a linear trend to each segment and calculate the RMS of the residuals from that trend, which represents the fluctuation at scale s.
            seg = y[start:start + s]
            t = np.arange(s, dtype=float)
            coeffs = np.polyfit(t, seg, 1)
            trend = coeffs[0] * t + coeffs[1]
            rms_vals.append(np.sqrt(np.mean((seg - trend) ** 2)))                   # Append the RMS value for this segment to the list of RMS values for the current scale s.
        if rms_vals:
            fluct.append(np.mean(rms_vals))
            valid_scales.append(s)

    fluct = np.asarray(fluct, dtype=float)                                          # Convert the list of fluctuation values to a NumPy array for further processing. We will use these values to estimate the DFA alpha exponent by fitting a line to the log-log plot of fluctuation versus scale.
    valid_scales = np.asarray(valid_scales, dtype=float)
    mask = (fluct > 0) & np.isfinite(fluct)
    if mask.sum() < 2:
        return np.nan
    slope, _ = np.polyfit(np.log(valid_scales[mask]), np.log(fluct[mask]), 1)       # Fit a linear model to the log-log data of valid scales and their corresponding fluctuation values to estimate the slope, which is the DFA alpha exponent. We return this slope as a float, representing the long-range correlation properties of the time series.
    return float(slope)


def _higuchi_fd(series, kmax=6):                                                    # Compute the Higuchi fractal dimension of a time series, handling edge cases for small sample sizes and invalid values. The Higuchi fractal dimension is a measure of the complexity of a time series, with higher values indicating more complex dynamics. We use a range of k values to compute the average length of the curve at different scales and then estimate the slope to determine the fractal dimension.
    x = np.asarray(series, dtype=float)
    x = x[np.isfinite(x)]
    n = x.size
    if n < 16:
        return np.nan

    lk = []
    k_vals = []
    for k in range(1, kmax + 1):                                            # Loop over different k values, which represent the scale at which we are measuring the length of the curve. For each k, we will compute the average length of the curve segments defined by that k and then use these lengths to estimate the slope for the fractal dimension calculation.
        lm = []
        for m in range(k):
            idx = np.arange(m, n, k)
            if idx.size < 2:
                continue
            length = np.sum(np.abs(np.diff(x[idx])))                        # Compute the length of the curve for the given k and m by summing the absolute differences of the time series values at the specified indices. This length is then normalized to account for the number of segments and the scale k.
            norm = (n - 1) / (((n - m - 1) // k) * k)                       # Normalize the length by the number of segments and the scale k to ensure that we are measuring the average length of the curve at the given scale. This normalization is crucial for accurately estimating the fractal dimension.
            lm.append((length * norm) / k)                                  # Append the normalized length for this k and m to the list of lengths for the current k. We will average these lengths across different m values to get a more robust estimate of the curve length at scale k.
        if lm:
            lk.append(np.mean(lm))                                          # Append the average length for this k to the list of lengths (lk) that we will use to estimate the slope for the fractal dimension calculation. We also keep track of the corresponding k values in k_vals for later use in fitting the line to estimate the slope.
            k_vals.append(k)

    lk = np.asarray(lk, dtype=float)
    k_vals = np.asarray(k_vals, dtype=float)
    mask = (lk > 0) & np.isfinite(lk)
    if mask.sum() < 2:                                                      # Check if we have at least two valid points to fit a line for estimating the slope. If not, we return NaN to indicate that the Higuchi fractal dimension cannot be computed reliably for this time series.
        return np.nan
    slope, _ = np.polyfit(np.log(1.0 / k_vals[mask]), np.log(lk[mask]), 1)  # Fit a linear model to the log-log data of 1/k and the corresponding average lengths (lk) to estimate the slope, which is used to calculate the Higuchi fractal dimension. The slope represents how the length of the curve changes with scale, and we return this slope as a float for consistency in our feature set.
    return float(slope)




def _robust_band_power(freqs, psd, band_low, band_high):                    #
    """Robustly integrate PSD over a band by interpolating available bins."""
    freqs = np.asarray(freqs, dtype=float)
    psd = np.asarray(psd, dtype=float)

    valid = np.isfinite(freqs) & np.isfinite(psd)                           # Ensure that we only consider finite values for frequencies and power spectral density (PSD) when performing the band power calculation. This step is crucial to avoid issues with NaN or infinite values that could arise from invalid data points, which would otherwise lead to incorrect integration results.
    freqs = freqs[valid]
    psd = psd[valid]
    if freqs.size < 2:                                                      # Check if we have at least two valid frequency bins to perform interpolation and integration. If there are fewer than 2 valid points, we cannot reliably estimate the band power, and we return NaN to indicate that the calculation cannot be performed.
        return np.nan

    order = np.argsort(freqs)                                               # Sort the frequency bins in ascending order to ensure that the interpolation and integration are performed correctly. The Welch method or other PSD estimation techniques may not always return frequency bins in sorted order, so we use np.argsort to get the indices that would sort the frequencies and then reorder both the frequencies and their corresponding PSD values accordingly. This step is essential for accurate interpolation and integration over the specified frequency band.
    freqs = freqs[order]    
    psd = psd[order]

    freqs, idx = np.unique(freqs, return_index=True)                        # Ensure that the frequency bins are unique and sorted, which is necessary for accurate interpolation. We use np.unique to get the unique frequencies and their corresponding indices, and then we sort the frequencies and PSD values accordingly. This step helps to prevent issues with duplicate frequency bins that could arise from the Welch method or other PSD estimation techniques, ensuring that our band power calculation is based on a clean set of frequency bins.
    psd = psd[idx]
    if freqs.size < 2:
        return np.nan

    if band_high <= freqs[0] or band_low >= freqs[-1]:                      # Check if the specified frequency band is completely outside the range of available frequency bins. If the upper bound of the band is less than or equal to the lowest frequency bin, or if the lower bound of the band is greater than or equal to the highest frequency bin, then there is no overlap between the band and the available frequencies, and we return NaN to indicate that the band power cannot be computed for this band.
        return np.nan

    left = max(band_low, freqs[0])
    right = min(band_high, freqs[-1])
    if right <= left:
        return np.nan

    step = max(1e-4, float(np.median(np.diff(freqs))))                      # Determine a reasonable step size for interpolation based on the median spacing of the frequency bins, ensuring that we have enough points to accurately integrate the PSD over the specified band. We set a minimum step size of 1e-4 to avoid issues with extremely small steps that could arise from closely spaced frequency bins, which would lead to excessive computational load without improving accuracy.
    n_grid = max(64, int(np.ceil((right - left) / step)))                   # Calculate the number of grid points for interpolation based on the width of the frequency band and the determined step size, ensuring that we have a sufficient number of points to capture the shape of the PSD within the band. We set a minimum of 64 grid points to ensure that we can accurately represent the PSD even for narrow bands, which is important for reliable integration results.
    grid = np.linspace(left, right, n_grid)                                 # Create a grid of frequency points within the specified band for interpolation, which will allow us to estimate the PSD values at these points and perform numerical integration to calculate the band power. The grid is defined from the left to the right boundary of the band, with a number of points determined by the width of the band and the step size, ensuring that we have a dense enough sampling of the PSD within the band for accurate integration.
    psd_interp = np.interp(grid, freqs, psd)                                # Interpolate the PSD values at the grid points using the available frequency bins and their corresponding PSD values. This step allows us to estimate the PSD at frequencies that may not be directly available from the original PSD estimation, providing a more accurate representation of the PSD within the specified band for integration.   

    power = float(np.trapz(psd_interp, grid))
    return power if np.isfinite(power) and power >= 0 else np.nan           # Integrate the interpolated PSD values over the grid of frequencies within the band using the trapezoidal rule to calculate the band power. We return the calculated power as a float, ensuring that it is finite and non-negative, as negative power values would not be meaningful in this context. If the calculated power is not finite or is negative, we return NaN to indicate that the band power cannot be reliably computed for this band.

def _extract_record_features(signal_lead, fs_hz):                           # Extract a set of ECG features from a single lead of the ECG signal, including heart rate variability metrics and waveform characteristics. We handle edge cases for short signals and ensure that we return NaN for features that cannot be computed reliably.
    x = np.asarray(signal_lead, dtype=float)
    x = x[np.isfinite(x)]
    if x.size < 500:
        return {k: np.nan for k in feature_names}

    baseline = float(np.median(x))
    centered = x - baseline

    prominence = max(1e-6, 0.6 * np.nanstd(centered))
    r_peaks, _ = find_peaks(centered, distance=max(1, int(0.25 * fs_hz)), prominence=prominence)

    if r_peaks.size >= 2:
        rr = np.diff(r_peaks) / fs_hz
        rr = rr[(rr > 0.25) & (rr < 2.5)]
    else:
        rr = np.array([], dtype=float)

    mean_rr = np.nan if rr.size == 0 else float(np.mean(rr))                            # Compute the mean RR interval, which is the average time between consecutive R peaks in the ECG signal. We return NaN if there are no valid RR intervals to avoid misleading results in subsequent calculations that depend on the mean RR interval.
    mean_hr = np.nan if not np.isfinite(mean_rr) or mean_rr <= 0 else 60.0 / mean_rr    # Compute the mean heart rate in beats per minute (BPM) based on the mean RR interval. We return NaN if the mean RR interval is not finite or non-positive, as this would indicate an invalid heart rate calculation.
    sdnn = np.nan if rr.size < 2 else float(np.std(rr, ddof=1))                         # Compute the standard deviation of RR intervals (SDNN), which is a measure of heart rate variability. We return NaN if there are fewer than 2 RR intervals, as the standard deviation cannot be computed reliably with less than 2 data points.
    drr = np.diff(rr) if rr.size >= 2 else np.array([], dtype=float)                    # Compute the differences between consecutive RR intervals (dRR), which are used to calculate additional heart rate variability metrics. We return an empty array if there are fewer than 2 RR intervals, as we cannot compute differences without at least 2 intervals.
    rmssd = np.nan if drr.size == 0 else float(np.sqrt(np.mean(drr ** 2)))              # Compute the root mean square of successive differences (RMSSD), which is another measure of heart rate variability. We return NaN if there are no valid dRR values to avoid misleading results in subsequent calculations that depend on RMSSD.
    pnn50 = np.nan if drr.size == 0 else float(np.mean(np.abs(drr) > 0.05))             # Compute the proportion of successive RR interval differences that are greater than 50 ms (pNN50), which is a measure of heart rate variability. We return NaN if there are no valid dRR values to avoid misleading results in subsequent calculations that depend on pNN50.

    pr_vals, qrs_vals, qt_vals = [], [], []
    st_vals, r_amp_vals, qrs_area_vals, sym_vals = [], [], [], []

    for r in r_peaks:
        qrs_on = max(0, r - int(0.04 * fs_hz))                                          # Estimate the QRS complex onset by looking back 40 ms from the R peak, ensuring we do not go below index 0. This is a common approach to approximate the start of the QRS complex, which typically begins shortly before the R peak.
        qrs_off = min(x.size - 1, r + int(0.06 * fs_hz))                                # Estimate the QRS complex offset by looking forward 60 ms from the R peak, ensuring we do not exceed the signal length. This is a common approach to approximate the end of the QRS complex, which typically ends shortly after the R peak.

        p_start = max(0, r - int(0.28 * fs_hz))                                         # Estimate the P wave start by looking back 280 ms from the R peak, ensuring we do not go below index 0. This is a common approach to approximate the start of the P wave, which typically begins well before the R peak.   
        p_end = max(p_start + 1, r - int(0.06 * fs_hz))                                 # Estimate the P wave end by looking back 60 ms from the R peak, ensuring it is at least one sample after the P wave start. This is a common approach to approximate the end of the P wave, which typically ends shortly before the QRS complex begins.
        p_seg = centered[p_start:p_end]                                                 # Extract the segment of the signal corresponding to the estimated P wave region, which we will analyze to find the P wave peak and calculate the PR interval. We will only proceed with this analysis if the segment has enough samples to be meaningful.  
        if p_seg.size > 4:      
            p_peak_rel = int(np.argmax(np.abs(p_seg)))                                  # Find the relative index of the P wave peak within the estimated P wave segment by identifying the sample with the maximum absolute value, which is a common method for locating the P wave peak. We will use this relative index to calculate the PR interval from the P wave onset to the QRS complex onset.                          
            p_peak = p_start + p_peak_rel           
            p_on = max(p_start, p_peak - int(0.04 * fs_hz))                             # Estimate the P wave onset by looking back 40 ms from the P wave peak, ensuring we do not go below the P wave start. This is a common approach to approximate the start of the P wave, which typically begins shortly before the P wave peak. We will use this estimated P wave onset to calculate the PR interval.
            pr_vals.append((qrs_on - p_on) / fs_hz)                                     # Calculate the PR interval as the time difference between the estimated QRS complex onset and the estimated P wave onset, and append this value to the list of PR intervals. The PR interval is an important feature that reflects the conduction time from the atria to the ventricles in the heart.

        qrs_vals.append((qrs_off - qrs_on) / fs_hz)                                     # Calculate the QRS duration as the time difference between the estimated QRS complex offset and onset, and append this value to the list of QRS durations. The QRS duration is an important feature that reflects the time taken for ventricular depolarization.
        qrs_area_vals.append(float(np.trapz(np.abs(centered[qrs_on:qrs_off + 1])) / fs_hz))     # Calculate the area under the QRS complex by integrating the absolute value of the signal between the estimated QRS onset and offset, and append this value to the list of QRS areas. The QRS area is a feature that reflects the overall magnitude of ventricular depolarization.
        r_amp_vals.append(float(centered[r]))

        st_idx = min(x.size - 1, qrs_off + int(0.08 * fs_hz))                           # Estimate the ST segment index by looking forward 80 ms from the estimated QRS complex offset, ensuring we do not exceed the signal length. This is a common approach to approximate the location of the ST segment, which typically begins shortly after the QRS complex ends. We will use this estimated ST segment index to calculate the ST deviation from the baseline.
        st_vals.append(float(centered[st_idx]))

        t_start = min(x.size - 2, r + int(0.10 * fs_hz))
        t_end = min(x.size - 1, r + int(0.50 * fs_hz))
        if t_end - t_start > 6:
            t_seg = centered[t_start:t_end]                                             # Extract the segment of the signal corresponding to the estimated T wave region, which we will analyze to find the T wave peak and calculate the QT interval and T wave symmetry index. We will only proceed with this analysis if the segment has enough samples to be meaningful.
            t_peak = t_start + int(np.argmax(np.abs(t_seg)))                            # Find the index of the T wave peak within the estimated T wave segment by identifying the sample with the maximum absolute value, which is a common method for locating the T wave peak. We will use this index to estimate the T wave onset and offset, which are necessary for calculating the QT interval and the T wave symmetry index.
            t_on = max(t_start, t_peak - int(0.12 * fs_hz))
            t_off = min(t_end, t_peak + int(0.16 * fs_hz))
            qt_vals.append((t_off - qrs_on) / fs_hz)                                    # Calculate the QT interval as the time difference between the estimated T wave offset and the estimated QRS complex onset, and append this value to the list of QT intervals. The QT interval is an important feature that reflects the total time for ventricular depolarization and repolarization.

            rise = max((t_peak - t_on) / fs_hz, 1e-6)                                   # Calculate the rise time of the T wave as the time difference between the estimated T wave peak and the estimated T wave onset, ensuring that we do not divide by zero in subsequent calculations by using a small positive value as a lower bound. The rise time is a feature that reflects how quickly the T wave reaches its peak after it begins.
            decay = max((t_off - t_peak) / fs_hz, 1e-6)                                 # Calculate the decay time of the T wave as the time difference between the estimated T wave offset and the estimated T wave peak, ensuring that we do not divide by zero in subsequent calculations by using a small positive value as a lower bound. The decay time is a feature that reflects how quickly the T wave returns to baseline after reaching its peak.    
            sym_vals.append(float(rise / decay))                                        # Calculate the T wave symmetry index as the ratio of the rise time to the decay time, and append this value to the list of T wave symmetry indices. The T wave symmetry index is a feature that reflects the shape of the T wave, with values close to 1 indicating a more symmetric T wave and values significantly different from 1 indicating an asymmetric T wave.

    pr_interval = _nanmean_or_nan(pr_vals)
    qrs_duration = _nanmean_or_nan(qrs_vals)
    qt_interval = _nanmean_or_nan(qt_vals)
    qtc_bazett = (np.nan if (not np.isfinite(qt_interval) or not np.isfinite(mean_rr) or mean_rr <= 0)
                  else float(qt_interval / np.sqrt(mean_rr)))                           # Calculate the corrected QT interval (QTc) using Bazett's formula, which adjusts the QT interval based on the heart rate (mean RR interval). We return NaN if the QT interval or mean RR interval is not finite or if the mean RR interval is non-positive, as this would indicate an invalid QTc calculation.
    st_deviation = _nanmean_or_nan(st_vals)
    r_amplitude = _nanmean_or_nan(r_amp_vals)

    if rr.size >= 3:
        rr_time = np.cumsum(rr)                                                         # Create a time array corresponding to the RR intervals by taking the cumulative sum of the RR intervals, which gives us the time points at which each R peak occurs relative to the first R peak. We will use this time array to compute the power spectral density of the RR intervals for heart rate variability analysis.
        rr_time -= rr_time[0]
        if rr_time[-1] > 0:                                                             # Only attempt to compute the power spectral density if the total duration of the RR intervals is greater than 0 seconds, which ensures that we have a valid time series for analysis. If the duration is not greater than 0, we will return empty arrays for frequencies and PSD to indicate that the spectral analysis cannot be performed.
            fs_rr = 4.0
            t_uniform = np.arange(0, rr_time[-1], 1.0 / fs_rr)                          # Create a uniform time grid for interpolation of the RR intervals, starting from 0 and ending at the last time point of the RR intervals, with a spacing of 1/fs_rr seconds. This uniform grid is necessary for computing the power spectral density using Welch's method, which requires evenly spaced samples.
            if t_uniform.size >= 8:
                rr_interp = np.interp(t_uniform, rr_time, rr)
                freqs_rr, psd_rr = welch(rr_interp, fs=fs_rr, nperseg=min(256, rr_interp.size)) # Compute the power spectral density (PSD) of the interpolated RR intervals using Welch's method, which estimates the PSD by dividing the time series into overlapping segments, applying a window function, and averaging the periodograms of these segments. We use a segment length of 256 samples or the size of the interpolated RR series, whichever is smaller, to ensure that we have enough segments for a reliable estimate while also accommodating shorter time series. The resulting frequencies and PSD values will be used to calculate heart rate variability metrics such as LF and HF power.
            else:
                freqs_rr, psd_rr = np.array([]), np.array([])
        else:
            freqs_rr, psd_rr = np.array([]), np.array([])
    else:
        freqs_rr, psd_rr = np.array([]), np.array([])

    if psd_rr.size > 0:
        # Use the same robust integration strategy for LF/HF/total power.
        lf = _robust_band_power(freqs_rr, psd_rr, 0.04, 0.15)
        hf = _robust_band_power(freqs_rr, psd_rr, 0.15, 0.40)
        total_power = _robust_band_power(freqs_rr, psd_rr, 0.04, 0.40)
    else:
        lf, hf, total_power = np.nan, np.nan, np.nan

    lf_hf_ratio = np.nan if (not np.isfinite(lf) or not np.isfinite(hf) or hf == 0) else float(lf / hf)         # Calculate the LF/HF ratio, which is a commonly used metric in heart rate variability analysis to assess the balance between sympathetic and parasympathetic nervous system activity. We return NaN if either LF or HF power is not finite or if HF power is zero, as this would indicate an invalid LF/HF ratio calculation.
    qrs_area = _nanmean_or_nan(qrs_area_vals)
    t_wave_symmetry = _nanmean_or_nan(sym_vals)

    win = 8
    kernel = np.ones(win, dtype=float) / win
    smooth = np.convolve(centered, kernel, mode="same")
    detail = centered - smooth
    wavelet_energy_j = float(np.mean(detail ** 2))

    freqs_sig, psd_sig = welch(centered, fs=fs_hz, nperseg=min(1024, centered.size))
    spectral_entropy = _spectral_entropy_from_psd(psd_sig)                                                      # Calculate the spectral entropy of the ECG signal by first computing the power spectral density (PSD) of the centered signal using Welch's method and then applying the spectral entropy calculation to the PSD. The spectral entropy is a measure of the complexity of the signal, with higher values indicating a more complex and less predictable signal. We return this value as a float for consistency in our feature set.

    apen = _approximate_entropy(rr, m=2)                                                                        # Calculate the approximate entropy of the RR intervals with an embedding dimension of 2, which is a measure of the regularity and complexity of the heart rate time series. A lower approximate entropy indicates a more regular and predictable time series, while a higher value indicates a more irregular and complex time series. We return this value as a float for consistency in our feature set.                                                        
    sampen = _sample_entropy(rr, m=2)                                                                           # Calculate the sample entropy of the RR intervals with an embedding dimension of 2, which is another measure of the regularity and complexity of the heart rate time series. Similar to approximate entropy, a lower sample entropy indicates a more regular and predictable time series, while a higher value indicates a more irregular and complex time series. We return this value as a float for consistency in our feature set.
    dfa_alpha = _dfa_alpha(rr)                                                                                  # Calculate the detrended fluctuation analysis (DFA) alpha exponent for the RR intervals, which is a measure of the long-range correlation properties of the heart rate time series. A DFA alpha value around 0.5 indicates uncorrelated randomness, values between 0.5 and 1 indicate persistent long-range correlations, and values above 1 indicate non-stationary behavior. We return this value as a float for consistency in our feature set.
    higuchi_fd = _higuchi_fd(rr, kmax=6)                                                                        # Calculate the Higuchi fractal dimension of the RR intervals with a maximum k value of 6, which is a measure of the complexity of the heart rate time series. Higher values indicate more complex dynamics, while lower values indicate simpler dynamics. We return this value as a float for consistency in our feature set.

    return {
        "mean_heart_rate": mean_hr,
        "sdnn": sdnn,
        "rmssd": rmssd,
        "pnn50": pnn50,
        "pr_interval": pr_interval,
        "qrs_duration": qrs_duration,
        "qt_interval": qt_interval,
        "qtc_bazett": qtc_bazett,
        "st_deviation": st_deviation,
        "r_wave_amplitude": r_amplitude,
        "lf_power": lf,
        "hf_power": hf,
        "lf_hf_ratio": lf_hf_ratio,
        "total_power": total_power,
        "qrs_area": qrs_area,
        "t_wave_symmetry_index": t_wave_symmetry,
        "wavelet_energy_scale_j": wavelet_energy_j,
        "spectral_entropy": spectral_entropy,
        "approximate_entropy": apen,
        "sample_entropy": sampen,
        "dfa_scaling_exponent": dfa_alpha,
        "higuchi_fractal_dimension": higuchi_fd,
    }


sampling_rate_hz = 500
lead_idx = 1 #using only the second lead. (started counting at 0!)
feature_names = [
    "mean_heart_rate",
    "sdnn",
    "rmssd",
    "pnn50",
    "pr_interval",
    "qrs_duration",
    "qt_interval",
    "qtc_bazett",
    "st_deviation",
    "r_wave_amplitude",
    "lf_power",
    "hf_power",
    "lf_hf_ratio",
    "total_power",
    "qrs_area",
    "t_wave_symmetry_index",
    "wavelet_energy_scale_j",
    "spectral_entropy",
    "approximate_entropy",
    "sample_entropy",
    "dfa_scaling_exponent",
    "higuchi_fractal_dimension",
]

all_features = []
for i in range(ecg_data["signals"].shape[0]):
    n_samples = int(ecg_data["metadata"].iloc[i]["n_samples"])
    signal_lead = ecg_data["signals"][i, lead_idx, :n_samples]
    all_features.append(_extract_record_features(signal_lead, sampling_rate_hz))

metrics_df = pd.DataFrame(all_features, columns=feature_names).astype(np.float32)
ecg_data["metrics"] = metrics_df

metrics_df = metrics_df.drop(columns=["lf_hf_ratio"], errors="ignore")
inf_columns = metrics_df.columns[np.isinf(metrics_df.to_numpy()).any(axis=0)]
metrics_df = metrics_df.drop(columns=inf_columns, errors="ignore")
ecg_data["metrics"] = metrics_df

print(f"ecg_data['metrics'].shape = {ecg_data['metrics'].shape}")
display(ecg_data["metrics"].head())


ecg_data['metrics'].shape = (45152, 20)


,mean_heart_rate,sdnn,rmssd,pnn50,pr_interval,qrs_duration,qt_interval,qtc_bazett,st_deviation,r_wave_amplitude,lf_power,hf_power,total_power,qrs_area,t_wave_symmetry_index,wavelet_energy_scale_j,spectral_entropy,approximate_entropy,dfa_scaling_exponent,higuchi_fractal_dimension
0,152.936371,0.121130,0.149176,0.625000,0.191308,0.098308,0.480880,0.767744,-15.307693,277.807678,1.007677e-03,0.001234,0.002242,12.596847,3.879046,591.871033,3.951111,0.357093,0.600147,1.907407
1,122.712593,0.252227,0.451089,0.888889,0.153500,0.100000,0.491600,0.703041,10.350000,457.549988,1.432477e-03,0.003581,0.005014,14.711500,0.833910,329.976685,3.902532,0.361952,0.613373,2.082512
2,106.595604,0.257837,0.498560,1.000000,0.147294,0.100000,0.488471,0.651077,-8.647058,557.176453,4.655576e-06,0.000068,0.000072,19.863411,0.981718,235.196701,3.758010,0.002224,0.173563,3.054454
3,162.229614,0.005626,0.005699,0.000000,0.158222,0.099852,0.537308,0.883512,-0.296296,702.555542,7.673008e-08,0.000015,0.000015,19.269037,0.969494,1335.810913,3.720086,0.210014,1.345857,1.739385
4,114.576706,0.248756,0.487043,1.000000,0.162105,0.100000,0.472947,0.653560,14.947369,488.947357,5.326216e-05,0.000093,0.000147,17.656948,1.633797,183.325226,3.568694,0.085218,0.063925,2.843646


In [4]:
# CELL GUIDE:
# Purpose: Run `ecg_data["metrics"].isna().sum() #check missing values` and continue downstream processing.
# Workflow:
# 1. ecg_data["metrics"].isna().sum() #check missing values
# 2. side note: codex did not check for missing values and used the full 20 features in the beginning.
# 3. dropping all columns with missing values would have been too agressive.
# 4. here, I decided to keep the first 18 and impute with the median
# 5. dfa_scaling_exponent and higuchi_fractal_dimension had the most missing values, so I dropped those two features

ecg_data["metrics"].isna().sum() #check missing values
# side note: codex did not check for missing values and used the full 20 features in the beginning. 
# dropping all columns with missing values would have been too agressive.
# here, I decided to keep the first 18 and impute with the median
# dfa_scaling_exponent and higuchi_fractal_dimension had the most missing values, so I dropped those two features

mean_heart_rate                 1
sdnn                            2
rmssd                           2
pnn50                           2
pr_interval                     0
qrs_duration                    0
qt_interval                     0
qtc_bazett                      1
st_deviation                    0
r_wave_amplitude                0
lf_power                        7
hf_power                        7
total_power                     7
qrs_area                        0
t_wave_symmetry_index           0
wavelet_energy_scale_j          0
spectral_entropy                0
approximate_entropy             7
dfa_scaling_exponent         1149
higuchi_fractal_dimension    5572
dtype: int64

In [5]:
# CELL GUIDE:
# Purpose: Prepare the shared 18-feature modeling matrix: drop two specified features, then median-impute remaining missing values.
# Workflow:
# 1. Prepare the shared 18-feature modeling matrix: drop two specified features, then median-impute remaining missing values.
# 2. from sklearn.impute import SimpleImputer
# 3. shared_feature_drop_cols_18F = ["higuchi_fractal_dimension", "dfa_scaling_exponent"]
# 4. Shared 18-feature base (used by SGD_18F, XGB_18F, and later SGD_13C feature selection after PCA).
# 5. shared_metrics_model_18F_df = ecg_data["metrics"].copy()
# 6. shared_metrics_model_18F_df = shared_metrics_model_18F_df.drop(columns=shared_feature_drop_cols_18F, errors="ignore")

# Prepare the shared 18-feature modeling matrix: drop two specified features, then median-impute remaining missing values.
from sklearn.impute import SimpleImputer

shared_feature_drop_cols_18F = ["higuchi_fractal_dimension", "dfa_scaling_exponent"]

# Shared 18-feature base (used by SGD_18F, XGB_18F, and later SGD_13C feature selection after PCA).
shared_metrics_model_18F_df = ecg_data["metrics"].copy()
shared_metrics_model_18F_df = shared_metrics_model_18F_df.drop(columns=shared_feature_drop_cols_18F, errors="ignore")
shared_metrics_model_18F_df = shared_metrics_model_18F_df.replace([np.inf, -np.inf], np.nan)

# NOTE: Fill missing values to ensure downstream estimators receive dense numeric input.
shared_metrics_model_18F_imputer = SimpleImputer(strategy="median")
shared_metrics_model_18F_df = pd.DataFrame(
    shared_metrics_model_18F_imputer.fit_transform(shared_metrics_model_18F_df),
    columns=shared_metrics_model_18F_df.columns,
    index=shared_metrics_model_18F_df.index,
).astype(np.float32).reset_index(drop=True)
shared_metadata_model_18F_df = ecg_data["metadata"].reset_index(drop=True).copy()

# Build multilabel targets from aligned metadata dx code lists (not textual diagnoses) once for all downstream models.
shared_dx_code_label_sets_18F = shared_metadata_model_18F_df["dx_code_list"].apply(
    lambda codes: sorted({str(code).strip() for code in codes if str(code).strip()})
)

ecg_data["metrics_model_18F"] = shared_metrics_model_18F_df
ecg_data["metadata_model_18F"] = shared_metadata_model_18F_df
ecg_data["dx_code_label_sets_model_18F"] = shared_dx_code_label_sets_18F

print(f"Dropped feature columns for shared 18F prep: {[col for col in shared_feature_drop_cols_18F if col in ecg_data['metrics'].columns]}")
print(f"Shared 18F modeling metrics shape: {ecg_data['metrics_model_18F'].shape}")
print(f"Shared 18F modeling metadata shape: {ecg_data['metadata_model_18F'].shape}")
print(f"Shared dx_code label-set length: {len(ecg_data['dx_code_label_sets_model_18F'])}")


Dropped feature columns for shared 18F prep: ['higuchi_fractal_dimension', 'dfa_scaling_exponent']
Shared 18F modeling metrics shape: (45152, 18)
Shared 18F modeling metadata shape: (45152, 9)
Shared dx_code label-set length: 45152


In [29]:
# CELL GUIDE:
# Purpose: Correlation heatmap for the engineered 18-feature matrix (styled like conditional-probability heatmap).
# Workflow:
# 1. Correlation heatmap for the engineered 18-feature matrix (styled like conditional-probability heatmap).
# 2. import plotly.graph_objects as go
# 3. from plotly.subplots import make_subplots
# 4. if "metrics_model_18F" not in ecg_data
# 5. corr_df_18f = ecg_data["metrics_model_18F"].copy()
# 6. corr_matrix_18f = corr_df_18f.corr(numeric_only=True)

# Correlation heatmap for the engineered 18-feature matrix (styled like conditional-probability heatmap).
import plotly.graph_objects as go
from plotly.subplots import make_subplots

if "metrics_model_18F" not in ecg_data:
    raise RuntimeError("Run the shared 18-feature modeling prep cell first.")

corr_df_18f = ecg_data["metrics_model_18F"].copy()
corr_matrix_18f = corr_df_18f.corr(numeric_only=True)
feature_names = corr_matrix_18f.columns.tolist()
feature_ids = list(range(1, len(feature_names) + 1))

# Style configuration aligned with the conditional-probability heatmap.
text_gray = graph_formatting.get("color_text", "#b8b8b8") if "graph_formatting" in globals() else "#b8b8b8"
bg_black = graph_formatting.get("color_background", "#000000") if "graph_formatting" in globals() else "#000000"
font_family = graph_formatting.get("font_family", "Helvetica Neue") if "graph_formatting" in globals() else "Helvetica Neue"
axis_size = graph_formatting.get("font_size_axis", 12) if "graph_formatting" in globals() else 12
base_size = graph_formatting.get("font_size_base", 16) if "graph_formatting" in globals() else 16

red_white_scale = [[0.0, "#ffffff"], [1.0, "#000000"]]
# Correlation in [-1,1] mapped to [0,1] for black-white scale.
corr_plot_values = (corr_matrix_18f.values + 1.0) / 2.0

feature_index_table = pd.DataFrame({
    "Index": feature_ids,
    "Feature": feature_names,
})

feature_labels_array = np.array(feature_names)
y_feature_grid = np.repeat(feature_labels_array[:, None], len(feature_names), axis=1)
x_feature_grid = np.repeat(feature_labels_array[None, :], len(feature_names), axis=0)
feature_pair_customdata = np.dstack((y_feature_grid, x_feature_grid, corr_matrix_18f.values))

table_border_color = "rgba(200, 200, 200, 0.35)"
table_font_size = max(9, axis_size - 2)
table_row_height = 23.2
table_domain_end = 0.42
total_table_rows = len(feature_ids) + 1  # +1 header
table_top = 1.0
table_bottom = 0.0
row_h = (table_top - table_bottom) / total_table_rows

corr_heatmap = make_subplots(
    rows=1,
    cols=2,
    specs=[[{"type": "table"}, {"type": "heatmap"}]],
    column_widths=[0.1, 0.1],
    horizontal_spacing=0.02,
)

corr_heatmap.add_trace(
    go.Table(
        columnwidth=[0.16, 0.84],
        header=dict(
            values=["Index", "Feature"],
            fill_color=bg_black,
            line_color="rgba(0,0,0,0)",
            line_width=0,
            font=dict(family=font_family, size=table_font_size, color=text_gray),
            align=["center", "left"],
            height=25,
        ),
        cells=dict(
            values=[feature_index_table["Index"], feature_index_table["Feature"]],
            fill_color=bg_black,
            line_color="rgba(0,0,0,0)",
            line_width=0,
            font=dict(family=font_family, size=table_font_size, color=text_gray),
            align=["center", "left"],
            height=table_row_height,
        ),
    ),
    row=1,
    col=1,
)

table_separator_rows = list(range(0, total_table_rows + 1))
for row_idx in table_separator_rows:
    y_pos = table_top - row_idx * 0.053
    corr_heatmap.add_shape(
        type="line",
        x0=0,
        x1=table_domain_end,
        y0=y_pos,
        y1=y_pos,
        xref="paper",
        yref="paper",
        line=dict(color=table_border_color, width=0.5),
        layer="above",
    )

corr_heatmap.add_trace(
    go.Heatmap(
        z=corr_plot_values,
        x=feature_ids,
        y=feature_ids,
        customdata=feature_pair_customdata,
        colorscale=red_white_scale,
        zmin=0,
        zmax=1,
        colorbar=dict(
            title="Correlation",
            titlefont=dict(family=font_family, size=axis_size, color=text_gray),
            tickfont=dict(family=font_family, size=axis_size, color=text_gray),
            tickvals=[0, 0.5, 1],
            ticktext=["-1", "0", "1"],
        ),
        hovertemplate=(
            "Feature A [%{y}] %{customdata[0]}<br>"
            "Feature B [%{x}] %{customdata[1]}<br>"
            "Pearson Correlation: %{customdata[2]:.3f}<extra></extra>"
        ),
    ),
    row=1,
    col=2,
)

# NOTE: Configure x-axis semantics and formatting for accurate interpretation.
corr_heatmap.update_xaxes(
    title_text="Feature Index",
    title_font=dict(family=font_family, size=axis_size, color=text_gray),
    tickmode="array",
    tickvals=feature_ids,
    ticktext=feature_ids,
    tickfont=dict(family=font_family, size=axis_size, color=text_gray),
    showgrid=False,
    zeroline=False,
    row=1,
    col=2,
)
# NOTE: Configure y-axis semantics and formatting for accurate interpretation.
corr_heatmap.update_yaxes(
    title_text="Feature Index",
    title_font=dict(family=font_family, size=axis_size, color=text_gray),
    tickmode="array",
    tickvals=feature_ids,
    ticktext=feature_ids,
    tickfont=dict(family=font_family, size=axis_size, color=text_gray),
    autorange="reversed",
    showgrid=False,
    zeroline=False,
    row=1,
    col=2,
)

# NOTE: Apply global styling (title, typography, spacing, colors) for readability consistency.
corr_heatmap.update_layout(
    title=dict(
        text=(
            "<span style='font-size:30px;font-weight:bold;'>    Time-Domain Metrics and Band Powers Correlate Most</span>"
            "<br><span style='font-size:20px;font-weight: normal;'>      Pairwise Pearson Correlations Of Engineered Features</span>"
        ),
        x=0,
        xanchor="left",
        pad=dict(t=50),
    ),
    paper_bgcolor=bg_black,
    plot_bgcolor=bg_black,
    font=dict(family=font_family, size=base_size, color=text_gray),
    margin=dict(
        t=graph_formatting.get("margin_top", 170) if "graph_formatting" in globals() else 170,
        r=graph_formatting.get("margin_right", 80) if "graph_formatting" in globals() else 80,
        b=graph_formatting.get("margin_bottom", 80) if "graph_formatting" in globals() else 80,
        l=graph_formatting.get("margin_left", 80) if "graph_formatting" in globals() else 80,
    ),
    height=696,
)

corr_heatmap.show()

# **SGD_18F**


<!-- CELL NOTE -->
> **Cell Note:** This section documents **SGD_18F** and provides context for the following code/results.
> Review it before executing downstream cells so assumptions and outputs remain aligned.


In [6]:
# CELL GUIDE:
# Purpose: Prepare the SGD_18F train/test split from the shared 18-feature median-imputed matrix (unmodified columns after the two specified drops).
# Workflow:
# 1. Prepare the SGD_18F train/test split from the shared 18-feature median-imputed matrix (unmodified columns after the t...
# 2. from sklearn.model_selection import train_test_split
# 3. from sklearn.preprocessing import MultiLabelBinarizer
# 4. required_keys = ["metrics_model_18F", "metadata_model_18F", "dx_code_label_sets_model_18F"]
# 5. missing_keys = [k for k in required_keys if k not in ecg_data]
# 6. if missing_keys

# Prepare the SGD_18F train/test split from the shared 18-feature median-imputed matrix (unmodified columns after the two specified drops).
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer

required_keys = ["metrics_model_18F", "metadata_model_18F", "dx_code_label_sets_model_18F"]
missing_keys = [k for k in required_keys if k not in ecg_data]
if missing_keys:
    raise RuntimeError(
        f"Run the shared feature-preparation cell after ecg_data['metrics'].isna().sum() first. Missing: {missing_keys}"
    )

SGD_18F_metrics_model_df = ecg_data["metrics_model_18F"].copy()
SGD_18F_metadata_model_df = ecg_data["metadata_model_18F"].copy()
SGD_18F_y_labels = ecg_data["dx_code_label_sets_model_18F"].copy()

print(f"SGD_18F modeling metrics shape: {SGD_18F_metrics_model_df.shape}")
print(f"SGD_18F modeling metadata shape: {SGD_18F_metadata_model_df.shape}")
print(f"SGD_18F feature count: {SGD_18F_metrics_model_df.shape[1]}")

SGD_18F_X = SGD_18F_metrics_model_df.copy()
# NOTE: Convert per-sample label sets into a multi-hot indicator matrix.
SGD_18F_mlb = MultiLabelBinarizer()
SGD_18F_y = SGD_18F_mlb.fit_transform(SGD_18F_y_labels)

SGD_18F_X_train, SGD_18F_X_test, SGD_18F_y_train, SGD_18F_y_test = train_test_split(
    SGD_18F_X,
    SGD_18F_y,
    test_size=0.20,
    random_state=42,
    shuffle=True,
)


SGD_18F modeling metrics shape: (45152, 18)
SGD_18F modeling metadata shape: (45152, 9)
SGD_18F feature count: 18


In [7]:
# CELL GUIDE:
# Purpose: Hyperparameter tune One-vs-Rest SGD_18F on the training split only using Ray Tune + cross-validation.
# Workflow:
# 1. Hyperparameter tune One-vs-Rest SGD_18F on the training split only using Ray Tune + cross-validation.
# 2. import ray
# 3. import random
# 4. from ray import tune
# 5. from ray.tune.schedulers import ASHAScheduler
# 6. from sklearn.linear_model import SGDClassifier

# Hyperparameter tune One-vs-Rest SGD_18F on the training split only using Ray Tune + cross-validation.
import ray
import random
from ray import tune
from ray.tune.schedulers import ASHAScheduler
from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import cross_validate
from sklearn.multiclass import OneVsRestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

SGD_18F_X_train_tune = (
    SGD_18F_X_train.to_numpy(dtype=np.float32)
    if hasattr(SGD_18F_X_train, "to_numpy")
    else np.asarray(SGD_18F_X_train, dtype=np.float32)
)
SGD_18F_y_train_tune = np.asarray(SGD_18F_y_train)

SGD_18F_ray_cv_folds = 3
SGD_18F_num_ray_trials = 30

param_dist = {
    "alpha": tune.loguniform(1e-6, 1e-1),
    "loss": tune.choice(["hinge", "log_loss", "modified_huber"]),
    "penalty": tune.choice(["l1", "l2", "elasticnet"]),
    "learning_rate": tune.choice(["optimal", "adaptive", "constant", "invscaling"]),
}

SGD_18F_scoring = {
    "precision_micro": "precision_micro",
    "precision_macro": "precision_macro",
    "recall_micro": "recall_micro",
    "f1_micro": "f1_micro",
    "f1_macro": "f1_macro",
}


def SGD_18F_ray_sgd_cv_objective(config, X_train_arr, y_train_arr, cv_splits):
    SGD_18F_model = make_pipeline(
        StandardScaler(),
        OneVsRestClassifier(
            SGDClassifier(
                loss=config["loss"],
                alpha=float(config["alpha"]),
                penalty=config["penalty"],
                learning_rate=config["learning_rate"],
                eta0=0.01,
                class_weight="balanced",
                max_iter=2000,
                tol=1e-3,
                random_state=42,
            )
        ),
    )

# NOTE: Run cross-validation to aggregate robust performance estimates across folds.
    SGD_18F_cv_scores = cross_validate(
        SGD_18F_model,
        X_train_arr,
        y_train_arr,
        cv=cv_splits,
        scoring=SGD_18F_scoring,
        n_jobs=1,
        error_score="raise",
    )

    tune.report({
        "precision_micro": float(np.mean(SGD_18F_cv_scores["test_precision_micro"])),
        "precision_macro": float(np.mean(SGD_18F_cv_scores["test_precision_macro"])),
        "recall_micro": float(np.mean(SGD_18F_cv_scores["test_recall_micro"])),
        "f1_micro": float(np.mean(SGD_18F_cv_scores["test_f1_micro"])),
        "f1_macro": float(np.mean(SGD_18F_cv_scores["test_f1_macro"])),
    })


np.random.seed(42)
random.seed(42)

try:
    from ray.tune.search.optuna import OptunaSearch
    SGD_18F_search_alg = OptunaSearch(seed=42)
    SGD_18F_search_backend = "optuna"
except Exception:
    from ray.tune.search.hyperopt import HyperOptSearch
    SGD_18F_search_alg = HyperOptSearch(random_state_seed=42)
    SGD_18F_search_backend = "hyperopt"

SGD_18F_scheduler = ASHAScheduler(
    max_t=1,
    grace_period=1,
    reduction_factor=2,
)
print(f"SGD_18F search backend: {SGD_18F_search_backend}")

if ray.is_initialized():
    ray.shutdown()
ray.init(ignore_reinit_error=True, include_dashboard=False, log_to_driver=False)

# NOTE: Launch Ray Tune hyperparameter search over the configured trial space.
SGD_18F_ray_analysis = tune.run(
    tune.with_parameters(
        SGD_18F_ray_sgd_cv_objective,
        X_train_arr=SGD_18F_X_train_tune,
        y_train_arr=SGD_18F_y_train_tune,
        cv_splits=SGD_18F_ray_cv_folds,
    ),
    config=param_dist,
    num_samples=SGD_18F_num_ray_trials,
    metric="f1_macro", # using f1_macro as the primary metric for hyperparameter optimization, as it provides a balanced measure of precision and recall across all classes in the multilabel classification task. This choice encourages the selection of models that perform well across all classes rather than optimizing for just the most common classes, which is important in a medical diagnosis context where rare conditions may be clinically significant.
    mode="max",
    resources_per_trial={"cpu": 1},
    search_alg=SGD_18F_search_alg,
    scheduler=SGD_18F_scheduler,
    verbose=1,
)

SGD_18F_ray_tune_results_df = SGD_18F_ray_analysis.dataframe().copy().rename(
    columns={
        "config/alpha": "alpha",
        "config/loss": "loss",
        "config/penalty": "penalty",
        "config/learning_rate": "learning_rate",
    }
)

for metric_name in ["precision_micro", "precision_macro", "recall_micro", "f1_micro", "f1_macro"]:
    if metric_name not in SGD_18F_ray_tune_results_df.columns:
        for alt_col in (f"last_result/{metric_name}", f"last_result.{metric_name}"):
            if alt_col in SGD_18F_ray_tune_results_df.columns:
                SGD_18F_ray_tune_results_df[metric_name] = SGD_18F_ray_tune_results_df[alt_col]
                break

SGD_18F_keep_columns = [
    col
    for col in [
        "trial_id",
        "alpha",
        "loss",
        "penalty",
        "learning_rate",
        "precision_micro",
        "precision_macro",
        "recall_micro",
        "f1_micro",
        "f1_macro",
        "training_iteration",
        "time_total_s",
    ]
    if col in SGD_18F_ray_tune_results_df.columns
]
SGD_18F_ray_tune_results_df = SGD_18F_ray_tune_results_df[SGD_18F_keep_columns].copy()

SGD_18F_sort_cols = [col for col in ["f1_macro", "f1_micro"] if col in SGD_18F_ray_tune_results_df.columns]
SGD_18F_ray_tune_results_df = SGD_18F_ray_tune_results_df.sort_values(
    SGD_18F_sort_cols,
    ascending=[False] * len(SGD_18F_sort_cols),
    na_position="last",
).reset_index(drop=True)

SGD_18F_best_by_metric_rows = []
for SGD_18F_metric in ["f1_macro", "f1_micro", "precision_macro", "precision_micro"]:
    if SGD_18F_metric in SGD_18F_ray_tune_results_df.columns and SGD_18F_ray_tune_results_df[SGD_18F_metric].notna().any():
        SGD_18F_best_row = SGD_18F_ray_tune_results_df.loc[SGD_18F_ray_tune_results_df[SGD_18F_metric].idxmax()].copy()
        SGD_18F_best_by_metric_rows.append(
            {
                "selection_metric": SGD_18F_metric,
                "trial_id": SGD_18F_best_row.get("trial_id"),
                "alpha": SGD_18F_best_row.get("alpha"),
                "loss": SGD_18F_best_row.get("loss"),
                "penalty": SGD_18F_best_row.get("penalty"),
                "learning_rate": SGD_18F_best_row.get("learning_rate"),
                "precision_micro": SGD_18F_best_row.get("precision_micro"),
                "precision_macro": SGD_18F_best_row.get("precision_macro"),
                "f1_micro": SGD_18F_best_row.get("f1_micro"),
                "f1_macro": SGD_18F_best_row.get("f1_macro"),
            }
        )
SGD_18F_best_configs_by_metric_df = pd.DataFrame(SGD_18F_best_by_metric_rows)

SGD_18F_probabilistic_results_df = SGD_18F_ray_tune_results_df[
    SGD_18F_ray_tune_results_df["loss"].astype(str).isin(["log_loss", "modified_huber"])
].copy()
if SGD_18F_probabilistic_results_df.empty:
    raise RuntimeError("No probabilistic SGD_18F Ray Tune trials found (required for probability cutoff tuning).")

SGD_18F_selected_row = SGD_18F_probabilistic_results_df.sort_values(
    ["f1_macro", "f1_micro", "precision_macro", "precision_micro"],
    ascending=[False, False, False, False],
    na_position="last",
).iloc[0]
SGD_18F_selected_trial_id = str(SGD_18F_selected_row["trial_id"])
SGD_18F_selected_config = {
    "alpha": float(SGD_18F_selected_row["alpha"]),
    "loss": str(SGD_18F_selected_row["loss"]),
    "penalty": str(SGD_18F_selected_row["penalty"]),
    "learning_rate": str(SGD_18F_selected_row["learning_rate"]),
}

print("SGD_18F best-performing Ray Tune configurations by metric:")
print(SGD_18F_best_configs_by_metric_df[[
    "selection_metric",
    "trial_id",
    "loss",
    "penalty",
    "learning_rate",
    "precision_micro",
    "precision_macro",
    "f1_micro",
    "f1_macro",
]].round(4))
print("\nSGD_18F selected probabilistic configuration for cutoff tuning/test evaluation:")
print({"trial_id": SGD_18F_selected_trial_id, **SGD_18F_selected_config})

# NOTE: Persist artifacts in ecg_data so later cells can reuse them without recomputation.
ecg_data.setdefault("ray_tune_sgd_18F", {})["results"] = SGD_18F_ray_tune_results_df.copy()
# NOTE: Persist artifacts in ecg_data so later cells can reuse them without recomputation.
ecg_data.setdefault("ray_tune_sgd_18F", {})["best_by_metric"] = SGD_18F_best_configs_by_metric_df.copy()
# NOTE: Persist artifacts in ecg_data so later cells can reuse them without recomputation.
ecg_data.setdefault("ray_tune_sgd_18F", {})["selected_trial_id"] = SGD_18F_selected_trial_id
# NOTE: Persist artifacts in ecg_data so later cells can reuse them without recomputation.
ecg_data.setdefault("ray_tune_sgd_18F", {})["selected_config"] = dict(SGD_18F_selected_config)
# NOTE: Persist artifacts in ecg_data so later cells can reuse them without recomputation.
ecg_data.setdefault("ray_tune_sgd_18F", {})["cv_folds"] = SGD_18F_ray_cv_folds
# NOTE: Persist artifacts in ecg_data so later cells can reuse them without recomputation.
ecg_data.setdefault("ray_tune_sgd_18F", {})["num_trials"] = SGD_18F_num_ray_trials

SGD_18F_ray_tune_results_df


2026-03-02 14:55:53,273	INFO tune.py:1009 -- Wrote the latest version of all result files and experiment state to '/Users/felixaugustin/ray_results/SGD_18F_ray_sgd_cv_objective_2026-03-02_14-35-20' in 0.0200s.
2026-03-02 14:55:53,298	INFO tune.py:1041 -- Total run time: 1232.44 seconds (1232.35 seconds for the tuning loop).


SGD_18F best-performing Ray Tune configurations by metric:
  selection_metric  trial_id      loss penalty learning_rate  precision_micro  \
0         f1_macro  fdc9a784     hinge      l1      adaptive           0.0675   
1         f1_micro  96991386  log_loss      l1      adaptive           0.0699   
2  precision_macro  4e6c655b     hinge      l1      adaptive           0.0673   
3  precision_micro  96991386  log_loss      l1      adaptive           0.0699   

   precision_macro  f1_micro  f1_macro  
0           0.0524    0.1238    0.0788  
1           0.0515    0.1277    0.0781  
2           0.0524    0.1236    0.0788  
3           0.0515    0.1277    0.0781  

SGD_18F selected probabilistic configuration for cutoff tuning/test evaluation:
{'trial_id': '96991386', 'alpha': 1.9616460353882896e-05, 'loss': 'log_loss', 'penalty': 'l1', 'learning_rate': 'adaptive'}


,trial_id,alpha,loss,penalty,learning_rate,precision_micro,precision_macro,recall_micro,f1_micro,f1_macro,training_iteration,time_total_s
0,fdc9a784,0.000059,hinge,l1,adaptive,0.067488,0.052391,0.747632,0.123800,0.078849,1,141.615721
1,d5416fc5,0.000020,hinge,elasticnet,adaptive,0.067794,0.052389,0.746865,0.124303,0.078784,1,252.097391
2,4e6c655b,0.000297,hinge,l1,adaptive,0.067344,0.052392,0.747248,0.123552,0.078771,1,255.316423
3,f1e7ae27,0.000053,hinge,l1,adaptive,0.067499,0.052292,0.748911,0.123834,0.078762,1,188.375641
4,9b9b63e0,0.000040,hinge,l1,adaptive,0.067466,0.052291,0.747959,0.123766,0.078722,1,153.028241
5,11514293,0.000045,hinge,l1,adaptive,0.067365,0.052272,0.747290,0.123587,0.078659,1,208.195476
6,86e4dec7,0.001526,hinge,l1,adaptive,0.066565,0.052313,0.745517,0.122217,0.078498,1,192.889385
7,96991386,0.000020,log_loss,l1,adaptive,0.069863,0.051491,0.744262,0.127736,0.078074,1,322.282929
8,13fb269d,0.000019,log_loss,l1,adaptive,0.069709,0.051489,0.744205,0.127477,0.078068,1,313.146032
9,e6ee910d,0.000003,log_loss,l2,adaptive,0.069574,0.051477,0.744134,0.127250,0.078035,1,227.878707


In [46]:
# CELL GUIDE:
# Purpose: SGD_18F target metric over time (chronological).
# Workflow:
# 1. SGD_18F target metric over time (chronological)
# 2. import numpy as np
# 3. import pandas as pd
# 4. import plotly.graph_objects as go
# 5. SGD_18F_results_source = None
# 6. if "SGD_18F_ray_tune_results_df" in globals()

# SGD_18F target metric over time (chronological)
import numpy as np
import pandas as pd
import plotly.graph_objects as go

SGD_18F_results_source = None
if "SGD_18F_ray_tune_results_df" in globals():
    SGD_18F_results_source = SGD_18F_ray_tune_results_df
elif isinstance(ecg_data, dict):
    SGD_18F_results_source = (ecg_data.get("ray_tune_sgd_18F") or {}).get("results")

if SGD_18F_results_source is None:
    raise RuntimeError("SGD_18F Ray results were not found. Run the SGD_18F tuning cell first.")

SGD_18F_time_df = SGD_18F_results_source.copy()
if SGD_18F_time_df.empty:
    raise RuntimeError("SGD_18F Ray results are empty.")

SGD_18F_target_metric = "f1_macro"
if SGD_18F_target_metric not in SGD_18F_time_df.columns:
    if "f1_micro" in SGD_18F_time_df.columns:
        SGD_18F_target_metric = "f1_micro"
    else:
        raise KeyError("Neither f1_macro nor f1_micro is available in SGD_18F Ray results.")

SGD_18F_time_sort_col = None
if "timestamp" in SGD_18F_time_df.columns:
    SGD_18F_time_df["_chrono"] = pd.to_numeric(SGD_18F_time_df["timestamp"], errors="coerce")
    SGD_18F_time_sort_col = "_chrono"
elif "date" in SGD_18F_time_df.columns:
    SGD_18F_time_df["_chrono"] = pd.to_datetime(SGD_18F_time_df["date"], errors="coerce")
    SGD_18F_time_sort_col = "_chrono"
elif "time_total_s" in SGD_18F_time_df.columns:
    SGD_18F_time_df["_chrono"] = pd.to_numeric(SGD_18F_time_df["time_total_s"], errors="coerce")
    SGD_18F_time_sort_col = "_chrono"
elif "training_iteration" in SGD_18F_time_df.columns:
    SGD_18F_time_df["_chrono"] = pd.to_numeric(SGD_18F_time_df["training_iteration"], errors="coerce")
    SGD_18F_time_sort_col = "_chrono"

if SGD_18F_time_sort_col is not None:
    SGD_18F_time_df = SGD_18F_time_df.sort_values(SGD_18F_time_sort_col, ascending=True, na_position="last")

SGD_18F_time_df = SGD_18F_time_df.reset_index(drop=True)
SGD_18F_time_df["trial_order"] = np.arange(1, len(SGD_18F_time_df) + 1)
SGD_18F_time_df["trial_id"] = SGD_18F_time_df.get("trial_id", "").astype(str)

SGD_18F_plot_style = graph_formatting if "graph_formatting" in globals() else {
    "font_family": "Helvetica Neue",
    "font_size_base": 16,
    "font_size_axis": 12,
    "color_text": "#b8b8b8",
    "color_background": "#000000",
    "margin_top": 170,
    "margin_right": 80,
    "margin_bottom": 80,
    "margin_left": 80,
}
SGD_18F_text = SGD_18F_plot_style.get("color_text", "#b8b8b8")
SGD_18F_bg = SGD_18F_plot_style.get("color_background", "#000000")
SGD_18F_color = SERIES_COLORS.get("f1_macro", "#636EFA") if "SERIES_COLORS" in globals() else "#636EFA"

SGD_18F_time_fig = go.Figure(
    data=[
        go.Scatter(
            x=SGD_18F_time_df["trial_order"],
            y=SGD_18F_time_df[SGD_18F_target_metric],
            mode="lines+markers",
            line=dict(width=2.2, color=SGD_18F_color),
            marker=dict(size=6, color=SGD_18F_color),
            customdata=np.column_stack([SGD_18F_time_df["trial_id"]]),
            hovertemplate=(
                "Chronological Order: %{x}<br>"
                "Trial ID: %{customdata[0]}<br>"
                f"{SGD_18F_target_metric.upper()}: %{{y:.4f}}<extra></extra>"
            ),
            name="SGD_18F",
        )
    ]
)

# NOTE: Configure x-axis semantics and formatting for accurate interpretation.
SGD_18F_time_fig.update_xaxes(
    title_text="Chronological Trial Order",
    title_font=dict(family=SGD_18F_plot_style["font_family"], size=SGD_18F_plot_style["font_size_axis"], color=SGD_18F_text),
    tickfont=dict(family=SGD_18F_plot_style["font_family"], size=SGD_18F_plot_style["font_size_axis"], color=SGD_18F_text),
    showgrid=True,
    gridcolor="#1f1f1f",
    gridwidth=0.2,
)

# NOTE: Configure y-axis semantics and formatting for accurate interpretation.
SGD_18F_time_fig.update_yaxes(
    title_font=dict(family=SGD_18F_plot_style["font_family"], size=SGD_18F_plot_style["font_size_axis"], color=SGD_18F_text),
    tickfont=dict(family=SGD_18F_plot_style["font_family"], size=SGD_18F_plot_style["font_size_axis"], color=SGD_18F_text),
    showgrid=True,
    gridcolor="#1f1f1f",
    gridwidth=0.2,
)

# NOTE: Apply global styling (title, typography, spacing, colors) for readability consistency.
SGD_18F_time_fig.update_layout(
    title=dict(
        text=(
            "<span style='font-size:30px;font-weight:bold;'>    Macro-F1 Improves Early, Then Plateaus Under Successive Halving</span>"
            "<br><span style='font-size:20px;font-weight: normal;'>      SGD_18F Target Metric - F1 Macro During Each Ray Tune Trial</span>"
        ),
        x=0,
        xanchor="left",
        pad=dict(t=50),
    ),
    paper_bgcolor=SGD_18F_bg,
    plot_bgcolor=SGD_18F_bg,
    font=dict(family=SGD_18F_plot_style["font_family"], size=SGD_18F_plot_style["font_size_base"], color=SGD_18F_text),
    hoverlabel=dict(font=dict(family=SGD_18F_plot_style["font_family"], size=SGD_18F_plot_style["font_size_axis"], color=SGD_18F_text)),
    legend=STANDARD_LEGEND if "STANDARD_LEGEND" in globals() else dict(orientation="h", yanchor="top", y=1.02, xanchor="left", x=0),
    margin=dict(
        t=SGD_18F_plot_style["margin_top"],
        r=SGD_18F_plot_style["margin_right"],
        b=SGD_18F_plot_style["margin_bottom"],
        l=SGD_18F_plot_style["margin_left"],
    ),
    height=700,
)

SGD_18F_time_fig.show()


In [ ]:
# CELL GUIDE:
# Purpose: Cross-validate probability cutoffs for the selected Ray-tuned SGD_18F configuration (compute-only).
# Workflow:
# 1. Cross-validate probability cutoffs for the selected Ray-tuned SGD_18F configuration (compute-only).
# 2. from sklearn.metrics import f1_score, precision_score, recall_score
# 3. from sklearn.model_selection import cross_val_predict
# 4. SGD_18F_threshold_cv_folds = 5
# 5. SGD_18F_probability_cutoffs = np.round(np.linspace(0.05, 0.95, 19), 2)
# 6. SGD_18F_selected_trial_id = '96991386'

# Cross-validate probability cutoffs for the selected Ray-tuned SGD_18F configuration (compute-only).
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.model_selection import cross_val_predict

SGD_18F_threshold_cv_folds = 5
# NOTE: Define a deterministic threshold grid for cutoff-sensitivity analysis.
SGD_18F_probability_cutoffs = np.round(np.linspace(0.05, 0.95, 19), 2)

SGD_18F_selected_trial_id = '96991386'
SGD_18F_selected_config = {'alpha': 1.9616460353882896e-05,
 'loss': 'log_loss',
 'penalty': 'l1',
 'learning_rate': 'adaptive'}

required_config_keys = ["alpha", "loss", "penalty", "learning_rate"]
missing_config_keys = [k for k in required_config_keys if k not in SGD_18F_selected_config or SGD_18F_selected_config[k] is None]
if missing_config_keys:
    raise KeyError(f"Missing SGD_18F selected configuration keys: {missing_config_keys}")

if SGD_18F_selected_config["loss"] not in {"log_loss", "modified_huber"}:
    raise ValueError(
        f"predict_proba is not supported for SGDClassifier(loss={SGD_18F_selected_config['loss']!r}). "
        "Use a probabilistic loss (e.g., 'log_loss' or 'modified_huber') to evaluate probability cutoffs."
    )

print(f"Using SGD_18F Ray Tune trial {SGD_18F_selected_trial_id} configuration:")
print(SGD_18F_selected_config)

SGD_18F_threshold_model = make_pipeline(
    StandardScaler(),
    OneVsRestClassifier(
        SGDClassifier(
            loss=SGD_18F_selected_config["loss"],
            alpha=float(SGD_18F_selected_config["alpha"]),
            penalty=SGD_18F_selected_config["penalty"],
            learning_rate=SGD_18F_selected_config["learning_rate"],
            eta0=0.01,
            class_weight="balanced",
            max_iter=2000,
            tol=1e-3,
            random_state=42,
        )
    ),
)

SGD_18F_X_train_threshold = (
    SGD_18F_X_train.to_numpy(dtype=np.float32)
    if hasattr(SGD_18F_X_train, "to_numpy")
    else np.asarray(SGD_18F_X_train, dtype=np.float32)
)
SGD_18F_y_train_threshold = (
    SGD_18F_y_train.to_numpy() if hasattr(SGD_18F_y_train, "to_numpy") else np.asarray(SGD_18F_y_train)
)

# NOTE: Generate out-of-fold predictions to estimate generalization without test leakage.
SGD_18F_y_train_proba = cross_val_predict(
    SGD_18F_threshold_model,
    SGD_18F_X_train_threshold,
    SGD_18F_y_train_threshold,
    cv=SGD_18F_threshold_cv_folds,
    method="predict_proba",
    n_jobs=1,
)

SGD_18F_y_train_proba = np.asarray(SGD_18F_y_train_proba)
if SGD_18F_y_train_proba.ndim == 1:
    SGD_18F_y_train_proba = SGD_18F_y_train_proba[:, None]

SGD_18F_y_true_threshold = np.asarray(SGD_18F_y_train_threshold)
if SGD_18F_y_true_threshold.ndim == 1:
    SGD_18F_y_true_threshold = SGD_18F_y_true_threshold[:, None]

SGD_18F_threshold_metric_rows = []
for cutoff in SGD_18F_probability_cutoffs:
    SGD_18F_y_pred_cutoff = (SGD_18F_y_train_proba >= cutoff).astype(int)
    SGD_18F_threshold_metric_rows.append(
        {
            "cutoff": float(cutoff),
            "f1_micro": f1_score(SGD_18F_y_true_threshold, SGD_18F_y_pred_cutoff, average="micro", zero_division=0),
            "f1_macro": f1_score(SGD_18F_y_true_threshold, SGD_18F_y_pred_cutoff, average="macro", zero_division=0),
            "precision_micro": precision_score(SGD_18F_y_true_threshold, SGD_18F_y_pred_cutoff, average="micro", zero_division=0),
            "recall_micro": recall_score(SGD_18F_y_true_threshold, SGD_18F_y_pred_cutoff, average="micro", zero_division=0),
            "precision_macro": precision_score(SGD_18F_y_true_threshold, SGD_18F_y_pred_cutoff, average="macro", zero_division=0),
            "recall_macro": recall_score(SGD_18F_y_true_threshold, SGD_18F_y_pred_cutoff, average="macro", zero_division=0),
        }
    )



Using SGD_18F Ray Tune trial 96991386 configuration:
{'alpha': 1.9616460353882896e-05, 'loss': 'log_loss', 'penalty': 'l1', 'learning_rate': 'adaptive'}


/opt/anaconda3/lib/python3.12/site-packages/sklearn/multiclass.py:90: UserWarning: Label not 24 is present in all training examples.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/multiclass.py:90: UserWarning: Label not 82 is present in all training examples.
  warnings.warn(


In [ ]:
# CELL GUIDE:
# Purpose: Run `SGD_18F_threshold_cv_df = pd.DataFrame(SGD_18F_threshold_metric_rows).sort_values("cutoff"` and continue downstream processing.
# Workflow:
# 1. SGD_18F_threshold_cv_df = pd.DataFrame(SGD_18F_threshold_metric_rows).sort_values("cutoff").reset_index(drop=True)
# 2. SGD_18F_best_cutoff_f1_micro = SGD_18F_threshold_cv_df.loc[SGD_18F_threshold_cv_df["f1_micro"].idxmax()]
# 3. SGD_18F_best_cutoff_f1_macro = SGD_18F_threshold_cv_df.loc[SGD_18F_threshold_cv_df["f1_macro"].idxmax()]
# 4. SGD_18F_optimal_cutoff = 0.45
# 5. ecg_data.setdefault("ray_tune_sgd_18F", {})["threshold_cv"] = {
# 6. }

SGD_18F_threshold_cv_df = pd.DataFrame(SGD_18F_threshold_metric_rows).sort_values("cutoff").reset_index(drop=True)
SGD_18F_best_cutoff_f1_micro = SGD_18F_threshold_cv_df.loc[SGD_18F_threshold_cv_df["f1_micro"].idxmax()]
SGD_18F_best_cutoff_f1_macro = SGD_18F_threshold_cv_df.loc[SGD_18F_threshold_cv_df["f1_macro"].idxmax()]

SGD_18F_optimal_cutoff = 0.45
# NOTE: Persist artifacts in ecg_data so later cells can reuse them without recomputation.
ecg_data.setdefault("ray_tune_sgd_18F", {})["threshold_cv"] = {
    "trial_id": SGD_18F_selected_trial_id,
    "config": dict(SGD_18F_selected_config),
    "cv_folds": SGD_18F_threshold_cv_folds,
    "optimal_cutoff": SGD_18F_optimal_cutoff,
    "metrics_by_cutoff": SGD_18F_threshold_cv_df.copy(),
}

SGD_18F_threshold_cv_df[[
    "cutoff", "f1_micro", "f1_macro", "precision_micro", "recall_micro", "precision_macro", "recall_macro"
]].round(4)

,cutoff,f1_micro,f1_macro,precision_micro,recall_micro,precision_macro,recall_macro
0,0.05,0.0671,0.0442,0.0347,0.9922,0.0256,0.7640
1,0.10,0.0716,0.0477,0.0372,0.9877,0.0280,0.7548
2,0.15,0.0761,0.0508,0.0396,0.9804,0.0301,0.7432
3,0.20,0.0810,0.0541,0.0422,0.9700,0.0324,0.7322
4,0.25,0.0863,0.0574,0.0452,0.9544,0.0348,0.7180
5,0.30,0.0924,0.0610,0.0486,0.9328,0.0374,0.6982
6,0.35,0.0992,0.0648,0.0525,0.9022,0.0403,0.6746
7,0.40,0.1068,0.0689,0.0570,0.8607,0.0436,0.6455
8,0.45,0.1153,0.0733,0.0621,0.8085,0.0472,0.6098
9,0.50,0.1249,0.0780,0.0682,0.7450,0.0515,0.5681


In [42]:
# CELL GUIDE:
# Purpose: SGD_18F cutoff visualizations merged into subplots (shared x-axis).
# Workflow:
# 1. SGD_18F cutoff visualizations merged into subplots (shared x-axis).
# 2. import plotly.graph_objects as go
# 3. from plotly.subplots import make_subplots
# 4. SGD_18F_plot_df = SGD_18F_threshold_cv_df.copy()
# 5. SGD_18F_plot_style = graph_formatting if "graph_formatting" in globals() else {
# 6. }

# SGD_18F cutoff visualizations merged into subplots (shared x-axis).
import plotly.graph_objects as go
from plotly.subplots import make_subplots

SGD_18F_plot_df = SGD_18F_threshold_cv_df.copy()
SGD_18F_plot_style = graph_formatting if "graph_formatting" in globals() else {
    "font_family": "Helvetica Neue", "font_size_base": 16, "font_size_axis": 12,
    "color_text": "#b8b8b8", "color_background": "#000000", "margin_top": 170,
    "margin_right": 80, "margin_bottom": 80, "margin_left": 80,
}
SGD_18F_text = SGD_18F_plot_style.get("color_text", "#b8b8b8")
SGD_18F_bg = SGD_18F_plot_style.get("color_background", "#000000")
SGD_18F_grid = "#1f1f1f"

SGD_18F_fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.14,
    subplot_titles=(
        "F1 Micro And F1 Macro",
        "Recall And Precision (Micro And Macro)",
    ),
)

# Left-align subplot subtitles and add breathing room.
for ann in SGD_18F_fig.layout.annotations:
    ann.update(x=0.0, xanchor="left", align="left", xref="paper", y=ann.y + 0.01)

# Top subplot: F1 metrics only.
for metric_name, color in [
    ("f1_micro", SERIES_COLORS["f1_micro"]),
    ("f1_macro", SERIES_COLORS["f1_macro"]),
]:
    SGD_18F_fig.add_trace(
        go.Scatter(
            x=SGD_18F_plot_df["cutoff"],
            y=SGD_18F_plot_df[metric_name],
            mode="lines+markers",
            name=metric_name.replace("_", " ").title(),
            line=dict(width=2.2, color=color),
            marker=dict(size=6, color=color),
            hovertemplate=(
                "Cutoff: %{x:.2f}<br>"
                + f"{metric_name.replace('_', ' ').title()}: %{{y:.4f}}<extra></extra>"
            ),
        ),
        row=1,
        col=1,
    )

# Bottom subplot: recall/precision micro+macro.
for metric_name, color in [
    ("recall_micro", SERIES_COLORS["recall_micro"]),
    ("recall_macro", SERIES_COLORS["recall_macro"]),
    ("precision_micro", SERIES_COLORS["precision_micro"]),
    ("precision_macro", SERIES_COLORS["precision_macro"]),
]:
    SGD_18F_fig.add_trace(
        go.Scatter(
            x=SGD_18F_plot_df["cutoff"],
            y=SGD_18F_plot_df[metric_name],
            mode="lines+markers",
            name=metric_name.replace("_", " ").title(),
            line=dict(width=2.2, color=color),
            marker=dict(size=6, color=color),
            hovertemplate=(
                "Cutoff: %{x:.2f}<br>"
                + f"{metric_name.replace('_', ' ').title()}: %{{y:.4f}}<extra></extra>"
            ),
        ),
        row=2,
        col=1,
    )

# Vertical reference lines at optimal cutoff in both subplots.
SGD_18F_optimal_cutoff = float(SGD_18F_optimal_cutoff)
for row_idx in [1, 2]:
    SGD_18F_fig.add_vline(
        x=SGD_18F_optimal_cutoff,
        line_dash="dash",
        line_color="#8a8a8a",
        line_width=1.6,
        opacity=0.9,
        row=row_idx,
        col=1,
    )

# Dynamic y-range for F1 subplot for better visibility.
SGD_18F_f1_min = float(SGD_18F_plot_df[["f1_micro", "f1_macro"]].min().min())
SGD_18F_f1_max = float(SGD_18F_plot_df[["f1_micro", "f1_macro"]].max().max())
SGD_18F_f1_pad = max(0.01, 0.15 * (SGD_18F_f1_max - SGD_18F_f1_min + 1e-12))

# X-axis label only on bottom subplot (no repetition).
# NOTE: Configure x-axis semantics and formatting for accurate interpretation.
SGD_18F_fig.update_xaxes(
    title_text="",
    tickformat=".2f",
    showgrid=True,
    gridcolor=SGD_18F_grid,
    gridwidth=0.2,
    zeroline=False,
    row=1,
    col=1,
)
# NOTE: Configure x-axis semantics and formatting for accurate interpretation.
SGD_18F_fig.update_xaxes(
    tickformat=".2f",
    showgrid=True,
    gridcolor=SGD_18F_grid,
    gridwidth=0.2,
    zeroline=False,
    row=2,
    col=1,
)

# Remove y-axis title label text in both subplots.
# NOTE: Configure y-axis semantics and formatting for accurate interpretation.
SGD_18F_fig.update_yaxes(
    title_text="",
    range=[max(0.0, SGD_18F_f1_min - SGD_18F_f1_pad), min(1.02, SGD_18F_f1_max + SGD_18F_f1_pad)],
    tickformat=".3f",
    showgrid=True,
    gridcolor=SGD_18F_grid,
    gridwidth=0.2,
    zeroline=False,
    row=1,
    col=1,
)
# NOTE: Configure y-axis semantics and formatting for accurate interpretation.
SGD_18F_fig.update_yaxes(
    title_text="",
    range=[0, 1.02],
    tickformat=".2f",
    showgrid=True,
    gridcolor=SGD_18F_grid,
    gridwidth=0.2,
    zeroline=False,
    row=2,
    col=1,
)

# NOTE: Apply global styling (title, typography, spacing, colors) for readability consistency.
SGD_18F_fig.update_layout(
    title=dict(
        text=(
            "<span style='font-size:30px;font-weight:bold;'>    Identifying 0.45 as the Most Viable Cutoff Despite Modest Overall Performance</span>"
            "<br><span style='font-size:20px;font-weight: normal;'>      SCG_18F: Performance Metrics Across Different Cutoffs</span>"
        ),
        x=0,
        xanchor="left",
        pad=dict(t=50),
    ),
    paper_bgcolor=SGD_18F_bg,
    plot_bgcolor=SGD_18F_bg,
    font=dict(family=SGD_18F_plot_style["font_family"], size=SGD_18F_plot_style["font_size_base"], color=SGD_18F_text),
    legend=STANDARD_LEGEND,
    margin=dict(
        t=SGD_18F_plot_style["margin_top"] + 90,
        r=SGD_18F_plot_style["margin_right"],
        b=SGD_18F_plot_style["margin_bottom"],
        l=SGD_18F_plot_style["margin_left"],
    ),
    height=980,
    hovermode="x unified",
)

SGD_18F_fig.show()


# **Conducting PCAs On 18 Metrics (Features)**


<!-- CELL NOTE -->
> **Cell Note:** This section documents **Conducting PCAs On 18 Metrics (Features)** and provides context for the following code/results.
> Review it before executing downstream cells so assumptions and outputs remain aligned.


In [43]:
# CELL GUIDE:
# Purpose: Run PCA on ecg_data["metrics"] and plot cumulative explained variance.
# Workflow:
# 1. Run PCA on ecg_data["metrics"] and plot cumulative explained variance.
# 2. from sklearn.decomposition import PCA
# 3. from sklearn.impute import SimpleImputer
# 4. from sklearn.preprocessing import StandardScaler
# 5. metrics_for_pca = ecg_data["metrics"].copy()
# 6. imputer = SimpleImputer(strategy="median")

# Run PCA on ecg_data["metrics"] and plot cumulative explained variance.
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

metrics_for_pca = ecg_data["metrics"].copy()

# NOTE: Fill missing values to ensure downstream estimators receive dense numeric input.
imputer = SimpleImputer(strategy="median")
# NOTE: Standardize feature scales so optimization is stable and comparable.
scaler = StandardScaler()

X_imputed = imputer.fit_transform(metrics_for_pca)
X_scaled = scaler.fit_transform(X_imputed)

# NOTE: Reduce dimensionality while retaining the dominant variance structure.
pca = PCA()
pca.fit(X_scaled)

explained_variance_ratio = pca.explained_variance_ratio_
cumulative_explained_variance = np.cumsum(explained_variance_ratio)
component_numbers = np.arange(1, cumulative_explained_variance.size + 1)

# Use existing formatting standards when available.
if "graph_formatting" in globals():
    pca_style = graph_formatting
else:
    pca_style = {
        "font_family": "Helvetica Neue",
        "font_size_base": 16,
        "font_size_axis": 12,
        "color_text": "#b8b8b8",
        "color_title": "white",
        "color_grid": "#4a0000",
        "color_background": "#000000",
        "color_line": "#636EFA",
        "margin_top": 170,
        "margin_right": 80,
        "margin_bottom": 80,
        "margin_left": 80,
    }

text_gray = pca_style["color_text"]
grid_red = pca_style["color_grid"]
pca_grid_dark = "#1f1f1f"
bg_black = pca_style["color_background"]

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=component_numbers,
        y=cumulative_explained_variance,
        mode="lines+markers",
        line=dict(width=2.0, color=pca_style["color_line"]),
        marker=dict(size=6, color=pca_style["color_line"]),
        hovertemplate="Principal Components: %{x}<br>Cumulative Explained Variance: %{y:.2%}<extra></extra>",
        showlegend=False,
    )
)

fig.add_hline(y=0.90, line_dash="dot", line_color="#8a8a8a", line_width=1)
fig.add_hline(y=0.95, line_dash="dot", line_color="#f21322", line_width=1)

# NOTE: Configure x-axis semantics and formatting for accurate interpretation.
fig.update_xaxes(
    title_text="Number Of Principal Components",
    dtick=1,
    showgrid=True,
    gridcolor=pca_grid_dark,
    gridwidth=0.2,
    zeroline=False,
    title_font=dict(family=pca_style["font_family"], size=pca_style["font_size_axis"], color=text_gray),
    tickfont=dict(family=pca_style["font_family"], size=pca_style["font_size_axis"], color=text_gray),
)
# NOTE: Configure y-axis semantics and formatting for accurate interpretation.
fig.update_yaxes(
    tickformat=".0%",
    range=[0, 1.02],
    showgrid=True,
    gridcolor=pca_grid_dark,
    gridwidth=0.2,
    zeroline=False,
    title_font=dict(family=pca_style["font_family"], size=pca_style["font_size_axis"], color=text_gray),
    tickfont=dict(family=pca_style["font_family"], size=pca_style["font_size_axis"], color=text_gray),
)
# NOTE: Apply global styling (title, typography, spacing, colors) for readability consistency.
fig.update_layout(
    title=dict(
        text=(
            "<span style='font-size:30px;font-weight:bold;'>    Capturing 95% of Variance Requires 13 Principal Components</span>"
            "<br><span style='font-size:20px;font-weight: normal;'>      Cumulative Explained Variance Across ECG Feature Components</span>"
        ),
        x=0,
        xanchor="left",
        pad=dict(t=50),
    ),
    paper_bgcolor=bg_black,
    plot_bgcolor=bg_black,
    font=dict(family=pca_style["font_family"], size=pca_style["font_size_base"], color=text_gray),
    hoverlabel=dict(font=dict(family=pca_style["font_family"], size=pca_style["font_size_axis"], color=text_gray)),
    margin=dict(
        t=pca_style["margin_top"],
        r=pca_style["margin_right"],
        b=pca_style["margin_bottom"],
        l=pca_style["margin_left"],
    ),
    height=720,
)

fig.show()

n_components_90 = int(np.searchsorted(cumulative_explained_variance, 0.90) + 1)
n_components_95 = int(np.searchsorted(cumulative_explained_variance, 0.95) + 1)
print(f"Components to reach 90% explained variance: {n_components_90}")
print(f"Components to reach 95% explained variance: {n_components_95}")


Components to reach 90% explained variance: 11
Components to reach 95% explained variance: 13


# **SGD_13C**


<!-- CELL NOTE -->
> **Cell Note:** This section documents **SGD_13C** and provides context for the following code/results.
> Review it before executing downstream cells so assumptions and outputs remain aligned.


In [15]:
# CELL GUIDE:
# Purpose: Train a One-vs-Rest SGD_13C classifier on 13 principal components (dimension reduction, not feature selection).
# Workflow:
# 1. Train a One-vs-Rest SGD_13C classifier on 13 principal components (dimension reduction, not feature selection).
# 2. from sklearn.decomposition import PCA
# 3. from sklearn.linear_model import SGDClassifier
# 4. from sklearn.metrics import f1_score, precision_score, recall_score
# 5. from sklearn.model_selection import cross_val_predict, cross_val_score, train_test_split
# 6. from sklearn.multiclass import OneVsRestClassifier

# Train a One-vs-Rest SGD_13C classifier on 13 principal components (dimension reduction, not feature selection).
from sklearn.decomposition import PCA
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.model_selection import cross_val_predict, cross_val_score, train_test_split
from sklearn.multiclass import OneVsRestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import MultiLabelBinarizer, StandardScaler

required_keys = ["metrics_model_18F", "metadata_model_18F", "dx_code_label_sets_model_18F"]
missing_keys = [k for k in required_keys if k not in ecg_data]
if missing_keys:
    raise RuntimeError(
        f"Run the shared 18-feature cleaning/label-prep cell first. Missing: {missing_keys}"
    )

SGD_13C_n_components = 13 #use 13 components, side note: codex used only 13 FEATURES first and not the components results from the PCA. This has been corrected here
SGD_13C_metrics_source_df = ecg_data["metrics_model_18F"].copy()
SGD_13C_metadata_model_df = ecg_data["metadata_model_18F"].copy()
SGD_13C_y_labels = ecg_data["dx_code_label_sets_model_18F"].copy()

# PCA is performed on standardized 18-feature inputs to obtain 13 principal-component dimensions.
# NOTE: Standardize feature scales so optimization is stable and comparable.
SGD_13C_pca_input_scaler = StandardScaler()
SGD_13C_X_scaled_for_pca = SGD_13C_pca_input_scaler.fit_transform(
    SGD_13C_metrics_source_df.to_numpy(dtype=np.float32)
)
# NOTE: Reduce dimensionality while retaining the dominant variance structure.
SGD_13C_pca = PCA(n_components=SGD_13C_n_components)
SGD_13C_X_pca = SGD_13C_pca.fit_transform(SGD_13C_X_scaled_for_pca).astype(np.float32)
SGD_13C_component_columns = [f"PC{i}" for i in range(1, SGD_13C_n_components + 1)]
SGD_13C_metrics_model_df = pd.DataFrame(SGD_13C_X_pca, columns=SGD_13C_component_columns)

ecg_data["metrics_model_SGD_13C"] = SGD_13C_metrics_model_df.copy()
ecg_data["metadata_model_SGD_13C"] = SGD_13C_metadata_model_df.copy()
ecg_data["dx_code_label_sets_model_SGD_13C"] = SGD_13C_y_labels.copy()
# NOTE: Persist artifacts in ecg_data so later cells can reuse them without recomputation.
ecg_data.setdefault("pca_models", {})["SGD_13C"] = {
    "n_components": SGD_13C_n_components,
    "explained_variance_ratio": SGD_13C_pca.explained_variance_ratio_.tolist(),
}

print(f"SGD_13C modeling metrics shape (principal components): {SGD_13C_metrics_model_df.shape}")
print(f"SGD_13C modeling metadata shape: {SGD_13C_metadata_model_df.shape}")
print(f"SGD_13C cumulative explained variance (13 PCs): {float(np.sum(SGD_13C_pca.explained_variance_ratio_)):.4f}")

SGD_13C_X = ecg_data["metrics_model_SGD_13C"].copy()
# NOTE: Convert per-sample label sets into a multi-hot indicator matrix.
SGD_13C_mlb = MultiLabelBinarizer()
SGD_13C_y = SGD_13C_mlb.fit_transform(SGD_13C_y_labels)

SGD_13C_X_train, SGD_13C_X_test, SGD_13C_y_train, SGD_13C_y_test = train_test_split(
    SGD_13C_X,
    SGD_13C_y,
    test_size=0.20,
    random_state=42,
    shuffle=True,
)


SGD_13C modeling metrics shape (principal components): (45152, 13)
SGD_13C modeling metadata shape: (45152, 9)
SGD_13C cumulative explained variance (13 PCs): 0.9851


In [ ]:
# CELL GUIDE:
# Purpose: Hyperparameter tune One-vs-Rest SGD_13C on the training split only using Ray Tune + cross-validation.
# Workflow:
# 1. Hyperparameter tune One-vs-Rest SGD_13C on the training split only using Ray Tune + cross-validation.
# 2. import ray
# 3. import random
# 4. from ray import tune
# 5. from ray.tune.schedulers import ASHAScheduler
# 6. from sklearn.linear_model import SGDClassifier

# Hyperparameter tune One-vs-Rest SGD_13C on the training split only using Ray Tune + cross-validation.
import ray
import random
from ray import tune
from ray.tune.schedulers import ASHAScheduler
from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import cross_validate
from sklearn.multiclass import OneVsRestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Use the existing training split from the previous SGD_13C cell (test set remains untouched).
SGD_13C_X_train_tune = SGD_13C_X_train.to_numpy(dtype=np.float32) if hasattr(SGD_13C_X_train, "to_numpy") else np.asarray(SGD_13C_X_train, dtype=np.float32)
SGD_13C_y_train_tune = np.asarray(SGD_13C_y_train)

SGD_13C_ray_cv_folds = 3
SGD_13C_num_ray_trials = 30

SGD_13C_param_dist = {
    "alpha": tune.loguniform(1e-6, 1e-1),
    "loss": tune.choice(["hinge", "log_loss", "modified_huber"]),
    "penalty": tune.choice(["l1", "l2", "elasticnet"]),
    "learning_rate": tune.choice(["optimal", "adaptive", "constant", "invscaling"]),
}

SGD_13C_scoring = {
    "precision_micro": "precision_micro",
    "recall_micro": "recall_micro",
    "f1_micro": "f1_micro",
    "f1_macro": "f1_macro",
}


def SGD_13C_ray_sgd_cv_objective(config, X_train_arr, y_train_arr, cv_splits):
    SGD_13C_model = make_pipeline(
        StandardScaler(),
        OneVsRestClassifier(
            SGDClassifier(
                loss=config["loss"],
                alpha=float(config["alpha"]),
                penalty=config["penalty"],
                learning_rate=config["learning_rate"],
                eta0=0.01,
                class_weight="balanced",
                max_iter=2000,
                tol=1e-3,
                random_state=42,
            )
        ),
    )

# NOTE: Run cross-validation to aggregate robust performance estimates across folds.
    SGD_13C_cv_scores = cross_validate(
        SGD_13C_model,
        X_train_arr,
        y_train_arr,
        cv=cv_splits,
        scoring=SGD_13C_scoring,
        n_jobs=1,
        error_score="raise",
    )

    tune.report({
    "precision_micro": float(np.mean(SGD_13C_cv_scores["test_precision_micro"])),
    "recall_micro": float(np.mean(SGD_13C_cv_scores["test_recall_micro"])),
    "f1_micro": float(np.mean(SGD_13C_cv_scores["test_f1_micro"])),
    "f1_macro": float(np.mean(SGD_13C_cv_scores["test_f1_macro"])),
})


np.random.seed(42)
random.seed(42)

try:
    from ray.tune.search.optuna import OptunaSearch
    SGD_13C_search_alg = OptunaSearch(seed=42)
    SGD_13C_search_backend = "optuna"
except Exception:
    from ray.tune.search.hyperopt import HyperOptSearch
    SGD_13C_search_alg = HyperOptSearch(random_state_seed=42)
    SGD_13C_search_backend = "hyperopt"

SGD_13C_scheduler = ASHAScheduler(
    max_t=1,
    grace_period=1,
    reduction_factor=2,
)
print(f"SGD_13C search backend: {SGD_13C_search_backend}")

if ray.is_initialized():
    ray.shutdown()
ray.init(ignore_reinit_error=True, include_dashboard=False, log_to_driver=False)

# NOTE: Launch Ray Tune hyperparameter search over the configured trial space.
SGD_13C_ray_analysis = tune.run(
    tune.with_parameters(
        SGD_13C_ray_sgd_cv_objective,
        X_train_arr=SGD_13C_X_train_tune,
        y_train_arr=SGD_13C_y_train_tune,
        cv_splits=SGD_13C_ray_cv_folds,
    ),
    config=SGD_13C_param_dist,
    num_samples=SGD_13C_num_ray_trials,
    metric="f1_macro",                  # Prioritize configurations with higher macro F1 (balanced performance across classes) as the main selection criterion.
    mode="max",
    resources_per_trial={"cpu": 1},
    search_alg=SGD_13C_search_alg,
    scheduler=SGD_13C_scheduler,
    verbose=1,
)

SGD_13C_ray_tune_results_df = SGD_13C_ray_analysis.dataframe().copy().rename(
    columns={
        "config/alpha": "alpha",
        "config/loss": "loss",
        "config/penalty": "penalty",
        "config/learning_rate": "learning_rate",
    }
)

for SGD_13C_metric_name in ["precision_micro", "recall_micro", "f1_micro", "f1_macro"]:
    if SGD_13C_metric_name not in SGD_13C_ray_tune_results_df.columns:
        for alt_col in (f"last_result/{SGD_13C_metric_name}", f"last_result.{SGD_13C_metric_name}"):
            if alt_col in SGD_13C_ray_tune_results_df.columns:
                SGD_13C_ray_tune_results_df[SGD_13C_metric_name] = SGD_13C_ray_tune_results_df[alt_col]
                break

SGD_13C_keep_columns = [
    col
    for col in [
        "trial_id",
        "alpha",
        "loss",
        "penalty",
        "learning_rate",
        "precision_micro",
        "recall_micro",
        "f1_micro",
        "f1_macro",
        "training_iteration",
        "time_total_s",
    ]
    if col in SGD_13C_ray_tune_results_df.columns
]
SGD_13C_ray_tune_results_df = SGD_13C_ray_tune_results_df[SGD_13C_keep_columns].copy()

SGD_13C_sort_cols = [col for col in ["f1_micro", "f1_macro"] if col in SGD_13C_ray_tune_results_df.columns]
if SGD_13C_sort_cols:
    SGD_13C_ray_tune_results_df = SGD_13C_ray_tune_results_df.sort_values(
        SGD_13C_sort_cols,
        ascending=[False] * len(SGD_13C_sort_cols),
        na_position="last",
    ).reset_index(drop=True)
else:
    SGD_13C_ray_tune_results_df = SGD_13C_ray_tune_results_df.reset_index(drop=True)

# Store results for reuse in later cells (e.g., threshold tuning by selected trial ID).
# NOTE: Persist artifacts in ecg_data so later cells can reuse them without recomputation.
ecg_data.setdefault("ray_tune_sgd_13C", {})["results"] = SGD_13C_ray_tune_results_df.copy()
# NOTE: Persist artifacts in ecg_data so later cells can reuse them without recomputation.
ecg_data.setdefault("ray_tune_sgd_13C", {})["cv_folds"] = SGD_13C_ray_cv_folds
# NOTE: Persist artifacts in ecg_data so later cells can reuse them without recomputation.
ecg_data.setdefault("ray_tune_sgd_13C", {})["num_trials"] = SGD_13C_num_ray_trials

SGD_13C_ray_tune_results_df

# Persist a default selected trial/config for downstream cells.
if not SGD_13C_ray_tune_results_df.empty:
    _row = SGD_13C_ray_tune_results_df.iloc[0]
# NOTE: Persist artifacts in ecg_data so later cells can reuse them without recomputation.
    ecg_data.setdefault("ray_tune_sgd_13C", {})["selected_trial_id"] = str(_row.get("trial_id", ""))
# NOTE: Persist artifacts in ecg_data so later cells can reuse them without recomputation.
    ecg_data.setdefault("ray_tune_sgd_13C", {})["selected_config"] = {
        "alpha": float(_row.get("alpha")) if pd.notna(_row.get("alpha")) else None,
        "loss": str(_row.get("loss")) if pd.notna(_row.get("loss")) else None,
        "penalty": str(_row.get("penalty")) if pd.notna(_row.get("penalty")) else None,
        "learning_rate": str(_row.get("learning_rate")) if pd.notna(_row.get("learning_rate")) else None,
    }


2026-03-02 15:21:25,244	INFO tune.py:1009 -- Wrote the latest version of all result files and experiment state to '/Users/felixaugustin/ray_results/SGD_13C_ray_sgd_cv_objective_2026-03-02_15-04-52' in 0.0212s.
2026-03-02 15:21:25,254	INFO tune.py:1041 -- Total run time: 992.91 seconds (992.74 seconds for the tuning loop).


In [45]:
# CELL GUIDE:
# Purpose: SGD_13C target metric over time (chronological).
# Workflow:
# 1. SGD_13C target metric over time (chronological)
# 2. import numpy as np
# 3. import pandas as pd
# 4. import plotly.graph_objects as go
# 5. SGD_13C_results_source = None
# 6. if "SGD_13C_ray_tune_results_df" in globals()

# SGD_13C target metric over time (chronological)
import numpy as np
import pandas as pd
import plotly.graph_objects as go

SGD_13C_results_source = None
if "SGD_13C_ray_tune_results_df" in globals():
    SGD_13C_results_source = SGD_13C_ray_tune_results_df
elif isinstance(ecg_data, dict):
    SGD_13C_results_source = (ecg_data.get("ray_tune_sgd_13C") or {}).get("results")

if SGD_13C_results_source is None:
    raise RuntimeError("SGD_13C Ray results were not found. Run the SGD_13C tuning cell first.")

SGD_13C_time_df = SGD_13C_results_source.copy()
if SGD_13C_time_df.empty:
    raise RuntimeError("SGD_13C Ray results are empty.")

SGD_13C_target_metric = "f1_macro"
if SGD_13C_target_metric not in SGD_13C_time_df.columns:
    if "f1_micro" in SGD_13C_time_df.columns:
        SGD_13C_target_metric = "f1_micro"
    else:
        raise KeyError("Neither f1_macro nor f1_micro is available in SGD_13C Ray results.")

SGD_13C_time_sort_col = None
if "timestamp" in SGD_13C_time_df.columns:
    SGD_13C_time_df["_chrono"] = pd.to_numeric(SGD_13C_time_df["timestamp"], errors="coerce")
    SGD_13C_time_sort_col = "_chrono"
elif "date" in SGD_13C_time_df.columns:
    SGD_13C_time_df["_chrono"] = pd.to_datetime(SGD_13C_time_df["date"], errors="coerce")
    SGD_13C_time_sort_col = "_chrono"
elif "time_total_s" in SGD_13C_time_df.columns:
    SGD_13C_time_df["_chrono"] = pd.to_numeric(SGD_13C_time_df["time_total_s"], errors="coerce")
    SGD_13C_time_sort_col = "_chrono"
elif "training_iteration" in SGD_13C_time_df.columns:
    SGD_13C_time_df["_chrono"] = pd.to_numeric(SGD_13C_time_df["training_iteration"], errors="coerce")
    SGD_13C_time_sort_col = "_chrono"

if SGD_13C_time_sort_col is not None:
    SGD_13C_time_df = SGD_13C_time_df.sort_values(SGD_13C_time_sort_col, ascending=True, na_position="last")

SGD_13C_time_df = SGD_13C_time_df.reset_index(drop=True)
SGD_13C_time_df["trial_order"] = np.arange(1, len(SGD_13C_time_df) + 1)
SGD_13C_time_df["trial_id"] = SGD_13C_time_df.get("trial_id", "").astype(str)

SGD_13C_plot_style = graph_formatting if "graph_formatting" in globals() else {
    "font_family": "Helvetica Neue",
    "font_size_base": 16,
    "font_size_axis": 12,
    "color_text": "#b8b8b8",
    "color_background": "#000000",
    "margin_top": 170,
    "margin_right": 80,
    "margin_bottom": 80,
    "margin_left": 80,
}
SGD_13C_text = SGD_13C_plot_style.get("color_text", "#b8b8b8")
SGD_13C_bg = SGD_13C_plot_style.get("color_background", "#000000")
SGD_13C_color = SERIES_COLORS.get("f1_macro", "#636EFA") if "SERIES_COLORS" in globals() else "#636EFA"

SGD_13C_time_fig = go.Figure(
    data=[
        go.Scatter(
            x=SGD_13C_time_df["trial_order"],
            y=SGD_13C_time_df[SGD_13C_target_metric],
            mode="lines+markers",
            line=dict(width=2.2, color=SGD_13C_color),
            marker=dict(size=6, color=SGD_13C_color),
            customdata=np.column_stack([SGD_13C_time_df["trial_id"]]),
            hovertemplate=(
                "Chronological Order: %{x}<br>"
                "Trial ID: %{customdata[0]}<br>"
                f"{SGD_13C_target_metric.upper()}: %{{y:.4f}}<extra></extra>"
            ),
            name="SGD_13C",
        )
    ]
)

# NOTE: Configure x-axis semantics and formatting for accurate interpretation.
SGD_13C_time_fig.update_xaxes(
    title_text="Chronological Trial Order",
    title_font=dict(family=SGD_13C_plot_style["font_family"], size=SGD_13C_plot_style["font_size_axis"], color=SGD_13C_text),
    tickfont=dict(family=SGD_13C_plot_style["font_family"], size=SGD_13C_plot_style["font_size_axis"], color=SGD_13C_text),
    showgrid=True,
    gridcolor="#1f1f1f",
    gridwidth=0.2,
)

# NOTE: Configure y-axis semantics and formatting for accurate interpretation.
SGD_13C_time_fig.update_yaxes(
    title_font=dict(family=SGD_13C_plot_style["font_family"], size=SGD_13C_plot_style["font_size_axis"], color=SGD_13C_text),
    tickfont=dict(family=SGD_13C_plot_style["font_family"], size=SGD_13C_plot_style["font_size_axis"], color=SGD_13C_text),
    showgrid=True,
    gridcolor="#1f1f1f",
    gridwidth=0.2,
)

# NOTE: Apply global styling (title, typography, spacing, colors) for readability consistency.
SGD_13C_time_fig.update_layout(
    title=dict(
        text=(
            "<span style='font-size:30px;font-weight:bold;'>    Early Volatility Followed by Performance Plateau Under ASHA</span>"
            "<br><span style='font-size:20px;font-weight: normal;'>      SGD_13C Target Metric - F1 Macro During Each Ray Tune Trial</span>"
        ),
        x=0,
        xanchor="left",
        pad=dict(t=50),
    ),
    paper_bgcolor=SGD_13C_bg,
    plot_bgcolor=SGD_13C_bg,
    font=dict(family=SGD_13C_plot_style["font_family"], size=SGD_13C_plot_style["font_size_base"], color=SGD_13C_text),
    hoverlabel=dict(font=dict(family=SGD_13C_plot_style["font_family"], size=SGD_13C_plot_style["font_size_axis"], color=SGD_13C_text)),
    legend=STANDARD_LEGEND if "STANDARD_LEGEND" in globals() else dict(orientation="h", yanchor="top", y=1.02, xanchor="left", x=0),
    margin=dict(
        t=SGD_13C_plot_style["margin_top"],
        r=SGD_13C_plot_style["margin_right"],
        b=SGD_13C_plot_style["margin_bottom"],
        l=SGD_13C_plot_style["margin_left"],
    ),
    height=700,
)

SGD_13C_time_fig.show()


In [18]:
# CELL GUIDE:
# Purpose: Run `SGD_13C_ray_tune_results_df` and continue downstream processing.
# Workflow:
# 1. SGD_13C_ray_tune_results_df

SGD_13C_ray_tune_results_df

,trial_id,alpha,loss,penalty,learning_rate,precision_micro,recall_micro,f1_micro,f1_macro,training_iteration,time_total_s
0,dc9c7a48,0.000003,log_loss,l2,adaptive,0.062816,0.723150,0.115591,0.072030,1,189.957983
1,b1563a9a,0.000028,log_loss,l2,adaptive,0.062681,0.723036,0.115360,0.072010,1,169.855972
2,aa03825b,0.000023,hinge,l1,adaptive,0.061904,0.728566,0.114112,0.073197,1,113.977779
3,00236ce1,0.000025,hinge,l1,adaptive,0.061842,0.728481,0.114005,0.072994,1,134.576442
4,3de5cc06,0.000188,modified_huber,elasticnet,adaptive,0.061842,0.721501,0.113920,0.071822,1,246.252167
5,75fed72b,0.000024,hinge,elasticnet,adaptive,0.061726,0.728338,0.113808,0.073041,1,141.724924
6,cd50f13d,0.000020,hinge,elasticnet,adaptive,0.061677,0.728254,0.113722,0.073059,1,170.808146
7,0fb02b9d,0.000156,modified_huber,elasticnet,adaptive,0.061706,0.721657,0.113690,0.071862,1,235.589262
8,c0f121bb,0.000033,hinge,elasticnet,adaptive,0.061618,0.728395,0.113624,0.072987,1,144.426355
9,5f28db40,0.027294,modified_huber,elasticnet,adaptive,0.061685,0.717818,0.113608,0.071278,1,210.823066


In [ ]:
# CELL GUIDE:
# Purpose: Cross-validate probability cutoffs for the selected Ray-tuned SGD_13C configuration (compute-only).
# Workflow:
# 1. Cross-validate probability cutoffs for the selected Ray-tuned SGD_13C configuration (compute-only).
# 2. import numpy as np
# 3. import pandas as pd
# 4. from sklearn.linear_model import SGDClassifier
# 5. from sklearn.metrics import f1_score, precision_score, recall_score
# 6. from sklearn.model_selection import cross_val_predict

# Cross-validate probability cutoffs for the selected Ray-tuned SGD_13C configuration (compute-only).
import numpy as np
import pandas as pd
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.model_selection import cross_val_predict
from sklearn.multiclass import OneVsRestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

SGD_13C_threshold_cv_folds = 5
# NOTE: Define a deterministic threshold grid for cutoff-sensitivity analysis.
SGD_13C_probability_cutoffs = np.round(np.linspace(0.05, 0.95, 19), 2)

SGD_13C_selected_trial_id = 'dc9c7a48'
SGD_13C_selected_config = {'alpha': 3.463037026119188e-06,
 'loss': 'log_loss',
 'penalty': 'l2',
 'learning_rate': 'adaptive'}

SGD_13C_threshold_model = make_pipeline(
    StandardScaler(),
    OneVsRestClassifier(
        SGDClassifier(
            loss=SGD_13C_selected_config["loss"],
            alpha=SGD_13C_selected_config["alpha"],
            penalty=SGD_13C_selected_config["penalty"],
            learning_rate=SGD_13C_selected_config["learning_rate"],
            eta0=0.01,
            class_weight="balanced", #important to handle class imbalance in multi-label setting
            max_iter=2000,
            tol=1e-3,
            random_state=42,
        )
    ),
)

SGD_13C_X_train_threshold = SGD_13C_X_train.to_numpy(dtype=np.float32) if hasattr(SGD_13C_X_train, "to_numpy") else np.asarray(SGD_13C_X_train, dtype=np.float32)
SGD_13C_y_train_threshold = SGD_13C_y_train.to_numpy() if hasattr(SGD_13C_y_train, "to_numpy") else np.asarray(SGD_13C_y_train)

# NOTE: Generate out-of-fold predictions to estimate generalization without test leakage.
SGD_13C_y_train_proba = cross_val_predict(
    SGD_13C_threshold_model,
    SGD_13C_X_train_threshold,
    SGD_13C_y_train_threshold,
    cv=SGD_13C_threshold_cv_folds,
    method="predict_proba",
    n_jobs=1,
)

SGD_13C_y_train_proba = np.asarray(SGD_13C_y_train_proba)
if SGD_13C_y_train_proba.ndim == 1:
    SGD_13C_y_train_proba = SGD_13C_y_train_proba[:, None]
SGD_13C_y_true_threshold = np.asarray(SGD_13C_y_train_threshold)
if SGD_13C_y_true_threshold.ndim == 1:
    SGD_13C_y_true_threshold = SGD_13C_y_true_threshold[:, None]

SGD_13C_threshold_metric_rows = []
for cutoff in SGD_13C_probability_cutoffs:
    SGD_13C_y_pred_cutoff = (SGD_13C_y_train_proba >= cutoff).astype(int)
    SGD_13C_threshold_metric_rows.append({
        "cutoff": float(cutoff),
        "f1_micro": f1_score(SGD_13C_y_true_threshold, SGD_13C_y_pred_cutoff, average="micro", zero_division=0),
        "f1_macro": f1_score(SGD_13C_y_true_threshold, SGD_13C_y_pred_cutoff, average="macro", zero_division=0),
        "precision_micro": precision_score(SGD_13C_y_true_threshold, SGD_13C_y_pred_cutoff, average="micro", zero_division=0),
        "recall_micro": recall_score(SGD_13C_y_true_threshold, SGD_13C_y_pred_cutoff, average="micro", zero_division=0),
        "precision_macro": precision_score(SGD_13C_y_true_threshold, SGD_13C_y_pred_cutoff, average="macro", zero_division=0),
        "recall_macro": recall_score(SGD_13C_y_true_threshold, SGD_13C_y_pred_cutoff, average="macro", zero_division=0),
    })


In [ ]:
# CELL GUIDE:
# Purpose: Run `SGD_13C_ray_trial_threshold_cv_df = pd.DataFrame(SGD_13C_threshold_metric_rows).sort_value` and continue downstream processing.
# Workflow:
# 1. SGD_13C_ray_trial_threshold_cv_df = pd.DataFrame(SGD_13C_threshold_metric_rows).sort_values("cutoff").reset_index(dro...
# 2. SGD_13C_best_cutoff_f1_micro = SGD_13C_ray_trial_threshold_cv_df.loc[SGD_13C_ray_trial_threshold_cv_df["f1_micro"].id...
# 3. SGD_13C_best_cutoff_f1_macro = SGD_13C_ray_trial_threshold_cv_df.loc[SGD_13C_ray_trial_threshold_cv_df["f1_macro"].id...
# 4. SGD_13C_optimal_cutoff = 0.45 # my optimised cutoff
# 5. ecg_data.setdefault("ray_tune_sgd_13C", {})["selected_trial_id"] = SGD_13C_selected_trial_id
# 6. ecg_data.setdefault("ray_tune_sgd_13C", {})["selected_config"] = dict(SGD_13C_selected_config)

SGD_13C_ray_trial_threshold_cv_df = pd.DataFrame(SGD_13C_threshold_metric_rows).sort_values("cutoff").reset_index(drop=True)
SGD_13C_best_cutoff_f1_micro = SGD_13C_ray_trial_threshold_cv_df.loc[SGD_13C_ray_trial_threshold_cv_df["f1_micro"].idxmax()]
SGD_13C_best_cutoff_f1_macro = SGD_13C_ray_trial_threshold_cv_df.loc[SGD_13C_ray_trial_threshold_cv_df["f1_macro"].idxmax()]

SGD_13C_optimal_cutoff = 0.45 # my optimised cutoff

# NOTE: Persist artifacts in ecg_data so later cells can reuse them without recomputation.
ecg_data.setdefault("ray_tune_sgd_13C", {})["selected_trial_id"] = SGD_13C_selected_trial_id
# NOTE: Persist artifacts in ecg_data so later cells can reuse them without recomputation.
ecg_data.setdefault("ray_tune_sgd_13C", {})["selected_config"] = dict(SGD_13C_selected_config)
# NOTE: Persist artifacts in ecg_data so later cells can reuse them without recomputation.
ecg_data.setdefault("ray_tune_sgd_13C", {})["threshold_cv"] = {
    "trial_id": SGD_13C_selected_trial_id,
    "config": dict(SGD_13C_selected_config),
    "cv_folds": SGD_13C_threshold_cv_folds,
    "optimal_cutoff": SGD_13C_optimal_cutoff,
    "metrics_by_cutoff": SGD_13C_ray_trial_threshold_cv_df.copy(),
}

SGD_13C_ray_trial_threshold_cv_df[[
    "cutoff", "f1_micro", "f1_macro", "precision_micro", "recall_micro", "precision_macro", "recall_macro"
]].round(4)

,cutoff,f1_micro,f1_macro,precision_micro,recall_micro,precision_macro,recall_macro
0,0.05,0.0590,0.0413,0.0304,0.9939,0.0239,0.7884
1,0.10,0.0625,0.0438,0.0323,0.9901,0.0256,0.7802
2,0.15,0.0661,0.0461,0.0342,0.9835,0.0273,0.7701
3,0.20,0.0701,0.0487,0.0363,0.9738,0.0291,0.7588
4,0.25,0.0747,0.0515,0.0389,0.9585,0.0311,0.7428
5,0.30,0.0800,0.0548,0.0418,0.9351,0.0335,0.7249
6,0.35,0.0863,0.0584,0.0453,0.9007,0.0362,0.7002
7,0.40,0.0940,0.0626,0.0497,0.8550,0.0394,0.6650
8,0.45,0.1030,0.0672,0.0550,0.7965,0.0430,0.6217
9,0.50,0.1129,0.0719,0.0612,0.7225,0.0473,0.5695


In [26]:
# CELL GUIDE:
# Purpose: SGD_13C cutoff visualizations merged into subplots (shared x-axis).
# Workflow:
# 1. SGD_13C cutoff visualizations merged into subplots (shared x-axis).
# 2. import plotly.graph_objects as go
# 3. from plotly.subplots import make_subplots
# 4. SGD_13C_plot_df = SGD_13C_ray_trial_threshold_cv_df.copy()
# 5. SGD_13C_plot_style = graph_formatting if "graph_formatting" in globals() else {
# 6. }

# SGD_13C cutoff visualizations merged into subplots (shared x-axis).
import plotly.graph_objects as go
from plotly.subplots import make_subplots

SGD_13C_plot_df = SGD_13C_ray_trial_threshold_cv_df.copy()
SGD_13C_plot_style = graph_formatting if "graph_formatting" in globals() else {
    "font_family": "Helvetica Neue", "font_size_base": 16, "font_size_axis": 12,
    "color_text": "#b8b8b8", "color_background": "#000000", "margin_top": 170,
    "margin_right": 80, "margin_bottom": 80, "margin_left": 80,
}
SGD_13C_text = SGD_13C_plot_style.get("color_text", "#b8b8b8")
SGD_13C_bg = SGD_13C_plot_style.get("color_background", "#000000")
SGD_13C_grid = "#1f1f1f"

SGD_13C_fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.14,
    subplot_titles=(
        "F1 Micro And F1 Macro",
        "Recall And Precision (Micro And Macro)",
    ),
)

# Left-align subplot subtitles and add breathing room.
for ann in SGD_13C_fig.layout.annotations:
    ann.update(x=0.0, xanchor="left", align="left", xref="paper", y=ann.y + 0.01)

# Top subplot: F1 metrics only.
for metric_name, color in [
    ("f1_micro", SERIES_COLORS["f1_micro"]),
    ("f1_macro", SERIES_COLORS["f1_macro"]),
]:
    SGD_13C_fig.add_trace(
        go.Scatter(
            x=SGD_13C_plot_df["cutoff"],
            y=SGD_13C_plot_df[metric_name],
            mode="lines+markers",
            name=metric_name.replace("_", " ").title(),
            line=dict(width=2.2, color=color),
            marker=dict(size=6, color=color),
            hovertemplate=(
                "Cutoff: %{x:.2f}<br>"
                + f"{metric_name.replace('_', ' ').title()}: %{{y:.4f}}<extra></extra>"
            ),
        ),
        row=1,
        col=1,
    )

# Bottom subplot: recall/precision micro+macro.
for metric_name, color in [
    ("recall_micro", SERIES_COLORS["recall_micro"]),
    ("recall_macro", SERIES_COLORS["recall_macro"]),
    ("precision_micro", SERIES_COLORS["precision_micro"]),
    ("precision_macro", SERIES_COLORS["precision_macro"]),
]:
    SGD_13C_fig.add_trace(
        go.Scatter(
            x=SGD_13C_plot_df["cutoff"],
            y=SGD_13C_plot_df[metric_name],
            mode="lines+markers",
            name=metric_name.replace("_", " ").title(),
            line=dict(width=2.2, color=color),
            marker=dict(size=6, color=color),
            hovertemplate=(
                "Cutoff: %{x:.2f}<br>"
                + f"{metric_name.replace('_', ' ').title()}: %{{y:.4f}}<extra></extra>"
            ),
        ),
        row=2,
        col=1,
    )

# Vertical reference lines at optimal cutoff in both subplots.
SGD_13C_optimal_cutoff = float(SGD_13C_optimal_cutoff)
for row_idx in [1, 2]:
    SGD_13C_fig.add_vline(
        x=SGD_13C_optimal_cutoff,
        line_dash="dash",
        line_color="#8a8a8a",
        line_width=1.6,
        opacity=0.9,
        row=row_idx,
        col=1,
    )

# Dynamic y-range for F1 subplot for better visibility.
SGD_13C_f1_min = float(SGD_13C_plot_df[["f1_micro", "f1_macro"]].min().min())
SGD_13C_f1_max = float(SGD_13C_plot_df[["f1_micro", "f1_macro"]].max().max())
SGD_13C_f1_pad = max(0.01, 0.15 * (SGD_13C_f1_max - SGD_13C_f1_min + 1e-12))

# X-axis label only on bottom subplot (no repetition).
# NOTE: Configure x-axis semantics and formatting for accurate interpretation.
SGD_13C_fig.update_xaxes(
    title_text="",
    tickformat=".2f",
    showgrid=True,
    gridcolor=SGD_13C_grid,
    gridwidth=0.2,
    zeroline=False,
    row=1,
    col=1,
)
# NOTE: Configure x-axis semantics and formatting for accurate interpretation.
SGD_13C_fig.update_xaxes(
    title_text="Probability Cutoff",
    tickformat=".2f",
    showgrid=True,
    gridcolor=SGD_13C_grid,
    gridwidth=0.2,
    zeroline=False,
    row=2,
    col=1,
)

# Remove y-axis title label text in both subplots.
# NOTE: Configure y-axis semantics and formatting for accurate interpretation.
SGD_13C_fig.update_yaxes(
    title_text="",
    range=[max(0.0, SGD_13C_f1_min - SGD_13C_f1_pad), min(1.02, SGD_13C_f1_max + SGD_13C_f1_pad)],
    tickformat=".3f",
    showgrid=True,
    gridcolor=SGD_13C_grid,
    gridwidth=0.2,
    zeroline=False,
    row=1,
    col=1,
)
# NOTE: Configure y-axis semantics and formatting for accurate interpretation.
SGD_13C_fig.update_yaxes(
    title_text="",
    range=[0, 1.02],
    tickformat=".2f",
    showgrid=True,
    gridcolor=SGD_13C_grid,
    gridwidth=0.2,
    zeroline=False,
    row=2,
    col=1,
)

# NOTE: Apply global styling (title, typography, spacing, colors) for readability consistency.
SGD_13C_fig.update_layout(
    title=dict(
        text=(
            "<span style='font-size:30px;font-weight:bold;'>    PCA Could Not Significantly Improve Modest Performance</span>"
            "<br><span style='font-size:20px;font-weight: normal;'>      SCG_13C: Performance Metrics Across Different Cutoffs</span>"
        ),
        x=0,
        xanchor="left",
        pad=dict(t=50),
    ),
    paper_bgcolor=SGD_13C_bg,
    plot_bgcolor=SGD_13C_bg,
    font=dict(family=SGD_13C_plot_style["font_family"], size=SGD_13C_plot_style["font_size_base"], color=SGD_13C_text),
    legend=STANDARD_LEGEND,
    margin=dict(
        t=SGD_13C_plot_style["margin_top"] + 90,
        r=SGD_13C_plot_style["margin_right"],
        b=SGD_13C_plot_style["margin_bottom"],
        l=SGD_13C_plot_style["margin_left"],
    ),
    height=980,
    hovermode="x unified",
)

SGD_13C_fig.show()


# **XGB_18F (Ray-Tuned, Unscaled Features)**


<!-- CELL NOTE -->
> **Cell Note:** This section documents **XGB_18F (Ray-Tuned, Unscaled Features)** and provides context for the following code/results.
> Review it before executing downstream cells so assumptions and outputs remain aligned.


In [47]:
# CELL GUIDE:
# Purpose: Prepare unscaled 18-feature inputs and multilabel targets for XGB_18F using the shared median-imputed 18-feature matrix.
# Workflow:
# 1. Prepare unscaled 18-feature inputs and multilabel targets for XGB_18F using the shared median-imputed 18-feature matrix.
# 2. from sklearn.model_selection import train_test_split
# 3. from sklearn.preprocessing import MultiLabelBinarizer
# 4. required_keys = ["metrics_model_18F", "metadata_model_18F", "dx_code_label_sets_model_18F"]
# 5. missing_keys = [k for k in required_keys if k not in ecg_data]
# 6. if missing_keys

# Prepare unscaled 18-feature inputs and multilabel targets for XGB_18F using the shared median-imputed 18-feature matrix.
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer

required_keys = ["metrics_model_18F", "metadata_model_18F", "dx_code_label_sets_model_18F"]
missing_keys = [k for k in required_keys if k not in ecg_data]
if missing_keys:
    raise RuntimeError(
        f"Run the shared feature-preparation cell after ecg_data['metrics'].isna().sum() first. Missing: {missing_keys}"
    )

XGB_18F_metrics_model_df = ecg_data["metrics_model_18F"].copy()
XGB_18F_metadata_model_df = ecg_data["metadata_model_18F"].copy()
XGB_18F_y_labels = ecg_data["dx_code_label_sets_model_18F"].copy()

# NOTE: Convert per-sample label sets into a multi-hot indicator matrix.
XGB_18F_mlb = MultiLabelBinarizer()
XGB_18F_y = XGB_18F_mlb.fit_transform(XGB_18F_y_labels)
XGB_18F_label_names = [str(lbl) for lbl in XGB_18F_mlb.classes_]

XGB_18F_X = XGB_18F_metrics_model_df.to_numpy(dtype=np.float32)
XGB_18F_y = np.asarray(XGB_18F_y, dtype=np.int32)

XGB_18F_X_train, XGB_18F_X_test, XGB_18F_y_train, XGB_18F_y_test = train_test_split(
    XGB_18F_X,
    XGB_18F_y,
    test_size=0.20,
    random_state=42,
    shuffle=True,
)

# Internal validation split used by Ray Tune (metrics + loss convergence) while keeping the held-out test split untouched.
XGB_18F_X_tune_train, XGB_18F_X_tune_val, XGB_18F_y_tune_train, XGB_18F_y_tune_val = train_test_split(
    XGB_18F_X_train,
    XGB_18F_y_train,
    test_size=0.20,
    random_state=42,
    shuffle=True,
)

print(f"XGB_18F feature matrix shape (unscaled): {XGB_18F_X.shape}")
print(f"XGB_18F feature count: {XGB_18F_X.shape[1]}")
print(f"XGB_18F multilabel target shape: {XGB_18F_y.shape}")
print(f"XGB_18F number of label columns: {XGB_18F_y.shape[1]} (expected ~93)")
print(f"XGB_18F train shape: {XGB_18F_X_train.shape}, test shape: {XGB_18F_X_test.shape}")
print(f"XGB_18F ray-tune train/val shapes: {XGB_18F_X_tune_train.shape}, {XGB_18F_X_tune_val.shape}")

# NOTE: Persist artifacts in ecg_data so later cells can reuse them without recomputation.
ecg_data.setdefault("xgb_18F", {})["feature_columns"] = list(XGB_18F_metrics_model_df.columns)
# NOTE: Persist artifacts in ecg_data so later cells can reuse them without recomputation.
ecg_data.setdefault("xgb_18F", {})["label_columns"] = XGB_18F_label_names


XGB_18F feature matrix shape (unscaled): (45152, 18)
XGB_18F feature count: 18
XGB_18F multilabel target shape: (45152, 94)
XGB_18F number of label columns: 94 (expected ~93)
XGB_18F train shape: (36121, 18), test shape: (9031, 18)
XGB_18F ray-tune train/val shapes: (28896, 18), (7225, 18)


In [48]:
# CELL GUIDE:
# Purpose: Ray Tune hyperparameter optimization for XGB_18F using One-vs-Rest XGBClassifier (binary logistic) and micro/macro metrics.
# Workflow:
# 1. Ray Tune hyperparameter optimization for XGB_18F using One-vs-Rest XGBClassifier (binary logistic) and micro/macro me...
# 2. import ray
# 3. import time
# 4. import random
# 5. from ray import tune
# 6. from ray.tune.schedulers import ASHAScheduler

# Ray Tune hyperparameter optimization for XGB_18F using One-vs-Rest XGBClassifier (binary logistic) and micro/macro metrics.
import ray
import time
import random
from ray import tune
from ray.tune.schedulers import ASHAScheduler
from sklearn.metrics import f1_score, precision_score, recall_score, log_loss
from sklearn.multiclass import OneVsRestClassifier
from xgboost import XGBClassifier

XGB_18F_BACKEND = "xgboost"
XGB_18F_num_ray_trials = 30
XGB_18F_objective_metric = "f1_macro" # Prioritize balanced performance across classes for the main selection criterion.
print(f"XGB_18F backend: {XGB_18F_BACKEND}")

param_space = {
    "max_depth": tune.choice([3, 4, 5, 6, 8]),
    "min_child_weight": tune.choice([1, 2, 5, 10]),
    "gamma": tune.uniform(0.0, 5.0),
    "subsample": tune.uniform(0.6, 1.0),
    "colsample_bytree": tune.uniform(0.6, 1.0),
    "reg_lambda": tune.loguniform(1e-2, 1e2),
    "reg_alpha": tune.loguniform(1e-8, 1.0),
    "learning_rate": tune.loguniform(1e-2, 3e-1),
    "n_estimators": tune.choice([300, 600, 900, 1200, 1600]),
}


def _xgb18f_ovr_model_from_config(config):
# NOTE: Instantiate gradient-boosted tree learner with current hyperparameters.
    base = XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        random_state=42,
        n_jobs=1,
        nthread=1,
        max_depth=int(config["max_depth"]),
        min_child_weight=float(config["min_child_weight"]),
        gamma=float(config["gamma"]),
        subsample=float(config["subsample"]),
        colsample_bytree=float(config["colsample_bytree"]),
        reg_lambda=float(config["reg_lambda"]),
        reg_alpha=float(config["reg_alpha"]),
        learning_rate=float(config["learning_rate"]),
        n_estimators=int(config["n_estimators"]),
    )
    return OneVsRestClassifier(base)


def _xgb18f_mean_label_logloss(y_true, y_proba):
    y_true = np.asarray(y_true, dtype=np.int32)
    y_proba = np.asarray(y_proba, dtype=np.float64)
    if y_proba.ndim == 1:
        y_proba = y_proba[:, None]

    losses = []
    for j in range(y_true.shape[1]):
        yj = y_true[:, j]
        pj = np.clip(y_proba[:, j], 1e-12, 1 - 1e-12)
        try:
            losses.append(log_loss(yj, pj, labels=[0, 1]))
        except ValueError:
            losses.append(float(np.mean(-(yj * np.log(pj) + (1 - yj) * np.log(1 - pj)))))
    return float(np.mean(losses)) if losses else np.nan


def XGB_18F_ray_objective(config, X_fit, y_fit, X_eval, y_eval):
    X_fit = np.asarray(X_fit, dtype=np.float32)
    X_eval = np.asarray(X_eval, dtype=np.float32)
    y_fit = np.asarray(y_fit, dtype=np.int32)
    y_eval = np.asarray(y_eval, dtype=np.int32)

    model = _xgb18f_ovr_model_from_config(config)
    model.fit(X_fit, y_fit)

    y_pred = model.predict(X_eval)
    y_proba = model.predict_proba(X_eval)

    metrics = {
        "precision_micro": precision_score(y_eval, y_pred, average="micro", zero_division=0),
        "precision_macro": precision_score(y_eval, y_pred, average="macro", zero_division=0),
        "recall_micro": recall_score(y_eval, y_pred, average="micro", zero_division=0),
        "recall_macro": recall_score(y_eval, y_pred, average="macro", zero_division=0),
        "f1_micro": f1_score(y_eval, y_pred, average="micro", zero_division=0),
        "f1_macro": f1_score(y_eval, y_pred, average="macro", zero_division=0),
        "val_logloss": _xgb18f_mean_label_logloss(y_eval, y_proba),
    }
    tune.report(metrics)


class XGB_18F_ETACallback(tune.Callback):
    def __init__(self, total_trials):
        self.total_trials = int(total_trials)
        self.completed_trials = 0
        self.start_time = time.time()

    def on_trial_complete(self, iteration, trials, trial, **info):
        self.completed_trials += 1
        elapsed = max(time.time() - self.start_time, 1e-6)
        avg_per_trial = elapsed / max(self.completed_trials, 1)
        remaining_trials = max(self.total_trials - self.completed_trials, 0)
        eta_seconds = int(avg_per_trial * remaining_trials)
        eta_min, eta_sec = divmod(eta_seconds, 60)
        print(
            f"[XGB_18F][ETA] completed {self.completed_trials}/{self.total_trials} trials | "
            f"remaining {remaining_trials} | ETA ~ {eta_min}m {eta_sec}s"
        )


np.random.seed(42)
random.seed(42)

try:
    from ray.tune.search.optuna import OptunaSearch
    XGB_18F_search_alg = OptunaSearch(seed=42)
    XGB_18F_search_backend = "optuna"
except Exception:
    from ray.tune.search.hyperopt import HyperOptSearch
    XGB_18F_search_alg = HyperOptSearch(random_state_seed=42)
    XGB_18F_search_backend = "hyperopt"

XGB_18F_scheduler = ASHAScheduler(
    max_t=1,
    grace_period=1,
    reduction_factor=2,
)
print(f"XGB_18F search backend: {XGB_18F_search_backend}")

if ray.is_initialized():
    ray.shutdown()
ray.init(ignore_reinit_error=True, include_dashboard=False, log_to_driver=False)

XGB_18F_eta_callback = XGB_18F_ETACallback(XGB_18F_num_ray_trials)

# NOTE: Launch Ray Tune hyperparameter search over the configured trial space.
XGB_18F_ray_analysis = tune.run(
    tune.with_parameters(
        XGB_18F_ray_objective,
        X_fit=XGB_18F_X_tune_train,
        y_fit=XGB_18F_y_tune_train,
        X_eval=XGB_18F_X_tune_val,
        y_eval=XGB_18F_y_tune_val,
    ),
    config=param_space,
    num_samples=XGB_18F_num_ray_trials,
    metric=XGB_18F_objective_metric,
    mode="max",
    resources_per_trial={"cpu": 1},
    callbacks=[XGB_18F_eta_callback],
    search_alg=XGB_18F_search_alg,
    scheduler=XGB_18F_scheduler,
    verbose=2,
)

XGB_18F_ray_tune_results_df = XGB_18F_ray_analysis.dataframe().copy().rename(
    columns={
        "config/max_depth": "max_depth",
        "config/min_child_weight": "min_child_weight",
        "config/gamma": "gamma",
        "config/subsample": "subsample",
        "config/colsample_bytree": "colsample_bytree",
        "config/reg_lambda": "reg_lambda",
        "config/reg_alpha": "reg_alpha",
        "config/learning_rate": "learning_rate",
        "config/n_estimators": "n_estimators",
    }
)

for metric_name in [
    "precision_micro", "precision_macro", "recall_micro", "recall_macro",
    "f1_micro", "f1_macro", "val_logloss"
]:
    if metric_name not in XGB_18F_ray_tune_results_df.columns:
        for alt_col in (f"last_result/{metric_name}", f"last_result.{metric_name}"):
            if alt_col in XGB_18F_ray_tune_results_df.columns:
                XGB_18F_ray_tune_results_df[metric_name] = XGB_18F_ray_tune_results_df[alt_col]
                break

XGB_18F_keep_cols = [
    c for c in [
        "trial_id",
        "max_depth", "min_child_weight", "gamma", "subsample", "colsample_bytree",
        "reg_lambda", "reg_alpha", "learning_rate", "n_estimators",
        "precision_micro", "precision_macro", "recall_micro", "recall_macro",
        "f1_micro", "f1_macro", "val_logloss",
        "training_iteration", "time_total_s"
    ] if c in XGB_18F_ray_tune_results_df.columns
]
XGB_18F_ray_tune_results_df = XGB_18F_ray_tune_results_df[XGB_18F_keep_cols].copy()

XGB_18F_ray_tune_results_df = XGB_18F_ray_tune_results_df.sort_values(
    [c for c in ["f1_macro", "f1_micro", "val_logloss"] if c in XGB_18F_ray_tune_results_df.columns],
    ascending=[False, False, True][:len([c for c in ["f1_macro", "f1_micro", "val_logloss"] if c in XGB_18F_ray_tune_results_df.columns])],
    na_position="last",
).reset_index(drop=True)

if XGB_18F_ray_tune_results_df.empty:
    raise RuntimeError("XGB_18F Ray Tune returned no trial results.")

XGB_18F_selected_trial_row = XGB_18F_ray_tune_results_df.iloc[0]
XGB_18F_selected_trial_id = str(XGB_18F_selected_trial_row["trial_id"])
XGB_18F_selected_config = {
    "max_depth": int(XGB_18F_selected_trial_row["max_depth"]),
    "min_child_weight": float(XGB_18F_selected_trial_row["min_child_weight"]),
    "gamma": float(XGB_18F_selected_trial_row["gamma"]),
    "subsample": float(XGB_18F_selected_trial_row["subsample"]),
    "colsample_bytree": float(XGB_18F_selected_trial_row["colsample_bytree"]),
    "reg_lambda": float(XGB_18F_selected_trial_row["reg_lambda"]),
    "reg_alpha": float(XGB_18F_selected_trial_row["reg_alpha"]),
    "learning_rate": float(XGB_18F_selected_trial_row["learning_rate"]),
    "n_estimators": int(XGB_18F_selected_trial_row["n_estimators"]),
}

print("XGB_18F selected trial/config (OVR + binary logistic):")
print({"trial_id": XGB_18F_selected_trial_id, **XGB_18F_selected_config})

# NOTE: Persist artifacts in ecg_data so later cells can reuse them without recomputation.
ecg_data.setdefault("xgb_18F", {})["ray_results"] = XGB_18F_ray_tune_results_df.copy()
# NOTE: Persist artifacts in ecg_data so later cells can reuse them without recomputation.
ecg_data.setdefault("xgb_18F", {})["selected_trial_id"] = XGB_18F_selected_trial_id
# NOTE: Persist artifacts in ecg_data so later cells can reuse them without recomputation.
ecg_data.setdefault("xgb_18F", {})["selected_config"] = dict(XGB_18F_selected_config)
# NOTE: Persist artifacts in ecg_data so later cells can reuse them without recomputation.
ecg_data.setdefault("xgb_18F", {})["num_trials"] = XGB_18F_num_ray_trials

XGB_18F_ray_tune_results_df


XGB_18F backend: xgboost
XGB_18F search backend: optuna


python(88560) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(88562) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(88564) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(88565) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(88575) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(88576) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
2026-03-02 16:28:03,365	INFO worker.py:1951 -- Started a local Ray instance.
2026-03-02 16:28:06,192	INFO tune.py:616 -- [output] This uses the legacy output and progress reporter, as Jupyter notebooks are not supported by the new engine, yet. For more information, please see https://github.com/ray-project/ray/issues/36949
[I 2026-03-02 16:28:06,228] A new study created in memory with name: optuna
2026-03-02 

Trial name,f1_macro,f1_micro,precision_macro,precision_micro,recall_macro,recall_micro,val_logloss
XGB_18F_ray_objective_0171aede,0.077749,0.534449,0.171886,0.78699,0.0641041,0.404611,0.0488902
XGB_18F_ray_objective_06e332e0,0.0804754,0.530553,0.136602,0.742543,0.0669179,0.412723,0.0519316
XGB_18F_ray_objective_1a5e712b,0.0829858,0.529103,0.157526,0.735988,0.0687223,0.413008,0.0522086
XGB_18F_ray_objective_1b4c5e0e,0.0536992,0.472405,0.129134,0.832136,0.0445342,0.329823,0.0497261
XGB_18F_ray_objective_368c94fc,0.0811059,0.52905,0.155165,0.754814,0.0664845,0.407244,0.0509474
XGB_18F_ray_objective_374cb0e8,0.082811,0.533303,0.160211,0.758933,0.0680786,0.411087,0.0509372
XGB_18F_ray_objective_37caef1b,0.0763175,0.52425,0.205734,0.821439,0.0619963,0.384971,0.0478249
XGB_18F_ray_objective_3b061391,0.0803225,0.527259,0.147588,0.744708,0.0663273,0.408098,0.0522302
XGB_18F_ray_objective_3c152f1f,0.0801927,0.53392,0.170273,0.765378,0.0660573,0.409948,0.0505423
XGB_18F_ray_objective_4881f8b4,0.0816897,0.527401,0.188595,0.772877,0.0652119,0.40027,0.0513556


[XGB_18F][ETA] completed 1/30 trials | remaining 29 | ETA ~ 65m 55s
[XGB_18F][ETA] completed 2/30 trials | remaining 28 | ETA ~ 43m 28s
[XGB_18F][ETA] completed 3/30 trials | remaining 27 | ETA ~ 30m 38s
[XGB_18F][ETA] completed 4/30 trials | remaining 26 | ETA ~ 28m 29s
[XGB_18F][ETA] completed 5/30 trials | remaining 25 | ETA ~ 27m 48s
[XGB_18F][ETA] completed 6/30 trials | remaining 24 | ETA ~ 24m 14s
[XGB_18F][ETA] completed 7/30 trials | remaining 23 | ETA ~ 22m 39s
[XGB_18F][ETA] completed 8/30 trials | remaining 22 | ETA ~ 23m 27s
[XGB_18F][ETA] completed 9/30 trials | remaining 21 | ETA ~ 20m 46s
[XGB_18F][ETA] completed 10/30 trials | remaining 20 | ETA ~ 19m 24s
[XGB_18F][ETA] completed 11/30 trials | remaining 19 | ETA ~ 17m 57s
[XGB_18F][ETA] completed 12/30 trials | remaining 18 | ETA ~ 16m 38s
[XGB_18F][ETA] completed 13/30 trials | remaining 17 | ETA ~ 14m 31s
[XGB_18F][ETA] completed 14/30 trials | remaining 16 | ETA ~ 13m 27s
[XGB_18F][ETA] completed 15/30 trials | rem

2026-03-02 16:55:58,932	INFO tune.py:1009 -- Wrote the latest version of all result files and experiment state to '/Users/felixaugustin/ray_results/XGB_18F_ray_objective_2026-03-02_16-28-06' in 0.0225s.


[XGB_18F][ETA] completed 30/30 trials | remaining 0 | ETA ~ 0m 0s


2026-03-02 16:55:58,946	INFO tune.py:1041 -- Total run time: 1672.75 seconds (1672.65 seconds for the tuning loop).


XGB_18F selected trial/config (OVR + binary logistic):
{'trial_id': '49f3eac4', 'max_depth': 6, 'min_child_weight': 5.0, 'gamma': 0.8040402570874933, 'subsample': 0.8194935157466344, 'colsample_bytree': 0.8767580790770773, 'reg_lambda': 4.053638704742312, 'reg_alpha': 6.225216726492816e-07, 'learning_rate': 0.11271327400947749, 'n_estimators': 1600}


,trial_id,max_depth,min_child_weight,gamma,subsample,colsample_bytree,reg_lambda,reg_alpha,learning_rate,n_estimators,precision_micro,precision_macro,recall_micro,recall_macro,f1_micro,f1_macro,val_logloss,training_iteration,time_total_s
0,49f3eac4,6,5,0.804040,0.819494,0.876758,4.053639,6.225217e-07,0.112713,1600,0.756149,0.190542,0.411300,0.068220,0.532793,0.083746,0.052669,1,612.868018
1,fce65176,8,10,0.043430,0.803906,0.622590,0.111966,2.185608e-08,0.290895,1600,0.692961,0.152331,0.418203,0.068550,0.521612,0.083196,0.075001,1,716.920764
2,1a5e712b,6,10,2.145990,0.746183,0.841735,0.204156,4.537736e-08,0.194663,1600,0.735988,0.157526,0.413008,0.068722,0.529103,0.082986,0.052209,1,439.349983
3,374cb0e8,6,10,1.998163,0.811674,0.834239,0.218421,1.077991e-08,0.169824,1600,0.758933,0.160211,0.411087,0.068079,0.533303,0.082811,0.050937,1,346.821763
4,a0d5a401,8,10,2.270486,0.742943,0.754385,0.125297,3.767458e-08,0.186188,1600,0.745887,0.148790,0.412937,0.068388,0.531581,0.082717,0.051782,1,501.083749
5,e80185a1,8,10,1.967856,0.763144,0.753820,0.104226,3.691331e-08,0.176466,1600,0.746159,0.157823,0.411229,0.068000,0.530232,0.082200,0.051991,1,512.647858
6,676131a9,4,5,2.553737,0.766964,0.688843,0.030162,5.022516e-06,0.247054,1600,0.742490,0.141403,0.411585,0.068191,0.529598,0.081985,0.052229,1,486.025836
7,622ae167,6,10,2.298356,0.779082,0.829979,0.138650,1.020887e-08,0.202416,1600,0.745123,0.160060,0.413150,0.067782,0.531563,0.081920,0.051350,1,454.899090
8,4881f8b4,8,1,1.354161,0.775589,0.631383,0.012630,5.025593e-01,0.171729,300,0.772877,0.188595,0.400270,0.065212,0.527401,0.081690,0.051356,1,141.296836
9,b9585af6,3,1,0.232252,0.843018,0.668210,0.018206,3.900177e-01,0.266904,300,0.758106,0.159753,0.397638,0.066095,0.521658,0.081354,0.052381,1,125.562182


In [79]:
# CELL GUIDE:
# Purpose: XGB_18F target metric over time (chronological).
# Workflow:
# 1. XGB_18F target metric over time (chronological)
# 2. import numpy as np
# 3. import pandas as pd
# 4. import plotly.graph_objects as go
# 5. XGB_18F_results_source = None
# 6. if "XGB_18F_ray_tune_results_df" in globals()

# XGB_18F target metric over time (chronological)
import numpy as np
import pandas as pd
import plotly.graph_objects as go

XGB_18F_results_source = None
if "XGB_18F_ray_tune_results_df" in globals():
    XGB_18F_results_source = XGB_18F_ray_tune_results_df
elif isinstance(ecg_data, dict):
    XGB_18F_results_source = (ecg_data.get("xgb_18F") or {}).get("ray_results")

if XGB_18F_results_source is None:
    raise RuntimeError("XGB_18F Ray results were not found. Run the XGB_18F tuning cell first.")

XGB_18F_time_df = XGB_18F_results_source.copy()
if XGB_18F_time_df.empty:
    raise RuntimeError("XGB_18F Ray results are empty.")

XGB_18F_target_metric = "f1_macro"
if XGB_18F_target_metric not in XGB_18F_time_df.columns:
    if "f1_micro" in XGB_18F_time_df.columns:
        XGB_18F_target_metric = "f1_micro"
    else:
        raise KeyError("Neither f1_macro nor f1_micro is available in XGB_18F Ray results.")

XGB_18F_time_sort_col = None
if "timestamp" in XGB_18F_time_df.columns:
    XGB_18F_time_df["_chrono"] = pd.to_numeric(XGB_18F_time_df["timestamp"], errors="coerce")
    XGB_18F_time_sort_col = "_chrono"
elif "date" in XGB_18F_time_df.columns:
    XGB_18F_time_df["_chrono"] = pd.to_datetime(XGB_18F_time_df["date"], errors="coerce")
    XGB_18F_time_sort_col = "_chrono"
elif "time_total_s" in XGB_18F_time_df.columns:
    XGB_18F_time_df["_chrono"] = pd.to_numeric(XGB_18F_time_df["time_total_s"], errors="coerce")
    XGB_18F_time_sort_col = "_chrono"
elif "training_iteration" in XGB_18F_time_df.columns:
    XGB_18F_time_df["_chrono"] = pd.to_numeric(XGB_18F_time_df["training_iteration"], errors="coerce")
    XGB_18F_time_sort_col = "_chrono"

if XGB_18F_time_sort_col is not None:
    XGB_18F_time_df = XGB_18F_time_df.sort_values(XGB_18F_time_sort_col, ascending=True, na_position="last")

XGB_18F_time_df = XGB_18F_time_df.reset_index(drop=True)
XGB_18F_time_df["trial_order"] = np.arange(1, len(XGB_18F_time_df) + 1)
XGB_18F_time_df["trial_id"] = XGB_18F_time_df.get("trial_id", "").astype(str)

XGB_18F_plot_style = graph_formatting if "graph_formatting" in globals() else {
    "font_family": "Helvetica Neue", "font_size_base": 16, "font_size_axis": 12,
    "color_text": "#b8b8b8", "color_background": "#000000",
    "margin_top": 170, "margin_right": 80, "margin_bottom": 80, "margin_left": 80,
}
XGB_18F_text = XGB_18F_plot_style.get("color_text", "#b8b8b8")
XGB_18F_bg = XGB_18F_plot_style.get("color_background", "#000000")
XGB_18F_color = SERIES_COLORS.get("f1_macro", "#636EFA") if "SERIES_COLORS" in globals() else "#636EFA"

XGB_18F_time_fig = go.Figure(data=[
    go.Scatter(
        x=XGB_18F_time_df["trial_order"],
        y=XGB_18F_time_df[XGB_18F_target_metric],
        mode="lines+markers",
        line=dict(width=2.2, color=XGB_18F_color),
        marker=dict(size=6, color=XGB_18F_color),
        customdata=np.column_stack([XGB_18F_time_df["trial_id"]]),
        hovertemplate=(
            "Chronological Order: %{x}<br>"
            "Trial ID: %{customdata[0]}<br>"
            + f"{XGB_18F_target_metric.upper()}: %{{y:.4f}}<extra></extra>"
        ),
        name="XGB_18F",
    )
])

# NOTE: Configure x-axis semantics and formatting for accurate interpretation.
XGB_18F_time_fig.update_xaxes(title_text="Chronological Trial Order", title_font=dict(family=XGB_18F_plot_style["font_family"], size=XGB_18F_plot_style["font_size_axis"], color=XGB_18F_text), tickfont=dict(family=XGB_18F_plot_style["font_family"], size=XGB_18F_plot_style["font_size_axis"], color=XGB_18F_text), showgrid=True, gridcolor="#1f1f1f", gridwidth=0.2)
# NOTE: Configure y-axis semantics and formatting for accurate interpretation.
XGB_18F_time_fig.update_yaxes(title_font=dict(family=XGB_18F_plot_style["font_family"], size=XGB_18F_plot_style["font_size_axis"], color=XGB_18F_text), tickfont=dict(family=XGB_18F_plot_style["font_family"], size=XGB_18F_plot_style["font_size_axis"], color=XGB_18F_text), showgrid=True, gridcolor="#1f1f1f", gridwidth=0.2)

# NOTE: Apply global styling (title, typography, spacing, colors) for readability consistency.
XGB_18F_time_fig.update_layout(
    title=dict(text=("<span style='font-size:30px;font-weight:bold;'>   F1 Macro Converges Upwards, While Finding Maximum Towards The End</span>" "<br><span style='font-size:20px;font-weight: normal;'>    XGB_18F Target Metric - F1 Macro During Each Ray Tune Trial</span>"), x=0, xanchor="left", pad=dict(t=50)),
    paper_bgcolor=XGB_18F_bg,
    plot_bgcolor=XGB_18F_bg,
    font=dict(family=XGB_18F_plot_style["font_family"], size=XGB_18F_plot_style["font_size_base"], color=XGB_18F_text),
    hoverlabel=dict(font=dict(family=XGB_18F_plot_style["font_family"], size=XGB_18F_plot_style["font_size_axis"], color=XGB_18F_text)),
    legend=STANDARD_LEGEND if "STANDARD_LEGEND" in globals() else dict(orientation="h", yanchor="top", y=1.02, xanchor="left", x=0),
    margin=dict(t=XGB_18F_plot_style["margin_top"], r=XGB_18F_plot_style["margin_right"], b=XGB_18F_plot_style["margin_bottom"], l=XGB_18F_plot_style["margin_left"]),
    height=700,
)

XGB_18F_time_fig.show()


In [ ]:
# CELL GUIDE:
# Purpose: Cross-validate probability cutoffs for the selected Ray-tuned XGB_18F configuration (compute-only, OVR binary logistic).
# Workflow:
# 1. Cross-validate probability cutoffs for the selected Ray-tuned XGB_18F configuration (compute-only, OVR binary logistic).
# 2. from sklearn.metrics import f1_score, precision_score, recall_score
# 3. from sklearn.model_selection import cross_val_predict
# 4. from sklearn.multiclass import OneVsRestClassifier
# 5. from xgboost import XGBClassifier
# 6. XGB_18F_threshold_cv_folds = 5

# Cross-validate probability cutoffs for the selected Ray-tuned XGB_18F configuration (compute-only, OVR binary logistic).
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.model_selection import cross_val_predict
from sklearn.multiclass import OneVsRestClassifier
from xgboost import XGBClassifier

XGB_18F_threshold_cv_folds = 5
# NOTE: Define a deterministic threshold grid for cutoff-sensitivity analysis.
XGB_18F_probability_cutoffs = np.round(np.linspace(0.05, 0.95, 19), 2)

XGB_18F_selected_config = {'max_depth': 6, # my optimised config for cutoff sweep
 'min_child_weight': 5.0,
 'gamma': 0.8040402570874933,
 'subsample': 0.8194935157466344,
 'colsample_bytree': 0.8767580790770773,
 'reg_lambda': 4.053638704742312,
 'reg_alpha': 6.225216726492816e-07,
 'learning_rate': 0.11271327400947749,
 'n_estimators': 1600}

XGB_18F_selected_trial_id = '49f3eac4' #frozen to my optimised config for cutoff sweep

if not XGB_18F_selected_config:
    raise RuntimeError("Run the XGB_18F Ray tuning cell first to select a configuration.")

# NOTE: Wrap binary base learner for multi-label one-vs-rest training.
XGB_18F_cv_model = OneVsRestClassifier(
    XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        random_state=42,
        n_jobs=1,
        nthread=1,
        **XGB_18F_selected_config,
    )
)

XGB_18F_X_train_cv = np.asarray(XGB_18F_X_train, dtype=np.float32)
XGB_18F_y_train_cv = np.asarray(XGB_18F_y_train, dtype=np.int32)

# NOTE: Generate out-of-fold predictions to estimate generalization without test leakage.
XGB_18F_oof_proba = cross_val_predict(
    XGB_18F_cv_model,
    XGB_18F_X_train_cv,
    XGB_18F_y_train_cv,
    cv=XGB_18F_threshold_cv_folds,
    method="predict_proba",
    n_jobs=1,
)

XGB_18F_oof_proba = np.asarray(XGB_18F_oof_proba)
if XGB_18F_oof_proba.ndim == 1:
    XGB_18F_oof_proba = XGB_18F_oof_proba[:, None]

XGB_18F_threshold_metric_rows = []
for cutoff in XGB_18F_probability_cutoffs:
    y_pred_cutoff = (XGB_18F_oof_proba >= cutoff).astype(int)
    XGB_18F_threshold_metric_rows.append({
        "cutoff": float(cutoff),
        "f1_micro": f1_score(XGB_18F_y_train_cv, y_pred_cutoff, average="micro", zero_division=0),
        "f1_macro": f1_score(XGB_18F_y_train_cv, y_pred_cutoff, average="macro", zero_division=0),
        "precision_micro": precision_score(XGB_18F_y_train_cv, y_pred_cutoff, average="micro", zero_division=0),
        "recall_micro": recall_score(XGB_18F_y_train_cv, y_pred_cutoff, average="micro", zero_division=0),
        "precision_macro": precision_score(XGB_18F_y_train_cv, y_pred_cutoff, average="macro", zero_division=0),
        "recall_macro": recall_score(XGB_18F_y_train_cv, y_pred_cutoff, average="macro", zero_division=0),
    })



/opt/anaconda3/lib/python3.12/site-packages/sklearn/multiclass.py:90: UserWarning:

Label not 24 is present in all training examples.

/opt/anaconda3/lib/python3.12/site-packages/sklearn/multiclass.py:90: UserWarning:

Label not 82 is present in all training examples.



In [ ]:
# CELL GUIDE:
# Purpose: Run `XGB_18F_threshold_cv_df = pd.DataFrame(XGB_18F_threshold_metric_rows).sort_values("cutoff"` and continue downstream processing.
# Workflow:
# 1. XGB_18F_threshold_cv_df = pd.DataFrame(XGB_18F_threshold_metric_rows).sort_values("cutoff").reset_index(drop=True)
# 2. XGB_18F_best_cutoff_f1_micro = XGB_18F_threshold_cv_df.loc[XGB_18F_threshold_cv_df["f1_micro"].idxmax()]
# 3. XGB_18F_best_cutoff_f1_macro = XGB_18F_threshold_cv_df.loc[XGB_18F_threshold_cv_df["f1_macro"].idxmax()]
# 4. XGB_18F_optimal_cutoff = 0.3 # my optimised cutoff based on the above results and visualizations
# 5. ecg_data.setdefault("xgb_18F", {})["threshold_cv"] = {
# 6. }

XGB_18F_threshold_cv_df = pd.DataFrame(XGB_18F_threshold_metric_rows).sort_values("cutoff").reset_index(drop=True)
XGB_18F_best_cutoff_f1_micro = XGB_18F_threshold_cv_df.loc[XGB_18F_threshold_cv_df["f1_micro"].idxmax()]
XGB_18F_best_cutoff_f1_macro = XGB_18F_threshold_cv_df.loc[XGB_18F_threshold_cv_df["f1_macro"].idxmax()]
XGB_18F_optimal_cutoff = 0.3 # my optimised cutoff based on the above results and visualizations

# NOTE: Persist artifacts in ecg_data so later cells can reuse them without recomputation.
ecg_data.setdefault("xgb_18F", {})["threshold_cv"] = {
    "trial_id": XGB_18F_selected_trial_id,
    "config": dict(XGB_18F_selected_config),
    "cv_folds": XGB_18F_threshold_cv_folds,
    "optimal_cutoff": XGB_18F_optimal_cutoff,
    "metrics_by_cutoff": XGB_18F_threshold_cv_df.copy(),
}

XGB_18F_threshold_cv_df[["cutoff", "f1_micro", "f1_macro", "precision_micro", "recall_micro", "precision_macro", "recall_macro"]].round(4)

,cutoff,f1_micro,f1_macro,precision_micro,recall_micro,precision_macro,recall_macro
0,0.05,0.4292,0.1228,0.3067,0.7147,0.0909,0.2012
1,0.10,0.4951,0.1241,0.4058,0.6348,0.1097,0.1524
2,0.15,0.5253,0.1206,0.4780,0.5831,0.1240,0.1290
3,0.20,0.5404,0.1137,0.5361,0.5448,0.1327,0.1124
4,0.25,0.5473,0.1095,0.5843,0.5147,0.1474,0.1021
5,0.30,0.5489,0.1045,0.6270,0.4881,0.1576,0.0934
6,0.35,0.5476,0.0992,0.6646,0.4656,0.1705,0.0859
7,0.40,0.5443,0.0947,0.7003,0.4451,0.1771,0.0799
8,0.45,0.5378,0.0894,0.7304,0.4255,0.1769,0.0742
9,0.50,0.5298,0.0850,0.7579,0.4073,0.1875,0.0693


In [51]:
# CELL GUIDE:
# Purpose: XGB_18F cutoff visualizations merged into subplots (shared x-axis).
# Workflow:
# 1. XGB_18F cutoff visualizations merged into subplots (shared x-axis).
# 2. import plotly.graph_objects as go
# 3. from plotly.subplots import make_subplots
# 4. XGB_18F_plot_df = XGB_18F_threshold_cv_df.copy()
# 5. XGB_18F_plot_style = graph_formatting if "graph_formatting" in globals() else {
# 6. }

# XGB_18F cutoff visualizations merged into subplots (shared x-axis).
import plotly.graph_objects as go
from plotly.subplots import make_subplots

XGB_18F_plot_df = XGB_18F_threshold_cv_df.copy()
XGB_18F_plot_style = graph_formatting if "graph_formatting" in globals() else {
    "font_family": "Helvetica Neue", "font_size_base": 16, "font_size_axis": 12,
    "color_text": "#b8b8b8", "color_background": "#000000", "margin_top": 170,
    "margin_right": 80, "margin_bottom": 80, "margin_left": 80,
}
XGB_18F_text = XGB_18F_plot_style.get("color_text", "#b8b8b8")
XGB_18F_bg = XGB_18F_plot_style.get("color_background", "#000000")
XGB_18F_grid = "#1f1f1f"

XGB_18F_fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.14,
    subplot_titles=(
        "F1 Micro And F1 Macro",
        "Recall And Precision (Micro And Macro)",
    ),
)

# Left-align subplot subtitles and add breathing room.
for ann in XGB_18F_fig.layout.annotations:
    ann.update(x=0.0, xanchor="left", align="left", xref="paper", y=ann.y + 0.01)

# Top subplot: F1 metrics only.
for metric_name, color in [
    ("f1_micro", SERIES_COLORS["f1_micro"]),
    ("f1_macro", SERIES_COLORS["f1_macro"]),
]:
    XGB_18F_fig.add_trace(
        go.Scatter(
            x=XGB_18F_plot_df["cutoff"],
            y=XGB_18F_plot_df[metric_name],
            mode="lines+markers",
            name=metric_name.replace("_", " ").title(),
            line=dict(width=2.2, color=color),
            marker=dict(size=6, color=color),
            hovertemplate=(
                "Cutoff: %{x:.2f}<br>"
                + f"{metric_name.replace('_', ' ').title()}: %{{y:.4f}}<extra></extra>"
            ),
        ),
        row=1,
        col=1,
    )

# Bottom subplot: recall/precision micro+macro.
for metric_name, color in [
    ("recall_micro", SERIES_COLORS["recall_micro"]),
    ("recall_macro", SERIES_COLORS["recall_macro"]),
    ("precision_micro", SERIES_COLORS["precision_micro"]),
    ("precision_macro", SERIES_COLORS["precision_macro"]),
]:
    XGB_18F_fig.add_trace(
        go.Scatter(
            x=XGB_18F_plot_df["cutoff"],
            y=XGB_18F_plot_df[metric_name],
            mode="lines+markers",
            name=metric_name.replace("_", " ").title(),
            line=dict(width=2.2, color=color),
            marker=dict(size=6, color=color),
            hovertemplate=(
                "Cutoff: %{x:.2f}<br>"
                + f"{metric_name.replace('_', ' ').title()}: %{{y:.4f}}<extra></extra>"
            ),
        ),
        row=2,
        col=1,
    )

# Vertical reference lines at optimal cutoff in both subplots.
XGB_18F_optimal_cutoff = float(XGB_18F_optimal_cutoff)
for row_idx in [1, 2]:
    XGB_18F_fig.add_vline(
        x=XGB_18F_optimal_cutoff,
        line_dash="dash",
        line_color="#8a8a8a",
        line_width=1.6,
        opacity=0.9,
        row=row_idx,
        col=1,
    )

# Dynamic y-range for F1 subplot for better visibility.
XGB_18F_f1_min = float(XGB_18F_plot_df[["f1_micro", "f1_macro"]].min().min())
XGB_18F_f1_max = float(XGB_18F_plot_df[["f1_micro", "f1_macro"]].max().max())
XGB_18F_f1_pad = max(0.01, 0.15 * (XGB_18F_f1_max - XGB_18F_f1_min + 1e-12))

# X-axis label only on bottom subplot (no repetition).
# NOTE: Configure x-axis semantics and formatting for accurate interpretation.
XGB_18F_fig.update_xaxes(
    title_text="",
    tickformat=".2f",
    showgrid=True,
    gridcolor=XGB_18F_grid,
    gridwidth=0.2,
    zeroline=False,
    row=1,
    col=1,
)
# NOTE: Configure x-axis semantics and formatting for accurate interpretation.
XGB_18F_fig.update_xaxes(
    tickformat=".2f",
    showgrid=True,
    gridcolor=XGB_18F_grid,
    gridwidth=0.2,
    zeroline=False,
    row=2,
    col=1,
)

# Remove y-axis title label text in both subplots.
# NOTE: Configure y-axis semantics and formatting for accurate interpretation.
XGB_18F_fig.update_yaxes(
    title_text="",
    range=[max(0.0, XGB_18F_f1_min - XGB_18F_f1_pad), min(1.02, XGB_18F_f1_max + XGB_18F_f1_pad)],
    tickformat=".3f",
    showgrid=True,
    gridcolor=XGB_18F_grid,
    gridwidth=0.2,
    zeroline=False,
    row=1,
    col=1,
)
# NOTE: Configure y-axis semantics and formatting for accurate interpretation.
XGB_18F_fig.update_yaxes(
    title_text="",
    range=[0, 1.02],
    tickformat=".2f",
    showgrid=True,
    gridcolor=XGB_18F_grid,
    gridwidth=0.2,
    zeroline=False,
    row=2,
    col=1,
)

# NOTE: Apply global styling (title, typography, spacing, colors) for readability consistency.
XGB_18F_fig.update_layout(
    title=dict(
        text=(
            "<span style='font-size:30px;font-weight:bold;'>    XGB_18F Outperforms the SGD Models but Still Struggles on Rare Classes</span>"
            "<br><span style='font-size:20px;font-weight: normal;'>      XGB_18F: Performance Metrics Across Different Cutoffs</span>"
        ),
        x=0,
        xanchor="left",
        pad=dict(t=50),
    ),
    paper_bgcolor=XGB_18F_bg,
    plot_bgcolor=XGB_18F_bg,
    font=dict(family=XGB_18F_plot_style["font_family"], size=XGB_18F_plot_style["font_size_base"], color=XGB_18F_text),
    legend=STANDARD_LEGEND,
    margin=dict(
        t=XGB_18F_plot_style["margin_top"] + 90,
        r=XGB_18F_plot_style["margin_right"],
        b=XGB_18F_plot_style["margin_bottom"],
        l=XGB_18F_plot_style["margin_left"],
    ),
    height=980,
    hovermode="x unified",
)

XGB_18F_fig.show()


# **Incremental PCA On Raw Signals (All Leads And Patients)**


<!-- CELL NOTE -->
> **Cell Note:** This section documents **Incremental PCA On Raw Signals (All Leads And Patients)** and provides context for the following code/results.
> Review it before executing downstream cells so assumptions and outputs remain aligned.


In [65]:
# CELL GUIDE:
# Purpose: Centralized resampling to 250 Hz for downstream Incremental PCA, XGB_100F, and CNN pipelines.
# Workflow:
# 1. Centralized resampling to 250 Hz for downstream Incremental PCA, XGB_100F, and CNN pipelines.
# 2. import math
# 3. import time
# 4. from fractions import Fraction
# 5. import numpy as np
# 6. from scipy.signal import resample_poly

# Centralized resampling to 250 Hz for downstream Incremental PCA, XGB_100F, and CNN pipelines.
import math
import time
from fractions import Fraction

import numpy as np
from scipy.signal import resample_poly

signals_all_raw = np.asarray(ecg_data["signals"], dtype=np.float32)
if signals_all_raw.ndim != 3:
    raise RuntimeError(f"Expected ecg_data['signals'] with shape (patients, leads, samples), got {signals_all_raw.shape}")

if "sampling_rate_hz" in ecg_data:
    source_sampling_rate_hz = float(ecg_data["sampling_rate_hz"])
elif "metadata" in ecg_data and "sampling_rate_hz" in ecg_data["metadata"].columns:
    source_sampling_rate_hz = float(np.nanmedian(ecg_data["metadata"]["sampling_rate_hz"].to_numpy(dtype=float)))
else:
    source_sampling_rate_hz = 500.0

target_sampling_rate_hz = 250.0
resample_ratio = Fraction(target_sampling_rate_hz / source_sampling_rate_hz).limit_denominator(2000)
resample_up = int(resample_ratio.numerator)
resample_down = int(resample_ratio.denominator)


def _format_eta_hms(seconds):
    seconds = max(0, int(seconds))
    h, rem = divmod(seconds, 3600)
    m, s = divmod(rem, 60)
    if h > 0:
        return f"{h}h {m}m {s}s"
    return f"{m}m {s}s"


resample_start = time.time()
patient_count = int(signals_all_raw.shape[0])
progress_step = max(1, patient_count // 20)

if math.isclose(source_sampling_rate_hz, target_sampling_rate_hz, rel_tol=0.0, abs_tol=1e-12):
    ecg_signal_250_hz = np.ascontiguousarray(signals_all_raw, dtype=np.float32)
    print(
        f"[Resample][ETA] completed {patient_count}/{patient_count} patients | "
        f"remaining 0 | ETA ~ 0m 0s"
    )
else:
    resampled_patients = []
    for patient_idx in range(patient_count):
        patient_signal = signals_all_raw[patient_idx]
        patient_resampled = resample_poly(
            patient_signal,
            up=resample_up,
            down=resample_down,
            axis=-1,
        ).astype(np.float32, copy=False)
        resampled_patients.append(np.ascontiguousarray(patient_resampled, dtype=np.float32))

        completed = patient_idx + 1
        if completed == 1 or completed % progress_step == 0 or completed == patient_count:
            elapsed = max(time.time() - resample_start, 1e-9)
            avg_per_patient = elapsed / completed
            remaining = patient_count - completed
            eta_seconds = avg_per_patient * remaining
            print(
                f"[Resample][ETA] completed {completed}/{patient_count} patients | "
                f"remaining {remaining} | ETA ~ {_format_eta_hms(eta_seconds)}"
            )

    ecg_signal_250_hz = np.ascontiguousarray(np.stack(resampled_patients, axis=0), dtype=np.float32)

resampling_seconds = time.time() - resample_start
ecg_data["signals_250_hz"] = ecg_signal_250_hz

print(f"Resampling: {source_sampling_rate_hz:.1f} Hz -> {target_sampling_rate_hz:.1f} Hz (up={resample_up}, down={resample_down})")
print(f"Resampled signal shape (patient, lead, sample): {ecg_signal_250_hz.shape}")
print(f"Resampling completed in {resampling_seconds:.2f}s")

[Resample][ETA] completed 1/45152 patients | remaining 45151 | ETA ~ 3h 19m 52s
[Resample][ETA] completed 2257/45152 patients | remaining 42895 | ETA ~ 1m 56s
[Resample][ETA] completed 4514/45152 patients | remaining 40638 | ETA ~ 1m 48s
[Resample][ETA] completed 6771/45152 patients | remaining 38381 | ETA ~ 1m 51s
[Resample][ETA] completed 9028/45152 patients | remaining 36124 | ETA ~ 1m 47s
[Resample][ETA] completed 11285/45152 patients | remaining 33867 | ETA ~ 1m 38s
[Resample][ETA] completed 13542/45152 patients | remaining 31610 | ETA ~ 1m 31s
[Resample][ETA] completed 15799/45152 patients | remaining 29353 | ETA ~ 1m 25s
[Resample][ETA] completed 18056/45152 patients | remaining 27096 | ETA ~ 1m 19s
[Resample][ETA] completed 20313/45152 patients | remaining 24839 | ETA ~ 1m 13s
[Resample][ETA] completed 22570/45152 patients | remaining 22582 | ETA ~ 1m 7s
[Resample][ETA] completed 24827/45152 patients | remaining 20325 | ETA ~ 1m 1s
[Resample][ETA] completed 27084/45152 patients

In [66]:
# CELL GUIDE:
# Purpose: Fit StandardScaler (batch-wise) and IncrementalPCA on all leads and all patients.
# Workflow:
# 1. Fit StandardScaler (batch-wise) and IncrementalPCA on all leads and all patients.
# 2. from sklearn.decomposition import IncrementalPCA
# 3. from sklearn.preprocessing import StandardScaler
# 4. batch_size = 128
# 5. n_components = 100
# 6. dtype = np.float32

# Fit StandardScaler (batch-wise) and IncrementalPCA on all leads and all patients.
from sklearn.decomposition import IncrementalPCA
from sklearn.preprocessing import StandardScaler

batch_size = 128
n_components = 100
dtype = np.float32

signals_all = np.asarray(ecg_signal_250_hz, dtype=dtype)
n_patients, n_leads, n_samples = signals_all.shape
n_rows = n_patients * n_leads

expected_rows, expected_samples = 541824, 2500
if (n_rows, n_samples) != (expected_rows, expected_samples):
    print(
        f"Warning: Incremental PCA input shape is ({n_rows}, {n_samples}), "
        f"expected ({expected_rows}, {expected_samples})."
    )

# Flatten to (patient * lead, sample_index) so every lead of every patient is included.
signal_rows = signals_all.reshape(n_rows, n_samples)

# Safety clip only if data dimensions are smaller than requested component count.
n_components_effective = min(n_components, n_rows, n_samples)
if n_components_effective != n_components:
    print(f"Adjusted n_components from {n_components} to {n_components_effective} due to data shape constraints.")

# NOTE: Standardize feature scales so optimization is stable and comparable.
inc_scaler = StandardScaler(copy=False)
for start in range(0, n_rows, batch_size):
    batch = signal_rows[start:start + batch_size].astype(dtype, copy=False)
    inc_scaler.partial_fit(batch)

# NOTE: Apply memory-efficient PCA for high-dimensional batched feature spaces.
inc_pca = IncrementalPCA(n_components=n_components_effective, batch_size=batch_size)
for start in range(0, n_rows, batch_size):
    batch = signal_rows[start:start + batch_size].astype(dtype, copy=False)
    batch_scaled = inc_scaler.transform(batch)
    inc_pca.partial_fit(batch_scaled)

ipca_explained_variance_ratio = np.asarray(inc_pca.explained_variance_ratio_, dtype=float)
ipca_cumulative_explained_variance = np.cumsum(ipca_explained_variance_ratio)
ipca_component_numbers = np.arange(1, ipca_explained_variance_ratio.size + 1)

# NOTE: Persist artifacts in ecg_data so later cells can reuse them without recomputation.
ecg_data.setdefault("incremental_pca", {})["scaler"] = inc_scaler
# NOTE: Persist artifacts in ecg_data so later cells can reuse them without recomputation.
ecg_data.setdefault("incremental_pca", {})["model"] = inc_pca
# NOTE: Persist artifacts in ecg_data so later cells can reuse them without recomputation.
ecg_data.setdefault("incremental_pca", {})["explained_variance_ratio"] = ipca_explained_variance_ratio
# NOTE: Persist artifacts in ecg_data so later cells can reuse them without recomputation.
ecg_data.setdefault("incremental_pca", {})["cumulative_explained_variance"] = ipca_cumulative_explained_variance
# NOTE: Persist artifacts in ecg_data so later cells can reuse them without recomputation.
ecg_data.setdefault("incremental_pca", {})["component_numbers"] = ipca_component_numbers
# NOTE: Persist artifacts in ecg_data so later cells can reuse them without recomputation.
ecg_data.setdefault("incremental_pca", {})["config"] = {
    "batch_size": batch_size,
    "n_components_requested": n_components,
    "n_components_effective": int(n_components_effective),
    "dtype": str(dtype),
    "rows": int(n_rows),
    "samples_per_row": int(n_samples),
}

print(f"Incremental PCA input shape (all leads x all patients): ({n_rows}, {n_samples})")
print("Expected shape target for 250 Hz pipeline: (541824, 5000)")
print(f"batch_size={batch_size}, n_components={n_components_effective}, dtype={dtype}")
print(f"Total explained variance (first {n_components_effective} components): {ipca_cumulative_explained_variance[-1]:.4f}")


Incremental PCA input shape (all leads x all patients): (541824, 2500)
Expected shape target for 250 Hz pipeline: (541824, 5000)
batch_size=128, n_components=100, dtype=<class 'numpy.float32'>
Total explained variance (first 100 components): 0.5199


In [68]:
# CELL GUIDE:
# Purpose: Visualize explained variance by component for IncrementalPCA.
# Workflow:
# 1. Visualize explained variance by component for IncrementalPCA.
# 2. import plotly.graph_objects as go
# 3. ipca_results = ecg_data.get("incremental_pca", {})
# 4. if not ipca_results
# 5. ipca_explained_variance_ratio = np.asarray(ipca_results["explained_variance_ratio"], dtype=float)
# 6. ipca_cumulative_explained_variance = np.asarray(ipca_results["cumulative_explained_variance"], dtype=float)

# Visualize explained variance by component for IncrementalPCA.
import plotly.graph_objects as go

ipca_results = ecg_data.get("incremental_pca", {})
if not ipca_results:
    raise RuntimeError("Incremental PCA results not found. Run the previous IncrementalPCA cell first.")

ipca_explained_variance_ratio = np.asarray(ipca_results["explained_variance_ratio"], dtype=float)
ipca_cumulative_explained_variance = np.asarray(ipca_results["cumulative_explained_variance"], dtype=float)
ipca_component_numbers = np.asarray(ipca_results["component_numbers"], dtype=int)
ipca_cfg = ipca_results.get("config", {})

if "graph_formatting" in globals():
    ipca_style = graph_formatting
else:
    ipca_style = {
        "font_family": "Helvetica Neue",
        "font_size_base": 16,
        "font_size_axis": 12,
        "color_text": "#b8b8b8",
        "color_title": "white",
        "color_grid": "#4a0000",
        "color_background": "#000000",
        "color_line": "#636EFA",
        "margin_top": 170,
        "margin_right": 80,
        "margin_bottom": 80,
        "margin_left": 80,
    }

text_gray = ipca_style["color_text"]
ipca_grid_dark = "#1f1f1f"
bg_black = ipca_style["color_background"]

ipca_fig = go.Figure()
ipca_fig.add_trace(
    go.Scatter(
        x=ipca_component_numbers,
        y=ipca_cumulative_explained_variance,
        mode="lines+markers",
        line=dict(width=2.0, color=ipca_style.get("color_line", "#636EFA")),
        marker=dict(size=6, color=ipca_style.get("color_line", "#636EFA")),
        hovertemplate="Component: %{x}<br>Cumulative Explained Variance: %{y:.2%}<extra></extra>",
        showlegend=False,
    )
)

ipca_fig.add_hline(y=0.90, line_dash="dot", line_color="#8a8a8a", line_width=1)
ipca_fig.add_hline(y=0.95, line_dash="dot", line_color="#f21322", line_width=1)

# NOTE: Configure x-axis semantics and formatting for accurate interpretation.
ipca_fig.update_xaxes(
    title_text="Number Of Principal Components",
    dtick=5,
    showgrid=True,
    gridcolor=ipca_grid_dark,
    gridwidth=0.2,
    zeroline=False,
    title_font=dict(family=ipca_style["font_family"], size=ipca_style["font_size_axis"], color=text_gray),
    tickfont=dict(family=ipca_style["font_family"], size=ipca_style["font_size_axis"], color=text_gray),
)
# NOTE: Configure y-axis semantics and formatting for accurate interpretation.
ipca_fig.update_yaxes(
    tickformat=".0%",
    range=[0, 1.02],
    showgrid=True,
    gridcolor=ipca_grid_dark,
    gridwidth=0.2,
    zeroline=False,
    title_font=dict(family=ipca_style["font_family"], size=ipca_style["font_size_axis"], color=text_gray),
    tickfont=dict(family=ipca_style["font_family"], size=ipca_style["font_size_axis"], color=text_gray),
)
# NOTE: Apply global styling (title, typography, spacing, colors) for readability consistency.
ipca_fig.update_layout(
    title=dict(
        text=(
            "<span style='font-size:30px;font-weight:bold;'>    At 100 Components Only 51.99% of Variance Is Explained</span>"
            f"<br><span style='font-size:20px;font-weight: normal;'>      Cumulative Explained Variance Across Resampled ECG Leads (250Hz)</span>"
        ),
        x=0,
        xanchor="left",
        pad=dict(t=50),
    ),
    paper_bgcolor=bg_black,
    plot_bgcolor=bg_black,
    font=dict(family=ipca_style["font_family"], size=ipca_style["font_size_base"], color=text_gray),
    margin=dict(
        t=ipca_style["margin_top"],
        r=ipca_style["margin_right"],
        b=ipca_style["margin_bottom"],
        l=ipca_style["margin_left"],
    ),
    height=760,
)

ipca_fig.show()


# **XGB_100F (Ray-Tuned On 100 Incremental PCA Components)**


<!-- CELL NOTE -->
> **Cell Note:** This section documents **XGB_100F (Ray-Tuned On 100 Incremental PCA Components)** and provides context for the following code/results.
> Review it before executing downstream cells so assumptions and outputs remain aligned.


In [69]:
# CELL GUIDE:
# Purpose: Prepare XGB_100F inputs from the 100 IncrementalPCA components (all leads -> patient-level aggregation), then Ray-tune OVR XGB.
# Workflow:
# 1. Prepare XGB_100F inputs from the 100 IncrementalPCA components (all leads -> patient-level aggregation), then Ray-tun...
# 2. import ray
# 3. import time
# 4. import random
# 5. from ray import tune
# 6. from ray.tune.schedulers import ASHAScheduler

# Prepare XGB_100F inputs from the 100 IncrementalPCA components (all leads -> patient-level aggregation), then Ray-tune OVR XGB.
import ray
import time
import random
from ray import tune
from ray.tune.schedulers import ASHAScheduler
from sklearn.metrics import f1_score, precision_score, recall_score, log_loss
from sklearn.model_selection import train_test_split
from sklearn.multiclass import OneVsRestClassifier
from xgboost import XGBClassifier

ipca_results = ecg_data.get("incremental_pca", {})
if not ipca_results or "model" not in ipca_results or "scaler" not in ipca_results:
    raise RuntimeError("Run the Incremental PCA fitting cell first.")
if "metadata" not in ecg_data or "dx_codes" not in ecg_data["metadata"].columns:
    raise RuntimeError("Missing general dx labels: expected ecg_data[\"metadata\"][\"dx_codes\"].")

XGB_100F_signals = np.asarray(ecg_signal_250_hz, dtype=np.float32)
XGB_100F_n_patients, XGB_100F_n_leads, XGB_100F_n_samples = XGB_100F_signals.shape
XGB_100F_signal_rows = XGB_100F_signals.reshape(XGB_100F_n_patients * XGB_100F_n_leads, XGB_100F_n_samples)

XGB_100F_batch_size = int((ipca_results.get("config") or {}).get("batch_size", 32))
XGB_100F_scaler = ipca_results["scaler"]
XGB_100F_pca = ipca_results["model"]
XGB_100F_components = int(XGB_100F_pca.n_components_)

# Batch-wise transform with fitted scaler + IncrementalPCA.
XGB_100F_row_components = np.empty((XGB_100F_signal_rows.shape[0], XGB_100F_components), dtype=np.float32)
for start in range(0, XGB_100F_signal_rows.shape[0], XGB_100F_batch_size):
    end = min(start + XGB_100F_batch_size, XGB_100F_signal_rows.shape[0])
    batch = XGB_100F_signal_rows[start:end].astype(np.float32, copy=False)
    batch_scaled = XGB_100F_scaler.transform(batch)
    XGB_100F_row_components[start:end] = XGB_100F_pca.transform(batch_scaled).astype(np.float32, copy=False)

# Aggregate lead-level embeddings to one 100D vector per patient.
XGB_100F_X = XGB_100F_row_components.reshape(XGB_100F_n_patients, XGB_100F_n_leads, XGB_100F_components).mean(axis=1)

XGB_100F_metadata_df = ecg_data["metadata"].copy()
if len(XGB_100F_metadata_df) != XGB_100F_X.shape[0]:
    raise RuntimeError(
        f"Mismatch between PCA-derived rows ({XGB_100F_X.shape[0]}) and metadata rows ({len(XGB_100F_metadata_df)})."
    )

XGB_100F_y_labels = XGB_100F_metadata_df["dx_codes"].fillna("").astype(str).apply(
    lambda raw: sorted({code.strip() for code in raw.split(",") if code.strip()})
)

from sklearn.preprocessing import MultiLabelBinarizer
# NOTE: Convert per-sample label sets into a multi-hot indicator matrix.
XGB_100F_mlb = MultiLabelBinarizer()
XGB_100F_y = XGB_100F_mlb.fit_transform(XGB_100F_y_labels)
XGB_100F_label_names = [str(lbl) for lbl in XGB_100F_mlb.classes_]

XGB_100F_X = np.asarray(XGB_100F_X, dtype=np.float32)
XGB_100F_y = np.asarray(XGB_100F_y, dtype=np.int32)

XGB_100F_X_train, XGB_100F_X_test, XGB_100F_y_train, XGB_100F_y_test = train_test_split(
    XGB_100F_X,
    XGB_100F_y,
    test_size=0.2,
    random_state=42,
    shuffle=True,
)

XGB_100F_X_tune_train, XGB_100F_X_tune_val, XGB_100F_y_tune_train, XGB_100F_y_tune_val = train_test_split(
    XGB_100F_X_train,
    XGB_100F_y_train,
    test_size=0.2,
    random_state=42,
    shuffle=True,
)

print(f"XGB_100F feature matrix shape (IPCA components): {XGB_100F_X.shape}")
print(f"XGB_100F feature count: {XGB_100F_X.shape[1]}")
print(f"XGB_100F multilabel target shape: {XGB_100F_y.shape}")
print(f"XGB_100F number of label columns: {XGB_100F_y.shape[1]} (expected ~93)")
print(f"XGB_100F train shape: {XGB_100F_X_train.shape}, test shape: {XGB_100F_X_test.shape}")
print(f"XGB_100F ray-tune train/val shapes: {XGB_100F_X_tune_train.shape}, {XGB_100F_X_tune_val.shape}")

# NOTE: Persist artifacts in ecg_data so later cells can reuse them without recomputation.
ecg_data.setdefault("xgb_100F", {})["feature_columns"] = [f"ipca_component_{i+1}" for i in range(XGB_100F_X.shape[1])]
# NOTE: Persist artifacts in ecg_data so later cells can reuse them without recomputation.
ecg_data.setdefault("xgb_100F", {})["label_columns"] = XGB_100F_label_names

XGB_100F_BACKEND = "xgboost"
XGB_100F_num_ray_trials = 30
XGB_100F_objective_metric = "f1_macro" # prioritize macro F1 for better performance on rare classes, but also report micro F1 and log loss
print(f"XGB_100F backend: {XGB_100F_BACKEND}")

XGB_100F_param_space = {
    "max_depth": tune.choice([3, 4, 5]),
    "min_child_weight": tune.choice([1, 2, 5]),
    "subsample": tune.uniform(0.6, 1.0),
    "colsample_bytree": tune.uniform(0.6, 1.0),
    "reg_lambda": tune.loguniform(1e-2, 1e2),
    "reg_alpha": tune.loguniform(1e-8, 1.0),
    "learning_rate": tune.loguniform(1e-2, 3e-1),
    "n_estimators": tune.choice([300, 600, 1000]),
}


def _xgb100f_base_estimator_from_config(config):
    return XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        random_state=42,
        n_jobs=1,
        nthread=1,
        early_stopping_rounds=50,
        max_depth=int(config["max_depth"]),
        min_child_weight=float(config["min_child_weight"]),
        subsample=float(config["subsample"]),
        colsample_bytree=float(config["colsample_bytree"]),
        reg_lambda=float(config["reg_lambda"]),
        reg_alpha=float(config["reg_alpha"]),
        learning_rate=float(config["learning_rate"]),
        n_estimators=int(config["n_estimators"]),
    )


def _xgb100f_fit_predict_ovr_with_eval(config, X_fit, y_fit, X_eval, y_eval):
    n_labels = y_fit.shape[1]
    y_pred = np.zeros((X_eval.shape[0], n_labels), dtype=np.int32)
    y_proba = np.zeros((X_eval.shape[0], n_labels), dtype=np.float32)

    for label_idx in range(n_labels):
        estimator = _xgb100f_base_estimator_from_config(config)
        estimator.fit(
            X_fit,
            y_fit[:, label_idx],
            eval_set=[(X_eval, y_eval[:, label_idx])],
            verbose=False,
        )
        label_proba = estimator.predict_proba(X_eval)
        if label_proba.ndim == 2 and label_proba.shape[1] > 1:
            label_proba = label_proba[:, 1]
        else:
            label_proba = label_proba.reshape(-1)

        y_proba[:, label_idx] = label_proba.astype(np.float32, copy=False)
        y_pred[:, label_idx] = (y_proba[:, label_idx] >= 0.5).astype(np.int32)

    return y_pred, y_proba


def _xgb100f_mean_label_logloss(y_true, y_proba):
    y_true = np.asarray(y_true, dtype=np.int32)
    y_proba = np.asarray(y_proba, dtype=np.float64)
    if y_proba.ndim == 1:
        y_proba = y_proba[:, None]

    losses = []
    for j in range(y_true.shape[1]):
        yj = y_true[:, j]
        pj = np.clip(y_proba[:, j], 1e-12, 1 - 1e-12)
        try:
            losses.append(log_loss(yj, pj, labels=[0, 1]))
        except ValueError:
            losses.append(float(np.mean(-(yj * np.log(pj) + (1 - yj) * np.log(1 - pj)))))
    return float(np.mean(losses)) if losses else np.nan


def XGB_100F_ray_objective(config, X_fit, y_fit, X_eval, y_eval):
    X_fit = np.asarray(X_fit, dtype=np.float32)
    X_eval = np.asarray(X_eval, dtype=np.float32)
    y_fit = np.asarray(y_fit, dtype=np.int32)
    y_eval = np.asarray(y_eval, dtype=np.int32)

    y_pred, y_proba = _xgb100f_fit_predict_ovr_with_eval(
        config=config,
        X_fit=X_fit,
        y_fit=y_fit,
        X_eval=X_eval,
        y_eval=y_eval,
    )

    metrics = {
        "precision_micro": precision_score(y_eval, y_pred, average="micro", zero_division=0),
        "precision_macro": precision_score(y_eval, y_pred, average="macro", zero_division=0),
        "recall_micro": recall_score(y_eval, y_pred, average="micro", zero_division=0),
        "recall_macro": recall_score(y_eval, y_pred, average="macro", zero_division=0),
        "f1_micro": f1_score(y_eval, y_pred, average="micro", zero_division=0),
        "f1_macro": f1_score(y_eval, y_pred, average="macro", zero_division=0),
        "val_logloss": _xgb100f_mean_label_logloss(y_eval, y_proba),
    }
    tune.report(metrics)


class XGB_100F_ETACallback(tune.Callback):
    def __init__(self, total_trials):
        self.total_trials = int(total_trials)
        self.completed_trials = 0
        self.start_time = time.time()

    def on_trial_complete(self, iteration, trials, trial, **info):
        self.completed_trials += 1
        elapsed = max(time.time() - self.start_time, 1e-6)
        avg_per_trial = elapsed / max(self.completed_trials, 1)
        remaining_trials = max(self.total_trials - self.completed_trials, 0)
        eta_seconds = int(avg_per_trial * remaining_trials)
        eta_min, eta_sec = divmod(eta_seconds, 60)
        print(
            f"[XGB_100F][ETA] completed {self.completed_trials}/{self.total_trials} trials | "
            f"remaining {remaining_trials} | ETA ~ {eta_min}m {eta_sec}s"
        )


np.random.seed(42)
random.seed(42)

try:
    from ray.tune.search.optuna import OptunaSearch
    XGB_100F_search_alg = OptunaSearch(seed=42)
    XGB_100F_search_backend = "optuna"
except Exception:
    from ray.tune.search.hyperopt import HyperOptSearch
    XGB_100F_search_alg = HyperOptSearch(random_state_seed=42)
    XGB_100F_search_backend = "hyperopt"

XGB_100F_scheduler = ASHAScheduler(
    max_t=1,
    grace_period=1,
    reduction_factor=2,
)
print(f"XGB_100F search backend: {XGB_100F_search_backend}")

if ray.is_initialized():
    ray.shutdown()
ray.init(ignore_reinit_error=True, include_dashboard=False, log_to_driver=False)

XGB_100F_eta_callback = XGB_100F_ETACallback(XGB_100F_num_ray_trials)

# NOTE: Launch Ray Tune hyperparameter search over the configured trial space.
XGB_100F_ray_analysis = tune.run(
    tune.with_parameters(
        XGB_100F_ray_objective,
        X_fit=XGB_100F_X_tune_train,
        y_fit=XGB_100F_y_tune_train,
        X_eval=XGB_100F_X_tune_val,
        y_eval=XGB_100F_y_tune_val,
    ),
    config=XGB_100F_param_space,
    num_samples=XGB_100F_num_ray_trials,
    metric=XGB_100F_objective_metric,
    mode="max",
    resources_per_trial={"cpu": 1},
    callbacks=[XGB_100F_eta_callback],
    search_alg=XGB_100F_search_alg,
    scheduler=XGB_100F_scheduler,
    verbose=2,
)

XGB_100F_ray_tune_results_df = XGB_100F_ray_analysis.dataframe().copy().rename(
    columns={
        "config/max_depth": "max_depth",
        "config/min_child_weight": "min_child_weight",
        "config/gamma": "gamma",
        "config/subsample": "subsample",
        "config/colsample_bytree": "colsample_bytree",
        "config/reg_lambda": "reg_lambda",
        "config/reg_alpha": "reg_alpha",
        "config/learning_rate": "learning_rate",
        "config/n_estimators": "n_estimators",
    }
)

for metric_name in [
    "precision_micro", "precision_macro", "recall_micro", "recall_macro",
    "f1_micro", "f1_macro", "val_logloss"
]:
    if metric_name not in XGB_100F_ray_tune_results_df.columns:
        for alt_col in (f"last_result/{metric_name}", f"last_result.{metric_name}"):
            if alt_col in XGB_100F_ray_tune_results_df.columns:
                XGB_100F_ray_tune_results_df[metric_name] = XGB_100F_ray_tune_results_df[alt_col]
                break

XGB_100F_keep_cols = [
    c for c in [
        "trial_id",
        "max_depth", "min_child_weight", "gamma", "subsample", "colsample_bytree",
        "reg_lambda", "reg_alpha", "learning_rate", "n_estimators",
        "precision_micro", "precision_macro", "recall_micro", "recall_macro",
        "f1_micro", "f1_macro", "val_logloss",
        "training_iteration", "time_total_s"
    ] if c in XGB_100F_ray_tune_results_df.columns
]
XGB_100F_ray_tune_results_df = XGB_100F_ray_tune_results_df[XGB_100F_keep_cols].copy()

XGB_100F_ray_tune_results_df = XGB_100F_ray_tune_results_df.sort_values(
    [c for c in ["f1_macro", "f1_micro", "val_logloss"] if c in XGB_100F_ray_tune_results_df.columns],
    ascending=[False, False, True][:len([c for c in ["f1_macro", "f1_micro", "val_logloss"] if c in XGB_100F_ray_tune_results_df.columns])],
    na_position="last",
).reset_index(drop=True)

if XGB_100F_ray_tune_results_df.empty:
    raise RuntimeError("XGB_100F Ray Tune returned no trial results.")

XGB_100F_selected_trial_row = XGB_100F_ray_tune_results_df.iloc[0]
XGB_100F_selected_trial_id = str(XGB_100F_selected_trial_row["trial_id"])
XGB_100F_selected_config = {
    "max_depth": int(XGB_100F_selected_trial_row["max_depth"]),
    "min_child_weight": float(XGB_100F_selected_trial_row["min_child_weight"]),
    "subsample": float(XGB_100F_selected_trial_row["subsample"]),
    "colsample_bytree": float(XGB_100F_selected_trial_row["colsample_bytree"]),
    "reg_lambda": float(XGB_100F_selected_trial_row["reg_lambda"]),
    "reg_alpha": float(XGB_100F_selected_trial_row["reg_alpha"]),
    "learning_rate": float(XGB_100F_selected_trial_row["learning_rate"]),
    "n_estimators": int(XGB_100F_selected_trial_row["n_estimators"]),
}

print("XGB_100F selected trial/config (OVR + binary logistic):")
print({"trial_id": XGB_100F_selected_trial_id, **XGB_100F_selected_config})

# NOTE: Persist artifacts in ecg_data so later cells can reuse them without recomputation.
ecg_data.setdefault("xgb_100F", {})["ray_results"] = XGB_100F_ray_tune_results_df.copy()
# NOTE: Persist artifacts in ecg_data so later cells can reuse them without recomputation.
ecg_data.setdefault("xgb_100F", {})["selected_trial_id"] = XGB_100F_selected_trial_id
# NOTE: Persist artifacts in ecg_data so later cells can reuse them without recomputation.
ecg_data.setdefault("xgb_100F", {})["selected_config"] = dict(XGB_100F_selected_config)
# NOTE: Persist artifacts in ecg_data so later cells can reuse them without recomputation.
ecg_data.setdefault("xgb_100F", {})["num_trials"] = XGB_100F_num_ray_trials

XGB_100F_ray_tune_results_df


XGB_100F feature matrix shape (IPCA components): (45152, 100)
XGB_100F feature count: 100
XGB_100F multilabel target shape: (45152, 94)
XGB_100F number of label columns: 94 (expected ~93)
XGB_100F train shape: (36121, 100), test shape: (9031, 100)
XGB_100F ray-tune train/val shapes: (28896, 100), (7225, 100)
XGB_100F backend: xgboost
XGB_100F search backend: optuna


python(92878) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(92879) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(92882) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(92883) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(92892) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(92893) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
2026-03-02 18:53:16,689	INFO worker.py:1951 -- Started a local Ray instance.
2026-03-02 18:53:20,631	INFO tune.py:616 -- [output] This uses the legacy output and progress reporter, as Jupyter notebooks are not supported by the new engine, yet. For more information, please see https://github.com/ray-project/ray/issues/36949
[I 2026-03-02 18:53:20,848] A new study created in memory with name: optuna
2026-03-02 

Trial name,f1_macro,f1_micro,precision_macro,precision_micro,recall_macro,recall_micro,val_logloss
XGB_100F_ray_objective_017b94c4,0.0377665,0.466781,0.0757402,0.85717,0.0318561,0.320714,0.053641
XGB_100F_ray_objective_02b7df97,0.0375667,0.448176,0.0643095,0.818148,0.0313737,0.308617,0.0548996
XGB_100F_ray_objective_059f700c,0.0356207,0.438036,0.0890609,0.854283,0.0292243,0.294528,0.0544023
XGB_100F_ray_objective_146bad15,0.0383939,0.451951,0.0785765,0.81113,0.0323312,0.313243,0.0548935
XGB_100F_ray_objective_34ccabeb,0.0397993,0.464739,0.0890927,0.835303,0.0335726,0.321924,0.0538394
XGB_100F_ray_objective_3ae761c6,0.0394134,0.45977,0.0948352,0.833957,0.0329724,0.31737,0.0540699
XGB_100F_ray_objective_487f2481,0.0245262,0.341544,0.0467001,0.859397,0.018868,0.213122,0.0580348
XGB_100F_ray_objective_4c1e3cda,0.0381106,0.465152,0.0608151,0.842351,0.0322865,0.321284,0.0537581
XGB_100F_ray_objective_4fff081a,0.0359898,0.449037,0.0678174,0.848191,0.0301765,0.305344,0.0541401
XGB_100F_ray_objective_5d54ceca,0.0380097,0.467487,0.0712913,0.836791,0.0324173,0.324344,0.0537403


[XGB_100F][ETA] completed 1/30 trials | remaining 29 | ETA ~ 192m 49s
[XGB_100F][ETA] completed 2/30 trials | remaining 28 | ETA ~ 127m 22s
[XGB_100F][ETA] completed 3/30 trials | remaining 27 | ETA ~ 90m 14s
[XGB_100F][ETA] completed 4/30 trials | remaining 26 | ETA ~ 65m 12s
[XGB_100F][ETA] completed 5/30 trials | remaining 25 | ETA ~ 52m 24s
[XGB_100F][ETA] completed 6/30 trials | remaining 24 | ETA ~ 58m 58s
[XGB_100F][ETA] completed 7/30 trials | remaining 23 | ETA ~ 51m 8s
[XGB_100F][ETA] completed 8/30 trials | remaining 22 | ETA ~ 44m 54s
[XGB_100F][ETA] completed 9/30 trials | remaining 21 | ETA ~ 41m 8s
[XGB_100F][ETA] completed 10/30 trials | remaining 20 | ETA ~ 38m 17s
[XGB_100F][ETA] completed 11/30 trials | remaining 19 | ETA ~ 35m 16s
[XGB_100F][ETA] completed 12/30 trials | remaining 18 | ETA ~ 33m 1s
[XGB_100F][ETA] completed 13/30 trials | remaining 17 | ETA ~ 31m 53s
[XGB_100F][ETA] completed 14/30 trials | remaining 16 | ETA ~ 27m 56s
[XGB_100F][ETA] completed 15/3

2026-03-02 19:32:10,773	INFO tune.py:1009 -- Wrote the latest version of all result files and experiment state to '/Users/felixaugustin/ray_results/XGB_100F_ray_objective_2026-03-02_18-53-20' in 0.0197s.


[XGB_100F][ETA] completed 30/30 trials | remaining 0 | ETA ~ 0m 0s


2026-03-02 19:32:10,792	INFO tune.py:1041 -- Total run time: 2330.16 seconds (2329.88 seconds for the tuning loop).


XGB_100F selected trial/config (OVR + binary logistic):
{'trial_id': 'd80b0ca2', 'max_depth': 3, 'min_child_weight': 5.0, 'subsample': 0.7660611256618486, 'colsample_bytree': 0.7730982444721965, 'reg_lambda': 0.053108374071179854, 'reg_alpha': 2.734720046539017e-07, 'learning_rate': 0.14007651482718234, 'n_estimators': 1000}


,trial_id,max_depth,min_child_weight,subsample,colsample_bytree,reg_lambda,reg_alpha,learning_rate,n_estimators,precision_micro,precision_macro,recall_micro,recall_macro,f1_micro,f1_macro,val_logloss,training_iteration,time_total_s
0,d80b0ca2,3,5,0.766061,0.773098,0.053108,2.734720e-07,0.140077,1000,0.839322,0.085792,0.320786,0.033660,0.464168,0.040083,0.053980,1,340.754526
1,e3bcaf59,3,1,0.843826,0.801072,0.016066,1.695008e-06,0.219593,1000,0.815955,0.076495,0.322422,0.033969,0.462205,0.040048,0.054808,1,366.923154
2,34ccabeb,3,5,0.712304,0.825708,0.014627,2.231949e-08,0.098011,1000,0.835303,0.089093,0.321924,0.033573,0.464739,0.039799,0.053839,1,468.392718
3,3ae761c6,3,5,0.769095,0.779701,0.036560,6.064709e-07,0.145414,1000,0.833957,0.094835,0.317370,0.032972,0.459770,0.039413,0.054070,1,382.463258
4,ac947f91,3,5,0.766069,0.763939,0.034684,4.485912e-07,0.123783,1000,0.836072,0.104294,0.322280,0.033171,0.465229,0.039265,0.053944,1,310.181768
5,b54dfe3e,3,5,0.878639,0.754652,0.040258,2.455297e-07,0.127004,1000,0.835843,0.089829,0.319932,0.032553,0.462742,0.038666,0.053923,1,290.896852
6,8955f0af,5,5,0.986253,0.923359,0.165369,6.044730e-08,0.102493,1000,0.856339,0.078131,0.324913,0.032583,0.471086,0.038405,0.053450,1,526.354426
7,146bad15,3,2,0.811860,0.696741,0.023573,1.505657e-01,0.213809,300,0.811130,0.078577,0.313243,0.032331,0.451951,0.038394,0.054893,1,327.328622
8,ca25e71e,5,5,0.744547,0.977851,0.018803,1.459516e-08,0.094356,1000,0.848947,0.092673,0.321141,0.032549,0.466002,0.038363,0.053549,1,507.988881
9,4c1e3cda,5,5,0.888588,0.778489,0.040955,1.651020e-08,0.127158,1000,0.842351,0.060815,0.321284,0.032287,0.465152,0.038111,0.053758,1,417.972760


In [84]:
# CELL GUIDE:
# Purpose: XGB_100F target metric over time (chronological).
# Workflow:
# 1. XGB_100F target metric over time (chronological)
# 2. import numpy as np
# 3. import pandas as pd
# 4. import plotly.graph_objects as go
# 5. XGB_100F_results_source = None
# 6. if "XGB_100F_ray_tune_results_df" in globals()

# XGB_100F target metric over time (chronological)
import numpy as np
import pandas as pd
import plotly.graph_objects as go

XGB_100F_results_source = None
if "XGB_100F_ray_tune_results_df" in globals():
    XGB_100F_results_source = XGB_100F_ray_tune_results_df
elif isinstance(ecg_data, dict):
    XGB_100F_results_source = (ecg_data.get("xgb_100F") or {}).get("ray_results")

if XGB_100F_results_source is None:
    raise RuntimeError("XGB_100F Ray results were not found. Run the XGB_100F tuning cell first.")

XGB_100F_time_df = XGB_100F_results_source.copy()
if XGB_100F_time_df.empty:
    raise RuntimeError("XGB_100F Ray results are empty.")

XGB_100F_target_metric = "f1_macro"
if XGB_100F_target_metric not in XGB_100F_time_df.columns:
    if "f1_micro" in XGB_100F_time_df.columns:
        XGB_100F_target_metric = "f1_micro"
    else:
        raise KeyError("Neither f1_macro nor f1_micro is available in XGB_100F Ray results.")

XGB_100F_time_sort_col = None
if "timestamp" in XGB_100F_time_df.columns:
    XGB_100F_time_df["_chrono"] = pd.to_numeric(XGB_100F_time_df["timestamp"], errors="coerce")
    XGB_100F_time_sort_col = "_chrono"
elif "date" in XGB_100F_time_df.columns:
    XGB_100F_time_df["_chrono"] = pd.to_datetime(XGB_100F_time_df["date"], errors="coerce")
    XGB_100F_time_sort_col = "_chrono"
elif "time_total_s" in XGB_100F_time_df.columns:
    XGB_100F_time_df["_chrono"] = pd.to_numeric(XGB_100F_time_df["time_total_s"], errors="coerce")
    XGB_100F_time_sort_col = "_chrono"
elif "training_iteration" in XGB_100F_time_df.columns:
    XGB_100F_time_df["_chrono"] = pd.to_numeric(XGB_100F_time_df["training_iteration"], errors="coerce")
    XGB_100F_time_sort_col = "_chrono"

if XGB_100F_time_sort_col is not None:
    XGB_100F_time_df = XGB_100F_time_df.sort_values(XGB_100F_time_sort_col, ascending=True, na_position="last")

XGB_100F_time_df = XGB_100F_time_df.reset_index(drop=True)
XGB_100F_time_df["trial_order"] = np.arange(1, len(XGB_100F_time_df) + 1)
XGB_100F_time_df["trial_id"] = XGB_100F_time_df.get("trial_id", "").astype(str)

XGB_100F_plot_style = graph_formatting if "graph_formatting" in globals() else {
    "font_family": "Helvetica Neue", "font_size_base": 16, "font_size_axis": 12,
    "color_text": "#b8b8b8", "color_background": "#000000",
    "margin_top": 170, "margin_right": 80, "margin_bottom": 80, "margin_left": 80,
}
XGB_100F_text = XGB_100F_plot_style.get("color_text", "#b8b8b8")
XGB_100F_bg = XGB_100F_plot_style.get("color_background", "#000000")
XGB_100F_color = SERIES_COLORS.get("f1_macro", "#636EFA") if "SERIES_COLORS" in globals() else "#636EFA"

XGB_100F_time_fig = go.Figure(data=[
    go.Scatter(
        x=XGB_100F_time_df["trial_order"],
        y=XGB_100F_time_df[XGB_100F_target_metric],
        mode="lines+markers",
        line=dict(width=2.2, color=XGB_100F_color),
        marker=dict(size=6, color=XGB_100F_color),
        customdata=np.column_stack([XGB_100F_time_df["trial_id"]]),
        hovertemplate=(
            "Chronological Order: %{x}<br>"
            "Trial ID: %{customdata[0]}<br>"
            + f"{XGB_100F_target_metric.upper()}: %{{y:.4f}}<extra></extra>"
        ),
        name="XGB_100F",
    )
])

# NOTE: Configure x-axis semantics and formatting for accurate interpretation.
XGB_100F_time_fig.update_xaxes(title_text="Chronological Trial Order", title_font=dict(family=XGB_100F_plot_style["font_family"], size=XGB_100F_plot_style["font_size_axis"], color=XGB_100F_text), tickfont=dict(family=XGB_100F_plot_style["font_family"], size=XGB_100F_plot_style["font_size_axis"], color=XGB_100F_text), showgrid=True, gridcolor="#1f1f1f", gridwidth=0.2)
# NOTE: Configure y-axis semantics and formatting for accurate interpretation.
XGB_100F_time_fig.update_yaxes(title_font=dict(family=XGB_100F_plot_style["font_family"], size=XGB_100F_plot_style["font_size_axis"], color=XGB_100F_text), tickfont=dict(family=XGB_100F_plot_style["font_family"], size=XGB_100F_plot_style["font_size_axis"], color=XGB_100F_text), showgrid=True, gridcolor="#1f1f1f", gridwidth=0.2)

# NOTE: Apply global styling (title, typography, spacing, colors) for readability consistency.
XGB_100F_time_fig.update_layout(
    title=dict(text=("<span style='font-size:30px;font-weight:bold;'>   Early Plateau with No Sustained Macro-F1 Improvement Across Trials</span>" "<br><span style='font-size:20px;font-weight: normal;'>     XGB_100F Target Metric - F1 Macro During Each Ray Tune Trial</span>"), x=0, xanchor="left", pad=dict(t=50)),
    paper_bgcolor=XGB_100F_bg,
    plot_bgcolor=XGB_100F_bg,
    font=dict(family=XGB_100F_plot_style["font_family"], size=XGB_100F_plot_style["font_size_base"], color=XGB_100F_text),
    hoverlabel=dict(font=dict(family=XGB_100F_plot_style["font_family"], size=XGB_100F_plot_style["font_size_axis"], color=XGB_100F_text)),
    legend=STANDARD_LEGEND if "STANDARD_LEGEND" in globals() else dict(orientation="h", yanchor="top", y=1.02, xanchor="left", x=0),
    margin=dict(t=XGB_100F_plot_style["margin_top"], r=XGB_100F_plot_style["margin_right"], b=XGB_100F_plot_style["margin_bottom"], l=XGB_100F_plot_style["margin_left"]),
    height=700,
)

XGB_100F_time_fig.show()


In [70]:
# CELL GUIDE:
# Purpose: Cross-validate probability cutoffs for the selected Ray-tuned XGB_100F configuration.
# Workflow:
# 1. Cross-validate probability cutoffs for the selected Ray-tuned XGB_100F configuration.
# 2. from sklearn.metrics import f1_score, precision_score, recall_score
# 3. from sklearn.model_selection import cross_val_predict
# 4. from sklearn.multiclass import OneVsRestClassifier
# 5. from xgboost import XGBClassifier
# 6. XGB_100F_threshold_cv_folds = 5

# Cross-validate probability cutoffs for the selected Ray-tuned XGB_100F configuration.
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.model_selection import cross_val_predict
from sklearn.multiclass import OneVsRestClassifier
from xgboost import XGBClassifier

XGB_100F_threshold_cv_folds = 5
# NOTE: Define a deterministic threshold grid for cutoff-sensitivity analysis.
XGB_100F_probability_cutoffs = np.round(np.linspace(0.05, 0.95, 19), 2)

XGB_100F_selected_config = {'max_depth': 3, 
                            'min_child_weight': 5.0, 
                            'subsample': 0.7660611256618486, 
                            'colsample_bytree': 0.7730982444721965, 
                            'reg_lambda': 0.053108374071179854, 
                            'reg_alpha': 2.734720046539017e-07, 
                            'learning_rate': 0.14007651482718234, 
                            'n_estimators': 1000}

XGB_100F_selected_trial_id = 'd80b0ca2'

# cross_val_predict cannot provide a fold-specific eval_set to XGBoost, so early stopping is disabled for OOF CV.
# NOTE: Wrap binary base learner for multi-label one-vs-rest training.
XGB_100F_cv_model = OneVsRestClassifier(
    XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        random_state=42,
        n_jobs=1,
        nthread=1,
        **XGB_100F_selected_config,
    )
)

XGB_100F_X_train_cv = np.asarray(XGB_100F_X_train, dtype=np.float32)
XGB_100F_y_train_cv = np.asarray(XGB_100F_y_train, dtype=np.int32)

# NOTE: Generate out-of-fold predictions to estimate generalization without test leakage.
XGB_100F_oof_proba = cross_val_predict(
    XGB_100F_cv_model,
    XGB_100F_X_train_cv,
    XGB_100F_y_train_cv,
    cv=XGB_100F_threshold_cv_folds,
    method="predict_proba",
    n_jobs=-1,
)

XGB_100F_oof_proba = np.asarray(XGB_100F_oof_proba)
if XGB_100F_oof_proba.ndim == 1:
    XGB_100F_oof_proba = XGB_100F_oof_proba[:, None]

# Also score cutoffs on the dedicated tuning-validation split.
XGB_100F_X_val_set = np.asarray(XGB_100F_X_tune_val, dtype=np.float32)
XGB_100F_y_val_set = np.asarray(XGB_100F_y_tune_val, dtype=np.int32)

if "_xgb100f_fit_predict_ovr_with_eval" in globals():
    _, XGB_100F_val_proba = _xgb100f_fit_predict_ovr_with_eval(
        config=XGB_100F_selected_config,
        X_fit=np.asarray(XGB_100F_X_tune_train, dtype=np.float32),
        y_fit=np.asarray(XGB_100F_y_tune_train, dtype=np.int32),
        X_eval=XGB_100F_X_val_set,
        y_eval=XGB_100F_y_val_set,
    )
else:
    # Fallback path if helper is unavailable in current kernel state.
    XGB_100F_cv_model.fit(np.asarray(XGB_100F_X_tune_train, dtype=np.float32), np.asarray(XGB_100F_y_tune_train, dtype=np.int32))
    XGB_100F_val_proba = XGB_100F_cv_model.predict_proba(XGB_100F_X_val_set)

XGB_100F_val_proba = np.asarray(XGB_100F_val_proba)
if XGB_100F_val_proba.ndim == 1:
    XGB_100F_val_proba = XGB_100F_val_proba[:, None]

XGB_100F_threshold_metric_rows = []
XGB_100F_validation_threshold_metric_rows = []
for cutoff in XGB_100F_probability_cutoffs:
    y_pred_cutoff = (XGB_100F_oof_proba >= cutoff).astype(int)
    XGB_100F_threshold_metric_rows.append({
        "cutoff": float(cutoff),
        "f1_micro": f1_score(XGB_100F_y_train_cv, y_pred_cutoff, average="micro", zero_division=0),
        "f1_macro": f1_score(XGB_100F_y_train_cv, y_pred_cutoff, average="macro", zero_division=0),
        "precision_micro": precision_score(XGB_100F_y_train_cv, y_pred_cutoff, average="micro", zero_division=0),
        "recall_micro": recall_score(XGB_100F_y_train_cv, y_pred_cutoff, average="micro", zero_division=0),
        "precision_macro": precision_score(XGB_100F_y_train_cv, y_pred_cutoff, average="macro", zero_division=0),
        "recall_macro": recall_score(XGB_100F_y_train_cv, y_pred_cutoff, average="macro", zero_division=0),
    })

    y_pred_val_cutoff = (XGB_100F_val_proba >= cutoff).astype(int)
    XGB_100F_validation_threshold_metric_rows.append({
        "cutoff": float(cutoff),
        "f1_micro": f1_score(XGB_100F_y_val_set, y_pred_val_cutoff, average="micro", zero_division=0),
        "f1_macro": f1_score(XGB_100F_y_val_set, y_pred_val_cutoff, average="macro", zero_division=0),
        "precision_micro": precision_score(XGB_100F_y_val_set, y_pred_val_cutoff, average="micro", zero_division=0),
        "recall_micro": recall_score(XGB_100F_y_val_set, y_pred_val_cutoff, average="micro", zero_division=0),
        "precision_macro": precision_score(XGB_100F_y_val_set, y_pred_val_cutoff, average="macro", zero_division=0),
        "recall_macro": recall_score(XGB_100F_y_val_set, y_pred_val_cutoff, average="macro", zero_division=0),
    })


python(95425) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(95426) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(95427) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(95430) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(95431) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(95432) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(95433) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(95435) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(95436) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(95437) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
/opt/anaconda3/lib/p

In [71]:
# CELL GUIDE:
# Purpose: Run `XGB_100F_threshold_cv_df = pd.DataFrame(XGB_100F_threshold_metric_rows).sort_values("cutof` and continue downstream processing.
# Workflow:
# 1. XGB_100F_threshold_cv_df = pd.DataFrame(XGB_100F_threshold_metric_rows).sort_values("cutoff").reset_index(drop=True)
# 2. XGB_100F_threshold_val_df = pd.DataFrame(XGB_100F_validation_threshold_metric_rows).sort_values("cutoff").reset_index...
# 3. XGB_100F_best_cutoff_f1_micro = XGB_100F_threshold_cv_df.loc[XGB_100F_threshold_cv_df["f1_micro"].idxmax()]
# 4. XGB_100F_best_cutoff_f1_macro = XGB_100F_threshold_cv_df.loc[XGB_100F_threshold_cv_df["f1_macro"].idxmax()]
# 5. XGB_100F_optimal_cutoff = float(XGB_100F_best_cutoff_f1_micro["cutoff"])
# 6. ecg_data.setdefault("xgb_100F", {})["threshold_cv"] = {

XGB_100F_threshold_cv_df = pd.DataFrame(XGB_100F_threshold_metric_rows).sort_values("cutoff").reset_index(drop=True)
XGB_100F_threshold_val_df = pd.DataFrame(XGB_100F_validation_threshold_metric_rows).sort_values("cutoff").reset_index(drop=True)
XGB_100F_best_cutoff_f1_micro = XGB_100F_threshold_cv_df.loc[XGB_100F_threshold_cv_df["f1_micro"].idxmax()]
XGB_100F_best_cutoff_f1_macro = XGB_100F_threshold_cv_df.loc[XGB_100F_threshold_cv_df["f1_macro"].idxmax()]
XGB_100F_optimal_cutoff = float(XGB_100F_best_cutoff_f1_micro["cutoff"])

# NOTE: Persist artifacts in ecg_data so later cells can reuse them without recomputation.
ecg_data.setdefault("xgb_100F", {})["threshold_cv"] = {
    "trial_id": XGB_100F_selected_trial_id,
    "config": dict(XGB_100F_selected_config),
    "cv_folds": XGB_100F_threshold_cv_folds,
    "optimal_cutoff": XGB_100F_optimal_cutoff,
    "metrics_by_cutoff": XGB_100F_threshold_cv_df.copy(),
    "validation_metrics_by_cutoff": XGB_100F_threshold_val_df.copy(),
}

XGB_100F_threshold_cv_df[["cutoff", "f1_micro", "f1_macro", "precision_micro", "recall_micro", "precision_macro", "recall_macro"]].round(4)

,cutoff,f1_micro,f1_macro,precision_micro,recall_micro,precision_macro,recall_macro
0,0.05,0.3853,0.0746,0.2720,0.6608,0.0659,0.1158
1,0.10,0.4509,0.0719,0.3684,0.5812,0.0817,0.0886
2,0.15,0.4842,0.0682,0.4481,0.5266,0.0933,0.0738
3,0.20,0.4992,0.0639,0.5146,0.4847,0.1026,0.0638
4,0.25,0.5051,0.0607,0.5748,0.4504,0.1089,0.0569
5,0.30,0.5051,0.0574,0.6286,0.4222,0.1121,0.0515
6,0.35,0.5020,0.0548,0.6782,0.3985,0.1195,0.0473
7,0.40,0.4946,0.0519,0.7217,0.3762,0.1232,0.0436
8,0.45,0.4852,0.0492,0.7596,0.3564,0.1340,0.0405
9,0.50,0.4745,0.0465,0.7923,0.3387,0.1380,0.0377


In [74]:
# CELL GUIDE:
# Purpose: XGB_100F cutoff visualizations merged into subplots (shared x-axis).
# Workflow:
# 1. XGB_100F cutoff visualizations merged into subplots (shared x-axis).
# 2. import plotly.graph_objects as go
# 3. from plotly.subplots import make_subplots
# 4. XGB_100F_plot_df = XGB_100F_threshold_cv_df.copy()
# 5. XGB_100F_plot_style = graph_formatting if "graph_formatting" in globals() else {
# 6. }

# XGB_100F cutoff visualizations merged into subplots (shared x-axis).
import plotly.graph_objects as go
from plotly.subplots import make_subplots

XGB_100F_plot_df = XGB_100F_threshold_cv_df.copy()
XGB_100F_plot_style = graph_formatting if "graph_formatting" in globals() else {
    "font_family": "Helvetica Neue",
    "font_size_base": 16,
    "font_size_axis": 12,
    "color_text": "#b8b8b8",
    "color_background": "#000000",
    "margin_top": 170,
    "margin_right": 80,
    "margin_bottom": 80,
    "margin_left": 80,
}
XGB_100F_text = XGB_100F_plot_style.get("color_text", "#b8b8b8")
XGB_100F_bg = XGB_100F_plot_style.get("color_background", "#000000")
XGB_100F_grid = "#1f1f1f"

XGB_100F_fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.14,
    subplot_titles=("F1 Micro and F1 Macro", "Recall And Precision"),
)
for ann in XGB_100F_fig.layout.annotations:
    ann.x = 0
    ann.xanchor = "left"
    ann.align = "left"

# Row 1: F1 metrics
for metric_name, color in [
    ("f1_micro", SERIES_COLORS["f1_micro"]),
    ("f1_macro", SERIES_COLORS["f1_macro"]),
]:
    pretty_name = metric_name.replace("_", " ").title()
    XGB_100F_fig.add_trace(
        go.Scatter(
            x=XGB_100F_plot_df["cutoff"],
            y=XGB_100F_plot_df[metric_name],
            mode="lines+markers",
            name=pretty_name,
            line=dict(width=2.2, color=color),
            marker=dict(size=6, color=color),
            hovertemplate=(
                "Cutoff: %{x:.2f}<br>" + pretty_name + ": %{y:.4f}<extra></extra>"
            ),
        ),
        row=1,
        col=1,
    )

# Row 2: recall + precision metrics
for metric_name, color in [
    ("recall_micro", SERIES_COLORS["recall_micro"]),
    ("recall_macro", SERIES_COLORS["recall_macro"]),
    ("precision_micro", SERIES_COLORS["precision_micro"]),
    ("precision_macro", SERIES_COLORS["precision_macro"]),
]:
    pretty_name = metric_name.replace("_", " ").title()
    XGB_100F_fig.add_trace(
        go.Scatter(
            x=XGB_100F_plot_df["cutoff"],
            y=XGB_100F_plot_df[metric_name],
            mode="lines+markers",
            name=pretty_name,
            line=dict(width=2.0, color=color),
            marker=dict(size=6, color=color),
            hovertemplate=(
                "Cutoff: %{x:.2f}<br>" + pretty_name + ": %{y:.4f}<extra></extra>"
            ),
        ),
        row=2,
        col=1,
    )

XGB_100F_optimal_cutoff = float(XGB_100F_optimal_cutoff)
for row in (1, 2):
    XGB_100F_fig.add_vline(
        x=XGB_100F_optimal_cutoff,
        line_width=1.6,
        line_dash="dash",
        line_color="#D62728",
        row=row,
        col=1,
    )

XGB_100F_f1_min = float(XGB_100F_plot_df[["f1_micro", "f1_macro"]].min().min())
XGB_100F_f1_max = float(XGB_100F_plot_df[["f1_micro", "f1_macro"]].max().max())
XGB_100F_f1_pad = max(0.01, 0.15 * (XGB_100F_f1_max - XGB_100F_f1_min + 1e-12))

# NOTE: Configure x-axis semantics and formatting for accurate interpretation.
XGB_100F_fig.update_xaxes(
    showticklabels=False,
    showgrid=True,
    gridcolor=XGB_100F_grid,
    gridwidth=0.2,
    zeroline=False,
    tickfont=dict(family=XGB_100F_plot_style["font_family"], size=XGB_100F_plot_style["font_size_axis"], color=XGB_100F_text),
    row=1,
    col=1,
)
# NOTE: Configure x-axis semantics and formatting for accurate interpretation.
XGB_100F_fig.update_xaxes(
    title_text="Probability Cutoff",
    showgrid=True,
    gridcolor=XGB_100F_grid,
    gridwidth=0.2,
    zeroline=False,
    tickfont=dict(family=XGB_100F_plot_style["font_family"], size=XGB_100F_plot_style["font_size_axis"], color=XGB_100F_text),
    title_font=dict(family=XGB_100F_plot_style["font_family"], size=XGB_100F_plot_style["font_size_axis"], color=XGB_100F_text),
    row=2,
    col=1,
)

# NOTE: Configure y-axis semantics and formatting for accurate interpretation.
XGB_100F_fig.update_yaxes(
    range=[max(0.0, XGB_100F_f1_min - XGB_100F_f1_pad), min(1.02, XGB_100F_f1_max + XGB_100F_f1_pad)],
    showgrid=True,
    gridcolor=XGB_100F_grid,
    gridwidth=0.2,
    zeroline=False,
    tickfont=dict(family=XGB_100F_plot_style["font_family"], size=XGB_100F_plot_style["font_size_axis"], color=XGB_100F_text),
    row=1,
    col=1,
)

# NOTE: Configure y-axis semantics and formatting for accurate interpretation.
XGB_100F_fig.update_yaxes(
    range=[0, 1.02],
    showgrid=True,
    gridcolor=XGB_100F_grid,
    gridwidth=0.2,
    zeroline=False,
    tickfont=dict(family=XGB_100F_plot_style["font_family"], size=XGB_100F_plot_style["font_size_axis"], color=XGB_100F_text),
    row=2,
    col=1,
)

# NOTE: Apply global styling (title, typography, spacing, colors) for readability consistency.
XGB_100F_fig.update_layout(
    title=dict(
        text=(
            "<span style='font-size:30px;font-weight:bold;'>    Lower Threshold Enhances Recall Without Sacrificing Overall Performance</span>"
            "<br><span style='font-size:20px;font-weight: normal;'>      XGB_100F: Performance Metrics Across Different Cutoffs</span>"
        ),
        x=0,
        xanchor="left",
        pad=dict(t=50),
    ),
    hovermode="x unified",
    paper_bgcolor=XGB_100F_bg,
    plot_bgcolor=XGB_100F_bg,
    font=dict(family=XGB_100F_plot_style["font_family"], size=XGB_100F_plot_style["font_size_base"], color=XGB_100F_text),
    margin=dict(
        t=XGB_100F_plot_style["margin_top"] + 90,
        r=XGB_100F_plot_style["margin_right"],
        b=XGB_100F_plot_style["margin_bottom"],
        l=XGB_100F_plot_style["margin_left"],
    ),
    height=980,
    legend=STANDARD_LEGEND,
)

XGB_100F_fig.show()


In [92]:
# CELL GUIDE:
# Purpose: 1) Use centralized 250 Hz ECG signals for CNN (resampling is handled upstream).
# Workflow:
# 1. 1) Use centralized 250 Hz ECG signals for CNN (resampling is handled upstream).
# 2. if "ecg_signal_250_hz" not in globals()
# 3. CNN_250Hz_resampled = np.asarray(ecg_signal_250_hz, dtype=np.float32)
# 4. print(f"CNN input signal shape (patient, lead, sample): {CNN_250Hz_resampled.shape}")

# 1) Use centralized 250 Hz ECG signals for CNN (resampling is handled upstream).
if "ecg_signal_250_hz" not in globals():
    raise RuntimeError("Run the centralized resampling cell under Incremental PCA first.")

CNN_250Hz_resampled = np.asarray(ecg_signal_250_hz, dtype=np.float32)
print(f"CNN input signal shape (patient, lead, sample): {CNN_250Hz_resampled.shape}")

CNN input signal shape (patient, lead, sample): (45152, 12, 2500)


In [ ]:
# CELL GUIDE:
# Purpose: Execute this cell as part of the notebook pipeline.
# Workflow:
# 1. Execute cell statements in order.



In [93]:
# CELL GUIDE:
# Purpose: Run `import copy` and continue downstream processing.
# Workflow:
# 1. import copy
# 2. import time
# 3. import numpy as np
# 4. import torch
# 5. import torch.nn as nn
# 6. import torch.optim as optim


import copy
import time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from torch.utils.data import DataLoader, TensorDataset


def _cnn_250hz_format_eta(seconds):
    seconds = max(0, int(seconds))
    h, rem = divmod(seconds, 3600)
    m, s = divmod(rem, 60)
    if h > 0:
        return f"{h}h {m}m {s}s"
    return f"{m}m {s}s"


# 3) Build multilabel targets.
if "dx_code_label_sets_model_18F" in ecg_data and len(ecg_data["dx_code_label_sets_model_18F"]) == CNN_250Hz_resampled.shape[0]:
    CNN_250Hz_label_sets = ecg_data["dx_code_label_sets_model_18F"]
elif "metadata" in ecg_data and "dx_code_list" in ecg_data["metadata"].columns and len(ecg_data["metadata"]) == CNN_250Hz_resampled.shape[0]:
    CNN_250Hz_label_sets = ecg_data["metadata"]["dx_code_list"].apply(
        lambda codes: sorted({str(code).strip() for code in codes if str(code).strip()})
    )
else:
    raise RuntimeError("Could not align labels to full signal matrix for CNN training.")

# NOTE: Convert per-sample label sets into a multi-hot indicator matrix.
CNN_250Hz_mlb = MultiLabelBinarizer()
CNN_250Hz_y = CNN_250Hz_mlb.fit_transform(CNN_250Hz_label_sets)
CNN_250Hz_y = np.asarray(CNN_250Hz_y, dtype=np.float32)

# 4) Split data: keep test set untouched; derive validation from training portion only.
CNN_250Hz_X_train_full, CNN_250Hz_X_test, CNN_250Hz_y_train_full, CNN_250Hz_y_test = train_test_split(
    CNN_250Hz_resampled,
    CNN_250Hz_y,
    test_size=0.2,
    random_state=42,
    shuffle=True,
)

CNN_250Hz_X_train, CNN_250Hz_X_val, CNN_250Hz_y_train, CNN_250Hz_y_val = train_test_split(
    CNN_250Hz_X_train_full,
    CNN_250Hz_y_train_full,
    test_size=0.2,
    random_state=42,
    shuffle=True,
)

print(
    f"CNN split shapes -> train: {CNN_250Hz_X_train.shape}, "
    f"val: {CNN_250Hz_X_val.shape}, test(untouched): {CNN_250Hz_X_test.shape}"
)

CNN split shapes -> train: (28896, 12, 2500), val: (7225, 12, 2500), test(untouched): (9031, 12, 2500)


In [94]:
# CELL GUIDE:
# Purpose: Run `CNN_250Hz_resampled.shape` and continue downstream processing.
# Workflow:
# 1. CNN_250Hz_resampled.shape

CNN_250Hz_resampled.shape

(45152, 12, 2500)

In [8]:
# CELL GUIDE:
# Purpose: 5) Define a compact 1D-ResNet (template-inspired).
# Workflow:
# 1. 5) Define a compact 1D-ResNet (template-inspired).
# 2. import os, random
# 3. import numpy as np
# 4. set random seed for deterministic behavior (note: may still have some nondeterminism due to certain CUDA operations o...
# 5. seed = 42
# 6. random.seed(seed)

# 5) Define a compact 1D-ResNet (template-inspired).

import os, random
import numpy as np

#set random seed for deterministic behavior (note: may still have some nondeterminism due to certain CUDA operations or multi-threading)
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Stronger (may error if an op is nondeterministic):
# torch.use_deterministic_algorithms(True)

class CNN_250Hz_BasicBlock1D(nn.Module):
    expansion = 1

    def __init__(self, in_channels, out_channels, stride=1, kernel_size=7):
        super().__init__()
        padding = kernel_size // 2
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size=kernel_size, stride=stride, padding=padding, bias=False) 
        self.bn1 = nn.BatchNorm1d(out_channels) 
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size=kernel_size, stride=1, padding=padding, bias=False) 
        self.bn2 = nn.BatchNorm1d(out_channels)

        if stride != 1 or in_channels != out_channels:
            self.downsample = nn.Sequential(
                nn.Conv1d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm1d(out_channels),
            )
        else:
            self.downsample = nn.Identity() 

    def forward(self, x):
        identity = self.downsample(x)
        out = self.conv1(x) #learns local time-patterns (kernel=7 ⇒ looks at 7 samples at a time per lead combination)
        out = self.bn1(out) #stabilizes training by normalizing activations across the batch
        out = self.relu(out) #introduce non-linearity 
        out = self.conv2(out) #refines patterns at the same temporal resolution (stride=1)
        out = self.bn2(out)
        out += identity #skip connection (either unchanged input or 1×1 conv projection) 
        out = self.relu(out) #nonlinearity after merging
        return out


class CNN_250Hz_ResNet1D(nn.Module):
    def __init__(self, in_channels, num_classes, layers=(2, 2, 2), base_channels=32):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv1d(in_channels, base_channels, kernel_size=15, stride=2, padding=7, bias=False), 
            nn.BatchNorm1d(base_channels),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(kernel_size=3, stride=2, padding=1), #initial downsampling to reduce sequence length and increase receptive field early on
        )

        self.layer1 = self._make_layer(base_channels, base_channels, blocks=layers[0], stride=1) #maintains temporal resolution, learns richer features
        self.layer2 = self._make_layer(base_channels, base_channels * 2, blocks=layers[1], stride=2) #downsamples by 2×, learns higher-level features over longer time spans
        self.layer3 = self._make_layer(base_channels * 2, base_channels * 4, blocks=layers[2], stride=2) #further downsampling and feature abstraction

        self.pool = nn.AdaptiveAvgPool1d(1) #global pooling to get fixed-size feature vector regardless of input length
        self.head = nn.Linear(base_channels * 4, num_classes)

    def _make_layer(self, in_channels, out_channels, blocks, stride):
        layers = [CNN_250Hz_BasicBlock1D(in_channels, out_channels, stride=stride)]
        for _ in range(1, blocks):
            layers.append(CNN_250Hz_BasicBlock1D(out_channels, out_channels, stride=1))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.pool(x).squeeze(-1)
        x = self.head(x)
        return x


# 6) Train with per-batch logging, validation loss, and early stopping.
CNN_250Hz_device = torch.device("cpu")
CNN_250Hz_train_ds = TensorDataset(
    torch.from_numpy(CNN_250Hz_X_train).contiguous(),
    torch.from_numpy(CNN_250Hz_y_train).contiguous(),
)
CNN_250Hz_val_ds = TensorDataset(
    torch.from_numpy(CNN_250Hz_X_val).contiguous(),
    torch.from_numpy(CNN_250Hz_y_val).contiguous(),
)

CNN_250Hz_train_loader = DataLoader(CNN_250Hz_train_ds, batch_size=32, shuffle=True, num_workers=0)
CNN_250Hz_val_loader = DataLoader(CNN_250Hz_val_ds, batch_size=64, shuffle=False, num_workers=0)

# Initialize model, loss, optimizer.
CNN_250Hz_model = CNN_250Hz_ResNet1D(
    in_channels=CNN_250Hz_X_train.shape[1],
    num_classes=CNN_250Hz_y_train.shape[1],
    layers=(2, 2, 2),
    base_channels=32,
).to(CNN_250Hz_device)


# NOTE: Use numerically stable sigmoid+binary cross-entropy for multi-label outputs.
CNN_250Hz_criterion = nn.BCEWithLogitsLoss()
# NOTE: Configure optimizer with decoupled weight decay for CNN training.
CNN_250Hz_optimizer = optim.AdamW(CNN_250Hz_model.parameters(), lr=1e-3, weight_decay=1e-4)

CNN_250Hz_epochs = 8
CNN_250Hz_early_stopping_patience = 3
CNN_250Hz_best_val_loss = float("inf")
CNN_250Hz_best_state = None
CNN_250Hz_no_improve_epochs = 0
CNN_250Hz_batch_log_every = 100

CNN_250Hz_train_start = time.time()
for epoch in range(1, CNN_250Hz_epochs + 1):
    CNN_250Hz_model.train()
    CNN_250Hz_epoch_start = time.time()
    epoch_loss = 0.0
    n_seen = 0
    num_batches = len(CNN_250Hz_train_loader)

    for batch_idx, (xb, yb) in enumerate(CNN_250Hz_train_loader, start=1):
        batch_start = time.time()
        xb = xb.to(CNN_250Hz_device, dtype=torch.float32, non_blocking=True).contiguous()
        yb = yb.to(CNN_250Hz_device, dtype=torch.float32, non_blocking=True).contiguous()

        CNN_250Hz_optimizer.zero_grad(set_to_none=True)
        logits = CNN_250Hz_model(xb)
        loss = CNN_250Hz_criterion(logits, yb)
        loss.backward()
        CNN_250Hz_optimizer.step()

        bs = int(xb.shape[0])
        batch_seconds = max(time.time() - batch_start, 1e-9)
        batch_sps = bs / batch_seconds

        epoch_loss += float(loss.detach().cpu().item()) * bs
        n_seen += bs

        if batch_idx == 1 or batch_idx % CNN_250Hz_batch_log_every == 0 or batch_idx == num_batches:
            print(
                f"[CNN_250Hz][epoch {epoch:02d}/{CNN_250Hz_epochs}] "
                f"batch {batch_idx}/{num_batches} - "
                f"loss: {float(loss.detach().cpu().item()):.6f} - "
                f"batch_time: {batch_seconds:.3f}s - "
                f"throughput: {batch_sps:.1f} samples/s"
            )

    # Lightweight validation pass for early stopping.
    CNN_250Hz_model.eval()
    val_loss_total = 0.0
    val_seen = 0
    with torch.no_grad():
        for xb_val, yb_val in CNN_250Hz_val_loader:
            xb_val = xb_val.to(CNN_250Hz_device, dtype=torch.float32, non_blocking=True).contiguous()
            yb_val = yb_val.to(CNN_250Hz_device, dtype=torch.float32, non_blocking=True).contiguous()
            val_logits = CNN_250Hz_model(xb_val)
            val_loss = CNN_250Hz_criterion(val_logits, yb_val)
            bs_val = int(xb_val.shape[0])
            val_loss_total += float(val_loss.detach().cpu().item()) * bs_val
            val_seen += bs_val

    CNN_250Hz_val_loss_epoch = val_loss_total / max(val_seen, 1)
    CNN_250Hz_train_loss_epoch = epoch_loss / max(n_seen, 1)

    CNN_250Hz_elapsed = max(time.time() - CNN_250Hz_train_start, 1e-9)
    CNN_250Hz_avg_epoch_seconds = CNN_250Hz_elapsed / epoch
    CNN_250Hz_remaining_epochs = CNN_250Hz_epochs - epoch
    CNN_250Hz_eta_seconds = CNN_250Hz_avg_epoch_seconds * CNN_250Hz_remaining_epochs
    CNN_250Hz_epoch_seconds = time.time() - CNN_250Hz_epoch_start

    print(
        f"[CNN_250Hz] epoch {epoch:02d}/{CNN_250Hz_epochs} - "
        f"train_loss: {CNN_250Hz_train_loss_epoch:.6f} - "
        f"val_loss: {CNN_250Hz_val_loss_epoch:.6f} - "
        f"epoch_time: {CNN_250Hz_epoch_seconds:.1f}s - "
        f"ETA: {_cnn_250hz_format_eta(CNN_250Hz_eta_seconds)}"
    )

    if CNN_250Hz_val_loss_epoch < CNN_250Hz_best_val_loss:
        CNN_250Hz_best_val_loss = CNN_250Hz_val_loss_epoch
        CNN_250Hz_best_state = copy.deepcopy(CNN_250Hz_model.state_dict())
        CNN_250Hz_no_improve_epochs = 0
        print(f"[CNN_250Hz][early-stopping] improved val_loss -> {CNN_250Hz_best_val_loss:.6f}")
    else:
        CNN_250Hz_no_improve_epochs += 1
        print(
            f"[CNN_250Hz][early-stopping] no improvement "
            f"({CNN_250Hz_no_improve_epochs}/{CNN_250Hz_early_stopping_patience})"
        )
        if CNN_250Hz_no_improve_epochs >= CNN_250Hz_early_stopping_patience:
            print("[CNN_250Hz][early-stopping] stopping training.")
            break

if CNN_250Hz_best_state is not None:
    CNN_250Hz_model.load_state_dict(CNN_250Hz_best_state)

# 7) Evaluate and report metrics on validation split (test set intentionally untouched).
CNN_250Hz_model.eval()
all_probs = []
all_true = []
with torch.no_grad():
    for xb, yb in CNN_250Hz_val_loader:
        xb = xb.to(CNN_250Hz_device, dtype=torch.float32, non_blocking=True).contiguous()
        logits = CNN_250Hz_model(xb)
        probs = torch.sigmoid(logits).cpu().numpy().astype(np.float32, copy=False) #convert logits to probabilities and move to CPU for metric calculations
        all_probs.append(probs)
        all_true.append(yb.numpy().astype(np.float32, copy=False))

CNN_250Hz_y_true = np.vstack(all_true).astype(np.int32, copy=False)
CNN_250Hz_y_prob = np.vstack(all_probs).astype(np.float32, copy=False)
CNN_250Hz_y_pred = (CNN_250Hz_y_prob >= 0.5).astype(np.int32)

CNN_250Hz_metrics = {
    "precision_micro": precision_score(CNN_250Hz_y_true, CNN_250Hz_y_pred, average="micro", zero_division=0),
    "precision_macro": precision_score(CNN_250Hz_y_true, CNN_250Hz_y_pred, average="macro", zero_division=0),
    "recall_micro": recall_score(CNN_250Hz_y_true, CNN_250Hz_y_pred, average="micro", zero_division=0),
    "recall_macro": recall_score(CNN_250Hz_y_true, CNN_250Hz_y_pred, average="macro", zero_division=0),
    "f1_micro": f1_score(CNN_250Hz_y_true, CNN_250Hz_y_pred, average="micro", zero_division=0),
    "f1_macro": f1_score(CNN_250Hz_y_true, CNN_250Hz_y_pred, average="macro", zero_division=0),
}

print(f"CNN labels/classes: {CNN_250Hz_y.shape[1]}")
print(f"Best validation loss: {CNN_250Hz_best_val_loss:.6f}")
print("CNN_250Hz metrics (validation split):")
for k in ["precision_micro", "precision_macro", "recall_micro", "recall_macro", "f1_micro", "f1_macro"]:
    print(f"  {k}: {CNN_250Hz_metrics[k]:.4f}")

# NOTE: Persist artifacts in ecg_data so later cells can reuse them without recomputation.
ecg_data.setdefault("cnn_250hz", {})["metrics_holdout"] = dict(CNN_250Hz_metrics)
# NOTE: Persist artifacts in ecg_data so later cells can reuse them without recomputation.
ecg_data.setdefault("cnn_250hz", {})["signal_shape_after"] = tuple(int(v) for v in CNN_250Hz_resampled.shape)
# NOTE: Persist artifacts in ecg_data so later cells can reuse them without recomputation.
ecg_data.setdefault("cnn_250hz", {})["best_val_loss"] = float(CNN_250Hz_best_val_loss)
# NOTE: Persist artifacts in ecg_data so later cells can reuse them without recomputation.
ecg_data.setdefault("cnn_250hz", {})["early_stopping_patience"] = int(CNN_250Hz_early_stopping_patience)

# NOTE: Persist artifacts in ecg_data so later cells can reuse them without recomputation.
ecg_data.setdefault("cnn_250hz", {})["split_shapes"] = {
    "train": tuple(int(v) for v in CNN_250Hz_X_train.shape),
    "val": tuple(int(v) for v in CNN_250Hz_X_val.shape),
    "test_untouched": tuple(int(v) for v in CNN_250Hz_X_test.shape),
}


[CNN_250Hz][epoch 01/8] batch 1/903 - loss: 0.691423 - batch_time: 0.407s - throughput: 78.5 samples/s
[CNN_250Hz][epoch 01/8] batch 100/903 - loss: 0.057227 - batch_time: 0.221s - throughput: 144.7 samples/s
[CNN_250Hz][epoch 01/8] batch 200/903 - loss: 0.040209 - batch_time: 0.197s - throughput: 162.2 samples/s
[CNN_250Hz][epoch 01/8] batch 300/903 - loss: 0.030150 - batch_time: 0.204s - throughput: 156.8 samples/s
[CNN_250Hz][epoch 01/8] batch 400/903 - loss: 0.039880 - batch_time: 0.264s - throughput: 121.1 samples/s
[CNN_250Hz][epoch 01/8] batch 500/903 - loss: 0.035156 - batch_time: 0.207s - throughput: 154.8 samples/s
[CNN_250Hz][epoch 01/8] batch 600/903 - loss: 0.045907 - batch_time: 0.199s - throughput: 160.5 samples/s
[CNN_250Hz][epoch 01/8] batch 700/903 - loss: 0.034618 - batch_time: 0.204s - throughput: 157.1 samples/s
[CNN_250Hz][epoch 01/8] batch 800/903 - loss: 0.043292 - batch_time: 0.190s - throughput: 168.6 samples/s
[CNN_250Hz][epoch 01/8] batch 900/903 - loss: 0.0

In [9]:
# CELL GUIDE:
# Purpose: Find the optimal probability cutoff for CNN_250Hz using validation-set predictions only.
# Workflow:
# 1. Find the optimal probability cutoff for CNN_250Hz using validation-set predictions only.
# 2. if "CNN_250Hz_y_true" not in globals() or "CNN_250Hz_y_prob" not in globals()
# 3. CNN_250Hz_cutoff_grid = np.round(np.linspace(0.05, 0.95, 19), 2)
# 4. CNN_250Hz_threshold_rows = []
# 5. for cutoff in CNN_250Hz_cutoff_grid
# 6. CNN_250Hz_validation_threshold_df = pd.DataFrame(CNN_250Hz_threshold_rows).sort_values("cutoff").reset_index(drop=True)

# Find the optimal probability cutoff for CNN_250Hz using validation-set predictions only.
if "CNN_250Hz_y_true" not in globals() or "CNN_250Hz_y_prob" not in globals():
    raise RuntimeError("Run the CNN_250Hz validation evaluation cell first to create CNN_250Hz_y_true/CNN_250Hz_y_prob.")

# NOTE: Define a deterministic threshold grid for cutoff-sensitivity analysis.
CNN_250Hz_cutoff_grid = np.round(np.linspace(0.05, 0.95, 19), 2)
CNN_250Hz_threshold_rows = []

for cutoff in CNN_250Hz_cutoff_grid:
    y_pred_cut = (CNN_250Hz_y_prob >= float(cutoff)).astype(np.int32)
    CNN_250Hz_threshold_rows.append({
        "cutoff": float(cutoff),
        "precision_micro": precision_score(CNN_250Hz_y_true, y_pred_cut, average="micro", zero_division=0),
        "precision_macro": precision_score(CNN_250Hz_y_true, y_pred_cut, average="macro", zero_division=0),
        "recall_micro": recall_score(CNN_250Hz_y_true, y_pred_cut, average="micro", zero_division=0),
        "recall_macro": recall_score(CNN_250Hz_y_true, y_pred_cut, average="macro", zero_division=0),
        "f1_micro": f1_score(CNN_250Hz_y_true, y_pred_cut, average="micro", zero_division=0),
        "f1_macro": f1_score(CNN_250Hz_y_true, y_pred_cut, average="macro", zero_division=0),
    })

CNN_250Hz_validation_threshold_df = pd.DataFrame(CNN_250Hz_threshold_rows).sort_values("cutoff").reset_index(drop=True)

CNN_250Hz_best_cutoff_f1_micro = CNN_250Hz_validation_threshold_df.loc[
    CNN_250Hz_validation_threshold_df["f1_micro"].idxmax()  #find the row with the highest f1_micro and get its cutoff and metrics
]
CNN_250Hz_best_cutoff_f1_macro = CNN_250Hz_validation_threshold_df.loc[
    CNN_250Hz_validation_threshold_df["f1_macro"].idxmax()  #find the row with the highest f1_macro and get its cutoff and metrics
]
CNN_250Hz_optimal_cutoff = float(CNN_250Hz_best_cutoff_f1_micro["cutoff"])

CNN_250Hz_metrics_optimal_cutoff = {
    "precision_micro": float(CNN_250Hz_best_cutoff_f1_micro["precision_micro"]),
    "precision_macro": float(CNN_250Hz_best_cutoff_f1_micro["precision_macro"]),
    "recall_micro": float(CNN_250Hz_best_cutoff_f1_micro["recall_micro"]),
    "recall_macro": float(CNN_250Hz_best_cutoff_f1_micro["recall_macro"]),
    "f1_micro": float(CNN_250Hz_best_cutoff_f1_micro["f1_micro"]),
    "f1_macro": float(CNN_250Hz_best_cutoff_f1_micro["f1_macro"]),
}

print(f"CNN_250Hz optimal validation cutoff (by f1_micro): {CNN_250Hz_optimal_cutoff:.2f}")
print(f"CNN_250Hz best validation cutoff (by f1_macro): {float(CNN_250Hz_best_cutoff_f1_macro['cutoff']):.2f}")

# NOTE: Persist artifacts in ecg_data so later cells can reuse them without recomputation.
ecg_data.setdefault("cnn_250hz", {})["validation_threshold_sweep"] = CNN_250Hz_validation_threshold_df.copy()
# NOTE: Persist artifacts in ecg_data so later cells can reuse them without recomputation.
ecg_data.setdefault("cnn_250hz", {})["optimal_cutoff"] = float(CNN_250Hz_optimal_cutoff)
# NOTE: Persist artifacts in ecg_data so later cells can reuse them without recomputation.
ecg_data.setdefault("cnn_250hz", {})["metrics_at_optimal_cutoff"] = dict(CNN_250Hz_metrics_optimal_cutoff)

CNN_250Hz_validation_threshold_df.round(4)


CNN_250Hz optimal validation cutoff (by f1_micro): 0.35
CNN_250Hz best validation cutoff (by f1_macro): 0.10


,cutoff,precision_micro,precision_macro,recall_micro,recall_macro,f1_micro,f1_macro
0,0.05,0.3561,0.1739,0.9161,0.4680,0.5128,0.2309
1,0.10,0.4742,0.2153,0.8639,0.3884,0.6123,0.2580
2,0.15,0.5536,0.2341,0.8185,0.3332,0.6605,0.2567
3,0.20,0.6165,0.2522,0.7772,0.2936,0.6876,0.2477
4,0.25,0.6713,0.2566,0.7450,0.2558,0.7062,0.2370
5,0.30,0.7144,0.2764,0.7193,0.2358,0.7169,0.2314
6,0.35,0.7476,0.2742,0.6937,0.2149,0.7196,0.2174
7,0.40,0.7757,0.2780,0.6698,0.1967,0.7188,0.2064
8,0.45,0.7998,0.2833,0.6465,0.1795,0.7150,0.1974
9,0.50,0.8223,0.2885,0.6252,0.1658,0.7103,0.1872


In [ ]:
# CELL GUIDE:
# Purpose: Run `CNN_250Hz_optimal_cutoff = 0.35` and continue downstream processing.
# Workflow:
# 1. CNN_250Hz_optimal_cutoff = 0.35
# 2. CNN_250Hz cutoff visualization (validation sweep) in the same merged-subplot style.
# 3. import plotly.graph_objects as go
# 4. from plotly.subplots import make_subplots
# 5. CNN_250Hz_plot_df = CNN_250Hz_validation_threshold_df.copy()
# 6. CNN_250Hz_plot_style = graph_formatting if "graph_formatting" in globals() else {


CNN_250Hz_optimal_cutoff = 0.35

# CNN_250Hz cutoff visualization (validation sweep) in the same merged-subplot style.
import plotly.graph_objects as go
from plotly.subplots import make_subplots


CNN_250Hz_plot_df = CNN_250Hz_validation_threshold_df.copy()
CNN_250Hz_plot_style = graph_formatting if "graph_formatting" in globals() else {
    "font_family": "Helvetica Neue",
    "font_size_base": 16,
    "font_size_axis": 12,
    "color_text": "#b8b8b8",
    "color_background": "#000000",
    "margin_top": 170,
    "margin_right": 80,
    "margin_bottom": 80,
    "margin_left": 80,
}
CNN_250Hz_text = CNN_250Hz_plot_style.get("color_text", "#b8b8b8")
CNN_250Hz_bg = CNN_250Hz_plot_style.get("color_background", "#000000")
CNN_250Hz_grid = "#1f1f1f"

CNN_250Hz_fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.14,
    subplot_titles=("F1 Micro and F1 Macro", "Recall And Precision"),
)
for ann in CNN_250Hz_fig.layout.annotations:
    ann.x = 0
    ann.xanchor = "left"
    ann.align = "left"

for metric_name, color in [
    ("f1_micro", SERIES_COLORS["f1_micro"]),
    ("f1_macro", SERIES_COLORS["f1_macro"]),
]:
    pretty_name = metric_name.replace("_", " ").title()
    CNN_250Hz_fig.add_trace(
        go.Scatter(
            x=CNN_250Hz_plot_df["cutoff"],
            y=CNN_250Hz_plot_df[metric_name],
            mode="lines+markers",
            name=pretty_name,
            line=dict(width=2.2, color=color),
            marker=dict(size=6, color=color),
            hovertemplate="Cutoff: %{x:.2f}<br>" + pretty_name + ": %{y:.4f}<extra></extra>",
        ),
        row=1,
        col=1,
    )

for metric_name, color in [
    ("recall_micro", SERIES_COLORS["recall_micro"]),
    ("recall_macro", SERIES_COLORS["recall_macro"]),
    ("precision_micro", SERIES_COLORS["precision_micro"]),
    ("precision_macro", SERIES_COLORS["precision_macro"]),
]:
    pretty_name = metric_name.replace("_", " ").title()
    CNN_250Hz_fig.add_trace(
        go.Scatter(
            x=CNN_250Hz_plot_df["cutoff"],
            y=CNN_250Hz_plot_df[metric_name],
            mode="lines+markers",
            name=pretty_name,
            line=dict(width=2.0, color=color),
            marker=dict(size=6, color=color),
            hovertemplate="Cutoff: %{x:.2f}<br>" + pretty_name + ": %{y:.4f}<extra></extra>",
        ),
        row=2,
        col=1,
    )

CNN_250Hz_optimal_cutoff = float(CNN_250Hz_optimal_cutoff)
for row in (1, 2):
    CNN_250Hz_fig.add_vline(
        x=CNN_250Hz_optimal_cutoff,
        line_width=1.6,
        line_dash="dash",
        line_color="#D62728",
        row=row,
        col=1,
    )

CNN_250Hz_f1_min = float(CNN_250Hz_plot_df[["f1_micro", "f1_macro"]].min().min())
CNN_250Hz_f1_max = float(CNN_250Hz_plot_df[["f1_micro", "f1_macro"]].max().max())
CNN_250Hz_f1_pad = max(0.01, 0.15 * (CNN_250Hz_f1_max - CNN_250Hz_f1_min + 1e-12))

# NOTE: Configure x-axis semantics and formatting for accurate interpretation.
CNN_250Hz_fig.update_xaxes(
    showticklabels=False,
    showgrid=True,
    gridcolor=CNN_250Hz_grid,
    gridwidth=0.2,
    zeroline=False,
    tickfont=dict(family=CNN_250Hz_plot_style["font_family"], size=CNN_250Hz_plot_style["font_size_axis"], color=CNN_250Hz_text),
    row=1,
    col=1,
)
# NOTE: Configure x-axis semantics and formatting for accurate interpretation.
CNN_250Hz_fig.update_xaxes(
    title_text="Probability Cutoff",
    showgrid=True,
    gridcolor=CNN_250Hz_grid,
    gridwidth=0.2,
    zeroline=False,
    tickfont=dict(family=CNN_250Hz_plot_style["font_family"], size=CNN_250Hz_plot_style["font_size_axis"], color=CNN_250Hz_text),
    title_font=dict(family=CNN_250Hz_plot_style["font_family"], size=CNN_250Hz_plot_style["font_size_axis"], color=CNN_250Hz_text),
    row=2,
    col=1,
)

# NOTE: Configure y-axis semantics and formatting for accurate interpretation.
CNN_250Hz_fig.update_yaxes(
    range=[max(0.0, CNN_250Hz_f1_min - CNN_250Hz_f1_pad), min(1.02, CNN_250Hz_f1_max + CNN_250Hz_f1_pad)],
    showgrid=True,
    gridcolor=CNN_250Hz_grid,
    gridwidth=0.2,
    zeroline=False,
    tickfont=dict(family=CNN_250Hz_plot_style["font_family"], size=CNN_250Hz_plot_style["font_size_axis"], color=CNN_250Hz_text),
    row=1,
    col=1,
)

# NOTE: Configure y-axis semantics and formatting for accurate interpretation.
CNN_250Hz_fig.update_yaxes(
    range=[0, 1.02],
    showgrid=True,
    gridcolor=CNN_250Hz_grid,
    gridwidth=0.2,
    zeroline=False,
    tickfont=dict(family=CNN_250Hz_plot_style["font_family"], size=CNN_250Hz_plot_style["font_size_axis"], color=CNN_250Hz_text),
    row=2,
    col=1,
)

# NOTE: Apply global styling (title, typography, spacing, colors) for readability consistency.
CNN_250Hz_fig.update_layout(
    title=dict(
        text=(
            "<span style='font-size:30px;font-weight:bold;'>    At 0.35, Overall F1 Peaks While Rare Class Sensitivity Remains Limited</span>"
            "<br><span style='font-size:20px;font-weight: normal;'>      CNN_250Hz: Performance Metrics Across Different Cutoffs</span>"
        ),
        x=0,
        xanchor="left",
        pad=dict(t=50),
    ),
    hovermode="x unified",
    paper_bgcolor=CNN_250Hz_bg,
    plot_bgcolor=CNN_250Hz_bg,
    font=dict(family=CNN_250Hz_plot_style["font_family"], size=CNN_250Hz_plot_style["font_size_base"], color=CNN_250Hz_text),
    margin=dict(
        t=CNN_250Hz_plot_style["margin_top"] + 90,
        r=CNN_250Hz_plot_style["margin_right"],
        b=CNN_250Hz_plot_style["margin_bottom"],
        l=CNN_250Hz_plot_style["margin_left"],
    ),
    height=980,
    legend=STANDARD_LEGEND,
)

CNN_250Hz_fig.show()

In [18]:
# CELL GUIDE:
# Purpose: Evaluate CNN_250Hz on untouched test set using fixed optimal threshold 0.35.
# Workflow:
# 1. Evaluate CNN_250Hz on untouched test set using fixed optimal threshold 0.35.
# 2. if "CNN_250Hz_model" not in globals() or "CNN_250Hz_X_test" not in globals() or "CNN_250Hz_y_test" not in globals()
# 3. CNN_250Hz_test_cutoff = 0.35
# 4. CNN_250Hz_test_ds = TensorDataset(
# 5. )
# 6. CNN_250Hz_test_loader = DataLoader(CNN_250Hz_test_ds, batch_size=64, shuffle=False, num_workers=2)

# Evaluate CNN_250Hz on untouched test set using fixed optimal threshold 0.35.
if "CNN_250Hz_model" not in globals() or "CNN_250Hz_X_test" not in globals() or "CNN_250Hz_y_test" not in globals():
    raise RuntimeError("Run the CNN training/evaluation cells first to create model and test split variables.")

CNN_250Hz_test_cutoff = 0.35

CNN_250Hz_test_ds = TensorDataset(
    torch.from_numpy(np.asarray(CNN_250Hz_X_test, dtype=np.float32)).contiguous(),
    torch.from_numpy(np.asarray(CNN_250Hz_y_test, dtype=np.float32)).contiguous(),
)
CNN_250Hz_test_loader = DataLoader(CNN_250Hz_test_ds, batch_size=64, shuffle=False, num_workers=2)

CNN_250Hz_model.eval()
CNN_250Hz_test_probs = []
CNN_250Hz_test_true = []

with torch.no_grad():
    for xb, yb in CNN_250Hz_test_loader:
        xb = xb.to(CNN_250Hz_device, dtype=torch.float32, non_blocking=True).contiguous()
        logits = CNN_250Hz_model(xb)
        probs = torch.sigmoid(logits).cpu().numpy().astype(np.float32, copy=False)
        CNN_250Hz_test_probs.append(probs)
        CNN_250Hz_test_true.append(yb.numpy().astype(np.float32, copy=False))

CNN_250Hz_y_test_true = np.vstack(CNN_250Hz_test_true).astype(np.int32, copy=False)
CNN_250Hz_y_test_prob = np.vstack(CNN_250Hz_test_probs).astype(np.float32, copy=False)
CNN_250Hz_y_test_pred = (CNN_250Hz_y_test_prob >= CNN_250Hz_test_cutoff).astype(np.int32)

CNN_250Hz_test_metrics_035 = {
    "precision_micro": precision_score(CNN_250Hz_y_test_true, CNN_250Hz_y_test_pred, average="micro", zero_division=0),
    "precision_macro": precision_score(CNN_250Hz_y_test_true, CNN_250Hz_y_test_pred, average="macro", zero_division=0),
    "recall_micro": recall_score(CNN_250Hz_y_test_true, CNN_250Hz_y_test_pred, average="micro", zero_division=0),
    "recall_macro": recall_score(CNN_250Hz_y_test_true, CNN_250Hz_y_test_pred, average="macro", zero_division=0),
    "f1_micro": f1_score(CNN_250Hz_y_test_true, CNN_250Hz_y_test_pred, average="micro", zero_division=0),
    "f1_macro": f1_score(CNN_250Hz_y_test_true, CNN_250Hz_y_test_pred, average="macro", zero_division=0),
}

print(f"CNN_250Hz test metrics @ cutoff {CNN_250Hz_test_cutoff:.2f}:")
for k in ["precision_micro", "precision_macro", "recall_micro", "recall_macro", "f1_micro", "f1_macro"]:
    print(f"  {k}: {CNN_250Hz_test_metrics_035[k]:.4f}")

# NOTE: Persist artifacts in ecg_data so later cells can reuse them without recomputation.
ecg_data.setdefault("cnn_250hz", {})["test_cutoff"] = float(CNN_250Hz_test_cutoff)
# NOTE: Persist artifacts in ecg_data so later cells can reuse them without recomputation.
ecg_data.setdefault("cnn_250hz", {})["metrics_test_at_0p35"] = dict(CNN_250Hz_test_metrics_035)


python(71907) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(71908) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(71913) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(71914) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


CNN_250Hz test metrics @ cutoff 0.35:
  precision_micro: 0.7478
  precision_macro: 0.2879
  recall_micro: 0.6845
  recall_macro: 0.2006
  f1_micro: 0.7147
  f1_macro: 0.2078


# **Code Validation & Testing**


<!-- CELL NOTE -->
> **Cell Note:** This section documents **Code Validation & Testing** and provides context for the following code/results.
> Review it before executing downstream cells so assumptions and outputs remain aligned.


In [95]:
# CELL GUIDE:
# Purpose: final tests.
# Workflow:
# 1. final tests
# 2. assert CNN_250Hz_resampled.ndim == 3, "Expected (patient, lead, sample)"
# 3. assert CNN_250Hz_y.ndim == 2, "Expected multilabel matrix (patient, class)"
# 4. assert CNN_250Hz_resampled.shape[0] == CNN_250Hz_y.shape[0], "Patients and labels misaligned"
# 5. assert CNN_250Hz_resampled.shape[1] in [1, 2, 12], "Unexpected number of leads"
# 6. assert CNN_250Hz_resampled.shape[2] > 1000, "Unexpected sample length"

#final tests
assert CNN_250Hz_resampled.ndim == 3, "Expected (patient, lead, sample)"
assert CNN_250Hz_y.ndim == 2, "Expected multilabel matrix (patient, class)"
assert CNN_250Hz_resampled.shape[0] == CNN_250Hz_y.shape[0], "Patients and labels misaligned"
assert CNN_250Hz_resampled.shape[1] in [1, 2, 12], "Unexpected number of leads"
assert CNN_250Hz_resampled.shape[2] > 1000, "Unexpected sample length"
print("OK: alignment + basic shapes")

OK: alignment + basic shapes


In [96]:
# CELL GUIDE:
# Purpose: Run `n = CNN_250Hz_resampled.shape[0]` and continue downstream processing.
# Workflow:
# 1. n = CNN_250Hz_resampled.shape[0]
# 2. n_train = CNN_250Hz_X_train.shape[0]
# 3. n_val = CNN_250Hz_X_val.shape[0]
# 4. n_test = CNN_250Hz_X_test.shape[0]
# 5. assert n_train + n_val + n_test == n, "Split sizes do not sum to total"
# 6. No NaNs/Infs

n = CNN_250Hz_resampled.shape[0]
n_train = CNN_250Hz_X_train.shape[0]
n_val = CNN_250Hz_X_val.shape[0]
n_test = CNN_250Hz_X_test.shape[0]
assert n_train + n_val + n_test == n, "Split sizes do not sum to total"

# No NaNs/Infs
for name, arr in [("X_train", CNN_250Hz_X_train), ("X_val", CNN_250Hz_X_val), ("X_test", CNN_250Hz_X_test)]:
    assert np.isfinite(arr).all(), f"{name} has NaN/Inf"
print("OK: split sizes + finite values")

OK: split sizes + finite values


In [98]:
# CELL GUIDE:
# Purpose: 5) Define a compact 1D-ResNet (template-inspired).
# Workflow:
# 1. 5) Define a compact 1D-ResNet (template-inspired).
# 2. import os, random
# 3. import numpy as np
# 4. set random seed for deterministic behavior (note: may still have some nondeterminism due to certain CUDA operations o...
# 5. seed = 42
# 6. random.seed(seed)

# 5) Define a compact 1D-ResNet (template-inspired).

import os, random
import numpy as np

#set random seed for deterministic behavior (note: may still have some nondeterminism due to certain CUDA operations or multi-threading)
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Stronger (may error if an op is nondeterministic):
# torch.use_deterministic_algorithms(True)

class CNN_250Hz_BasicBlock1D(nn.Module):
    expansion = 1

    def __init__(self, in_channels, out_channels, stride=1, kernel_size=7):
        super().__init__()
        padding = kernel_size // 2
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size=kernel_size, stride=stride, padding=padding, bias=False) 
        self.bn1 = nn.BatchNorm1d(out_channels) 
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size=kernel_size, stride=1, padding=padding, bias=False) 
        self.bn2 = nn.BatchNorm1d(out_channels)

        if stride != 1 or in_channels != out_channels:
            self.downsample = nn.Sequential(
                nn.Conv1d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm1d(out_channels),
            )
        else:
            self.downsample = nn.Identity() 

    def forward(self, x):
        identity = self.downsample(x)
        out = self.conv1(x) #learns local time-patterns (kernel=7 ⇒ looks at 7 samples at a time per lead combination)
        out = self.bn1(out) #stabilizes training by normalizing activations across the batch
        out = self.relu(out) #introduce non-linearity 
        out = self.conv2(out) #refines patterns at the same temporal resolution (stride=1)
        out = self.bn2(out)
        out += identity #skip connection (either unchanged input or 1×1 conv projection) 
        out = self.relu(out) #nonlinearity after merging
        return out


class CNN_250Hz_ResNet1D(nn.Module):
    def __init__(self, in_channels, num_classes, layers=(2, 2, 2), base_channels=32):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv1d(in_channels, base_channels, kernel_size=15, stride=2, padding=7, bias=False), 
            nn.BatchNorm1d(base_channels),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(kernel_size=3, stride=2, padding=1), #initial downsampling to reduce sequence length and increase receptive field early on
        )

        self.layer1 = self._make_layer(base_channels, base_channels, blocks=layers[0], stride=1) #maintains temporal resolution, learns richer features
        self.layer2 = self._make_layer(base_channels, base_channels * 2, blocks=layers[1], stride=2) #downsamples by 2×, learns higher-level features over longer time spans
        self.layer3 = self._make_layer(base_channels * 2, base_channels * 4, blocks=layers[2], stride=2) #further downsampling and feature abstraction

        self.pool = nn.AdaptiveAvgPool1d(1) #global pooling to get fixed-size feature vector regardless of input length
        self.head = nn.Linear(base_channels * 4, num_classes)

    def _make_layer(self, in_channels, out_channels, blocks, stride):
        layers = [CNN_250Hz_BasicBlock1D(in_channels, out_channels, stride=stride)]
        for _ in range(1, blocks):
            layers.append(CNN_250Hz_BasicBlock1D(out_channels, out_channels, stride=1))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.pool(x).squeeze(-1)
        x = self.head(x)
        return x


# 6) Train with per-batch logging, validation loss, and early stopping.
CNN_250Hz_device = torch.device("cpu")
CNN_250Hz_train_ds = TensorDataset(
    torch.from_numpy(CNN_250Hz_X_train).contiguous(),
    torch.from_numpy(CNN_250Hz_y_train).contiguous(),
)
CNN_250Hz_val_ds = TensorDataset(
    torch.from_numpy(CNN_250Hz_X_val).contiguous(),
    torch.from_numpy(CNN_250Hz_y_val).contiguous(),
)

CNN_250Hz_train_loader = DataLoader(CNN_250Hz_train_ds, batch_size=32, shuffle=True, num_workers=0)
CNN_250Hz_val_loader = DataLoader(CNN_250Hz_val_ds, batch_size=64, shuffle=False, num_workers=0)

# Initialize model, loss, optimizer.
CNN_250Hz_model = CNN_250Hz_ResNet1D(
    in_channels=CNN_250Hz_X_train.shape[1],
    num_classes=CNN_250Hz_y_train.shape[1],
    layers=(2, 2, 2),
    base_channels=32,
).to(CNN_250Hz_device)


# NOTE: Use numerically stable sigmoid+binary cross-entropy for multi-label outputs.
CNN_250Hz_criterion = nn.BCEWithLogitsLoss()
# NOTE: Configure optimizer with decoupled weight decay for CNN training.
CNN_250Hz_optimizer = optim.AdamW(CNN_250Hz_model.parameters(), lr=1e-3, weight_decay=1e-4)

In [99]:
# CELL GUIDE:
# Purpose: Run `CNN_250Hz_model.eval()` and continue downstream processing.
# Workflow:
# 1. CNN_250Hz_model.eval()
# 2. with torch.no_grad()
# 3. print("OK: forward pass")

CNN_250Hz_model.eval()
with torch.no_grad():
    xb = torch.from_numpy(CNN_250Hz_X_train[:2]).float()
    out = CNN_250Hz_model(xb)
    assert out.shape == (2, CNN_250Hz_y_train.shape[1]), f"Bad output shape: {out.shape}"
    assert torch.isfinite(out).all(), "Model output has NaN/Inf"
print("OK: forward pass")

OK: forward pass


In [ ]:
# CELL GUIDE:
# Purpose: Run `CNN_250Hz_model.train()` and continue downstream processing.
# Workflow:
# 1. CNN_250Hz_model.train()
# 2. xb = torch.from_numpy(CNN_250Hz_X_train[:4]).float()
# 3. yb = torch.from_numpy(CNN_250Hz_y_train[:4]).float()
# 4. CNN_250Hz_optimizer.zero_grad(set_to_none=True)
# 5. logits = CNN_250Hz_model(xb)
# 6. loss = CNN_250Hz_criterion(logits, yb)

CNN_250Hz_model.train()
xb = torch.from_numpy(CNN_250Hz_X_train[:4]).float()
yb = torch.from_numpy(CNN_250Hz_y_train[:4]).float()

CNN_250Hz_optimizer.zero_grad(set_to_none=True)
logits = CNN_250Hz_model(xb)
loss = CNN_250Hz_criterion(logits, yb)
assert torch.isfinite(loss).item(), "Loss is NaN/Inf"
loss.backward()
CNN_250Hz_optimizer.step()
print("OK: one training step")

In [ ]:
# CELL GUIDE:
# Purpose: Run `CNN_250Hz_model.eval()` and continue downstream processing.
# Workflow:
# 1. CNN_250Hz_model.eval()
# 2. with torch.no_grad()
# 3. assert (probs >= 0).all() and (probs <= 1).all(), "Probabilities not in [0,1]"
# 4. print("OK: probability range")

CNN_250Hz_model.eval()
with torch.no_grad():
    xb = torch.from_numpy(CNN_250Hz_X_val[:8]).float()
    probs = torch.sigmoid(CNN_250Hz_model(xb)).numpy()
assert (probs >= 0).all() and (probs <= 1).all(), "Probabilities not in [0,1]"
print("OK: probability range")

## Model Card: CNN_250Hz

**Status:** Draft (derived from current notebook source and executed outputs)  
**Model type:** Multi-label 1D CNN (ResNet-style) for 12-lead ECG classification

### 1) Intended Use
- Predict multiple diagnosis labels (94 classes) from 12-lead ECG waveforms resampled to 250 Hz.
- Support threshold-based decisioning and comparison against non-CNN baselines in this notebook.

### 2) Training Data and Splits
- Input tensor shape: `(45152, 12, 2500)`
- Train split: `(28896, 12, 2500)`
- Validation split: `(7225, 12, 2500)`
- Test split (untouched during training): `(9031, 12, 2500)`
- Class count: `94`

### 3) Model Architecture and Optimization
- Backbone: Compact 1D ResNet-style network with residual blocks.
- Residual layout: `layers=(2, 2, 2)`
- Base channels: `32`
- Loss: `BCEWithLogitsLoss`
- Optimizer: `AdamW` (`lr=1e-3`, `weight_decay=1e-4`)
- Early stopping patience: `3`
- Random seed: `42`
- Trained epochs shown in notebook run: `8`

![CNN Model Card](./modelcard.png)

### 4) Validation Performance (Default sigmoid thresholding output run)
- Best validation loss: `0.030192`
- `precision_micro`: `0.8223`
- `precision_macro`: `0.2885`
- `recall_micro`: `0.6252`
- `recall_macro`: `0.1658`
- `f1_micro`: `0.7103`
- `f1_macro`: `0.1872`

### 5) Validation Cutoff Sweep
- Best cutoff by `f1_micro`: `0.35`


### 6) Test-Set Evaluation (Fixed threshold)
- Test cutoff used: `0.35`
- `precision_micro`: `0.7478`
- `precision_macro`: `0.2879`
- `recall_micro`: `0.6845`
- `recall_macro`: `0.2006`
- `f1_micro`: `0.7147`
- `f1_macro`: `0.2078`

### 7) Limitations and Risks
- Macro metrics remain substantially lower than micro metrics, indicating imbalance-sensitive behavior and weaker rare-class performance.
- This can be mitigated by implementing a minimum threshold for label prevalence in the sample (i.e. >50).
- Performance depends strongly on threshold policy; selecting by micro vs macro F1 changes operational behavior.

### 8) Reproducibility Notes
- Keep preprocessing, split seeds, and threshold policy fixed for fair model comparison.
- Track both micro and macro metrics for deployment decisions under class imbalance.



<!-- CELL NOTE -->
> **Cell Note:** This section documents **Model Card: CNN_250Hz** and provides context for the following code/results.
> Review it before executing downstream cells so assumptions and outputs remain aligned.
